# Landscape Data Commons LPI — Protocol-aware canonical code dictionary

This notebook resolves the national LPI `code` universe while preserving the fact that LDC harmonizes observations from multiple monitoring implementations.

Three observation scopes are retained:

1. **Top canopy** — `TopCanopy` only.
2. **Canopy multilayer** — `TopCanopy + Lower1...Lower7`.
3. **Raw/all contacts** — every recorded layer, including `SoilSurface`.

> **Important:** Top + Lower is a canopy-contact scope, **not complete vegetation any-hit**. Basal plant hits can occur at `SoilSurface`; true vegetation any-hit must be constructed later from the raw tall LPI table after code identity is resolved.

The raw observed code is never overwritten.

## Resolution precedence

1. Protocol/domain definitions (AIM / NRI / NWERN lineage), with context restrictions where needed.
2. Existing curated `Species_List` from prior NCA/SamplePoint work.
3. USDA PLANTS accepted-symbol match.
4. USDA PLANTS synonym-symbol match.
5. Recognized generic / unknown plant-code families.
6. Unresolved/manual-review queue.

A central design rule is that some codes are **context dependent**. Therefore the notebook does not force every code into a universal `code → meaning` mapping. Ambiguous codes such as `W` and `AG` are retained as context-dependent in the master dictionary and exported with explicit layer-specific protocol rules for later row-level resolution.


# LDC LPI Code Dictionary — Taxonomic and USDA Attribute Resolution

This notebook resolves the observed LDC/LPI code universe through:

1. preservation and canonicalization of raw observed codes,
2. protocol-aware classification of nonplant and no-canopy codes,
3. curated project species metadata,
4. USDA PLANTS accepted-symbol and synonym resolution,
5. cached USDA ecological attributes.

The notebook intentionally stops **before functional-group derivation**.

Functional-group assignment is treated as a separate ecological decision stage.

In [2]:
# =========================================================================
# 1. IMPORTS AND PATHS
# =========================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

BASE_DIR = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present"
)

LPI_CODE_FILE = (
    BASE_DIR
    / "LDC_LPI_observed_code_unique.csv"
)

USDA_PLANTS_SOURCE = (
    BASE_DIR
    / "plantlst.txt"
)

OUTPUT_DIR = (
    BASE_DIR
    / "species_dictionary_outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

USDA_ATTRIBUTE_CACHE = (
    OUTPUT_DIR
    / "USDA_PLANTS_ecological_attributes.csv"
)

DICT_PREFG_FILE = (
    OUTPUT_DIR
    / "LDC_LPI_species_dictionary_pre_functional_group.csv"
)

USDA_ATTR_FINAL_FILE = (
    OUTPUT_DIR
    / "USDA_PLANTS_ecological_attributes_final.csv"
)

CHECKPOINT_METADATA_FILE = (
    OUTPUT_DIR
    / "LDC_LPI_species_dictionary_pre_functional_group_metadata.json"
)

print("LPI code file:")
print(LPI_CODE_FILE)

print("\nUSDA PLANTS file:")
print(USDA_PLANTS_SOURCE)

print("\nUSDA attribute cache:")
print(USDA_ATTRIBUTE_CACHE)

LPI code file:
C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\LDC_LPI_observed_code_unique.csv

USDA PLANTS file:
C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\plantlst.txt

USDA attribute cache:
C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\species_dictionary_outputs\USDA_PLANTS_ecological_attributes.csv


In [3]:
# =========================================================================
# 2. LOAD AND CANONICALIZE OBSERVED LPI CODE UNIVERSE
# =========================================================================

codes = pd.read_csv(
    LPI_CODE_FILE,
    dtype={"code": "string"},
    low_memory=False
)

print("Input rows:", f"{len(codes):,}")


# -------------------------------------------------------------------------
# Preserve the original source code
# -------------------------------------------------------------------------

codes["observed_code_raw"] = codes["code"]


# -------------------------------------------------------------------------
# Identify true source blanks BEFORE canonicalization
# -------------------------------------------------------------------------

codes["source_code_was_blank"] = (
    codes["observed_code_raw"].isna()
    | codes["observed_code_raw"].str.strip().eq("")
)


# -------------------------------------------------------------------------
# Canonical observed code
#
# IMPORTANT:
# "__NO_CANOPY__" = blank source TopCanopy observation
# "NONE"          = legitimate literal USDA plant symbol
# -------------------------------------------------------------------------

codes["observed_code"] = (
    codes["observed_code_raw"]
    .astype("string")
    .str.strip()
)

codes.loc[
    codes["source_code_was_blank"],
    "observed_code"
] = "__NO_CANOPY__"


# -------------------------------------------------------------------------
# Basic QA
# -------------------------------------------------------------------------

print(
    "Canonical observed codes:",
    f"{codes['observed_code'].nunique(dropna=False):,}"
)

print(
    "Source-blank rows:",
    f"{codes['source_code_was_blank'].sum():,}"
)

print(
    "Literal NONE rows:",
    f"{codes['observed_code'].eq('NONE').sum():,}"
)

display(
    codes[
        codes["observed_code"].isin(
            ["__NO_CANOPY__", "NONE"]
        )
    ][
        [
            "observed_code",
            "observed_code_raw",
            "source_code_was_blank",
            "n_records",
            "n_top",
            "n_lower",
            "n_soil_surface",
        ]
    ]
)

Input rows: 10,960
Canonical observed codes: 10,960
Source-blank rows: 1
Literal NONE rows: 1


,observed_code,observed_code_raw,source_code_was_blank,n_records,n_top,n_lower,n_soil_surface
15,__NO_CANOPY__,<NA>,True,442612,442612,0,0
2527,NONE,NONE,False,398,394,4,0


In [4]:
# =========================================================================
# 3. CONSERVATIVE PROTOCOL-AWARE RULES
# =========================================================================

protocol_rules = pd.DataFrame(
    [
        ("__NO_CANOPY__", "No top-canopy contact", "NoCanopy"),

        ("S",  "Soil", "NonPlant"),
        ("R",  "Rock", "NonPlant"),
        ("BR", "Bedrock", "NonPlant"),
        ("GR", "Gravel", "NonPlant"),
        ("CB", "Cobble", "NonPlant"),
        ("ST", "Stone", "NonPlant"),
        ("BY", "Boulder", "NonPlant"),

        ("HL", "Herbaceous litter", "NonPlant"),
        ("WL", "Woody litter", "NonPlant"),
        ("NL", "Other litter", "NonPlant"),

        ("L",  "Lichen", "NonPlant"),
        ("LC", "Lichen / biotic crust", "NonPlant"),
        ("VL", "Vagrant lichen", "NonPlant"),
        ("M",  "Moss", "NonPlant"),

        ("EL", "Embedded litter", "NonPlant"),
        ("D",  "Duff", "NonPlant"),
        ("WA", "Water", "NonPlant"),

        # Source/protocol-dependent meanings
        ("W",  "Context-dependent W", "ContextDependent"),
        ("AG", "Context-dependent AG", "ContextDependent"),
        ("PC", "Context-dependent PC", "ContextDependent"),

        # Not safely interpreted yet
        ("O",  "Unresolved protocol code O", "ProtocolUnresolved"),
        ("PT", "Unresolved protocol code PT", "ProtocolUnresolved"),
    ],
    columns=[
        "observed_code",
        "protocol_label",
        "protocol_code_class",
    ]
)

print("Protocol rules:", len(protocol_rules))

display(protocol_rules)

Protocol rules: 23


,observed_code,protocol_label,protocol_code_class
0,__NO_CANOPY__,No top-canopy contact,NoCanopy
1,S,Soil,NonPlant
2,R,Rock,NonPlant
3,BR,Bedrock,NonPlant
4,GR,Gravel,NonPlant
5,CB,Cobble,NonPlant
6,ST,Stone,NonPlant
7,BY,Boulder,NonPlant
8,HL,Herbaceous litter,NonPlant
9,WL,Woody litter,NonPlant


In [5]:
# =========================================================================
# 4. CONSTRUCT FRESH MASTER DICTIONARY
# =========================================================================

dictionary = codes.copy()

dictionary = dictionary.merge(
    protocol_rules,
    on="observed_code",
    how="left",
    validate="many_to_one",
)

dictionary["code_class"] = (
    dictionary["protocol_code_class"]
    .astype("string")
)

dictionary["resolved_label"] = (
    dictionary["protocol_label"]
    .astype("string")
)


# -------------------------------------------------------------------------
# QA
# -------------------------------------------------------------------------

print(
    "Dictionary rows:",
    f"{len(dictionary):,}"
)

print(
    "Unique observed codes:",
    f"{dictionary['observed_code'].nunique(dropna=False):,}"
)

print("\nCurrent classification:")

display(
    dictionary["code_class"]
    .value_counts(dropna=False)
    .rename_axis("code_class")
    .reset_index(name="n_codes")
)

Dictionary rows: 10,960
Unique observed codes: 10,960

Current classification:


,code_class,n_codes
0,<NA>,10937
1,NonPlant,17
2,ContextDependent,3
3,ProtocolUnresolved,2
4,NoCanopy,1


In [6]:
# =========================================================================
# 5. LOAD USDA PLANTS TAXONOMIC REFERENCE — DEFENSIVE PARSER
# =========================================================================

print("USDA source:")
print(USDA_PLANTS_SOURCE)


# -------------------------------------------------------------------------
# Inspect raw header first
# -------------------------------------------------------------------------

with open(
    USDA_PLANTS_SOURCE,
    "r",
    encoding="utf-8-sig",
    errors="replace"
) as f:
    raw_header = f.readline()

print("\nRaw file header:")
print(repr(raw_header))


# -------------------------------------------------------------------------
# Expected USDA PLANTS fields
# -------------------------------------------------------------------------

required_usda_cols = [
    "Symbol",
    "Synonym Symbol",
    "Scientific Name with Author",
    "Common Name",
    "Family",
]


# -------------------------------------------------------------------------
# Try likely delimiters and keep the parse that recovers the expected schema
# -------------------------------------------------------------------------

candidate_separators = [
    "\t",
    "|",
    ",",
]

usda_raw = None
separator_used = None

for sep in candidate_separators:

    try:
        test = pd.read_csv(
            USDA_PLANTS_SOURCE,
            sep=sep,
            dtype="string",
            low_memory=False,
            encoding="utf-8-sig",
        )

        # Normalize column names
        test.columns = (
            test.columns
            .astype(str)
            .str.replace("\ufeff", "", regex=False)
            .str.strip()
            .str.strip('"')
            .str.strip("'")
        )

        n_required_found = sum(
            c in test.columns
            for c in required_usda_cols
        )

        print(
            f"Separator {repr(sep)}:"
            f" {len(test.columns)} columns,"
            f" {n_required_found}/{len(required_usda_cols)}"
            " required fields found"
        )

        if n_required_found == len(required_usda_cols):
            usda_raw = test
            separator_used = sep
            break

    except Exception as e:
        print(
            f"Separator {repr(sep)} failed:"
            f" {type(e).__name__}: {e}"
        )


# -------------------------------------------------------------------------
# Fallback: Python delimiter inference
# -------------------------------------------------------------------------

if usda_raw is None:

    print("\nTrying automatic delimiter inference...")

    test = pd.read_csv(
        USDA_PLANTS_SOURCE,
        sep=None,
        engine="python",
        dtype="string",
        encoding="utf-8-sig",
    )

    test.columns = (
        test.columns
        .astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
        .str.strip('"')
        .str.strip("'")
    )

    missing = [
        c for c in required_usda_cols
        if c not in test.columns
    ]

    if not missing:
        usda_raw = test
        separator_used = "auto"


# -------------------------------------------------------------------------
# Fail informatively only if no parser recovered the schema
# -------------------------------------------------------------------------

if usda_raw is None:

    raise ValueError(
        "Could not recover the expected USDA PLANTS schema.\n"
        "Inspect the raw header printed above before proceeding."
    )


print("\nUSDA file parsed successfully.")
print("Separator used:", repr(separator_used))
print("Rows:", f"{len(usda_raw):,}")
print("Columns:", len(usda_raw.columns))

print("\nParsed columns:")
for col in usda_raw.columns:
    print(f"  {repr(col)}")


# -------------------------------------------------------------------------
# Final schema check
# -------------------------------------------------------------------------

missing_usda_cols = [
    c for c in required_usda_cols
    if c not in usda_raw.columns
]

assert not missing_usda_cols, (
    "USDA schema still missing: "
    + ", ".join(missing_usda_cols)
)


# -------------------------------------------------------------------------
# Normalize USDA symbol fields
# -------------------------------------------------------------------------

for col in [
    "Symbol",
    "Synonym Symbol",
]:
    usda_raw[col] = (
        usda_raw[col]
        .astype("string")
        .str.strip()
    )

    usda_raw.loc[
        usda_raw[col].eq(""),
        col
    ] = pd.NA


# -------------------------------------------------------------------------
# Separate accepted taxa and synonym records
# -------------------------------------------------------------------------

accepted_rows = usda_raw[
    usda_raw["Synonym Symbol"].isna()
].copy()

synonym_rows = usda_raw[
    usda_raw["Synonym Symbol"].notna()
].copy()


print(
    "\nAccepted-record rows:",
    f"{len(accepted_rows):,}"
)

print(
    "Synonym-record rows:",
    f"{len(synonym_rows):,}"
)

USDA source:
C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\plantlst.txt

Raw file header:
'"Symbol","Synonym Symbol","Scientific Name with Author","Common Name","Family"\n'
Separator '\t': 1 columns, 0/5 required fields found
Separator '|': 1 columns, 0/5 required fields found
Separator ',': 5 columns, 5/5 required fields found

USDA file parsed successfully.
Separator used: ','
Rows: 93,157
Columns: 5

Parsed columns:
  'Symbol'
  'Synonym Symbol'
  'Scientific Name with Author'
  'Common Name'
  'Family'

Accepted-record rows: 48,994
Synonym-record rows: 44,163


In [7]:
# =========================================================================
# 6. BUILD USDA ACCEPTED-TAXON TABLE
# =========================================================================

usda_accepted = (
    accepted_rows[
        [
            "Symbol",
            "Scientific Name with Author",
            "Common Name",
            "Family",
        ]
    ]
    .dropna(subset=["Symbol"])
    .drop_duplicates()
    .copy()
)


# -------------------------------------------------------------------------
# Check whether any accepted symbol has genuinely conflicting taxonomy
# -------------------------------------------------------------------------

accepted_symbol_counts = (
    usda_accepted
    .groupby("Symbol", dropna=False)
    .size()
)

accepted_conflicts = (
    accepted_symbol_counts[
        accepted_symbol_counts > 1
    ]
)

print(
    "Accepted USDA symbols:",
    f"{usda_accepted['Symbol'].nunique():,}"
)

print(
    "Accepted symbols with >1 distinct metadata row:",
    f"{len(accepted_conflicts):,}"
)

if len(accepted_conflicts) > 0:
    display(
        usda_accepted[
            usda_accepted["Symbol"].isin(
                accepted_conflicts.index
            )
        ]
        .sort_values("Symbol")
        .head(100)
    )


# -------------------------------------------------------------------------
# For lookup purposes, retain one metadata row per accepted symbol.
#
# We are not changing taxonomy here; duplicate identical rows have already
# been removed above. Any remaining conflict is preserved in the diagnostic.
# -------------------------------------------------------------------------

usda_accepted_lookup = (
    usda_accepted
    .drop_duplicates(
        subset="Symbol",
        keep="first"
    )
    .rename(
        columns={
            "Symbol":
                "USDA_accepted_symbol",

            "Scientific Name with Author":
                "USDA_scientific_name",

            "Common Name":
                "USDA_common_name",

            "Family":
                "USDA_family",
        }
    )
    .copy()
)

print(
    "\nAccepted lookup rows:",
    f"{len(usda_accepted_lookup):,}"
)

Accepted USDA symbols: 48,994
Accepted symbols with >1 distinct metadata row: 0

Accepted lookup rows: 48,994


In [8]:
# =========================================================================
# 7. BUILD USDA SYNONYM → ACCEPTED SYMBOL MAP
# =========================================================================

usda_synonym_pairs = (
    synonym_rows[
        [
            "Synonym Symbol",
            "Symbol",
        ]
    ]
    .dropna(
        subset=[
            "Synonym Symbol",
            "Symbol",
        ]
    )
    .drop_duplicates()
    .rename(
        columns={
            "Synonym Symbol":
                "USDA_synonym_symbol",

            "Symbol":
                "USDA_accepted_symbol",
        }
    )
    .copy()
)


# -------------------------------------------------------------------------
# Detect synonym symbols pointing to >1 accepted symbol
# -------------------------------------------------------------------------

synonym_target_counts = (
    usda_synonym_pairs
    .groupby("USDA_synonym_symbol")[
        "USDA_accepted_symbol"
    ]
    .nunique()
)

ambiguous_synonyms = (
    synonym_target_counts[
        synonym_target_counts > 1
    ]
)

print(
    "Unique USDA synonym symbols:",
    f"{usda_synonym_pairs['USDA_synonym_symbol'].nunique():,}"
)

print(
    "Ambiguous synonym symbols:",
    f"{len(ambiguous_synonyms):,}"
)


# -------------------------------------------------------------------------
# Exclude only genuinely ambiguous synonym mappings
# -------------------------------------------------------------------------

unambiguous_synonym_pairs = (
    usda_synonym_pairs[
        ~usda_synonym_pairs[
            "USDA_synonym_symbol"
        ].isin(
            ambiguous_synonyms.index
        )
    ]
    .drop_duplicates(
        subset="USDA_synonym_symbol",
        keep="first"
    )
    .copy()
)

print(
    "Usable synonym mappings:",
    f"{len(unambiguous_synonym_pairs):,}"
)

Unique USDA synonym symbols: 44,163
Ambiguous synonym symbols: 0
Usable synonym mappings: 44,163


In [9]:
# =========================================================================
# 8. RESOLVE OBSERVED CODES TO USDA TAXONOMY
# =========================================================================

# -------------------------------------------------------------------------
# Taxonomy eligibility
#
# Only codes not already identified as protocol/nonplant are eligible.
# Literal NONE is eligible.
# __NO_CANOPY__ is not.
# -------------------------------------------------------------------------

taxonomy_eligible = (
    ~dictionary["source_code_was_blank"]
    & ~dictionary["code_class"].isin(
        [
            "NoCanopy",
            "NonPlant",
            "ContextDependent",
            "ProtocolUnresolved",
        ]
    )
)

print(
    "Taxonomy-eligible codes:",
    f"{taxonomy_eligible.sum():,}"
)


# -------------------------------------------------------------------------
# Build fast lookup structures
# -------------------------------------------------------------------------

accepted_symbols = set(
    usda_accepted_lookup[
        "USDA_accepted_symbol"
    ]
    .dropna()
)

synonym_to_accepted = dict(
    zip(
        unambiguous_synonym_pairs[
            "USDA_synonym_symbol"
        ],
        unambiguous_synonym_pairs[
            "USDA_accepted_symbol"
        ],
    )
)


# -------------------------------------------------------------------------
# Initialize resolution columns
# -------------------------------------------------------------------------

dictionary["USDA_accepted_symbol"] = pd.NA
dictionary["USDA_match_method"] = pd.NA


# -------------------------------------------------------------------------
# 1. Exact accepted-symbol matches
# -------------------------------------------------------------------------

exact_mask = (
    taxonomy_eligible
    & dictionary["observed_code"].isin(
        accepted_symbols
    )
)

dictionary.loc[
    exact_mask,
    "USDA_accepted_symbol"
] = dictionary.loc[
    exact_mask,
    "observed_code"
]

dictionary.loc[
    exact_mask,
    "USDA_match_method"
] = "accepted_symbol"


# -------------------------------------------------------------------------
# 2. Synonym matches among codes still unresolved
# -------------------------------------------------------------------------

synonym_mask = (
    taxonomy_eligible
    & dictionary["USDA_accepted_symbol"].isna()
    & dictionary["observed_code"].isin(
        synonym_to_accepted
    )
)

dictionary.loc[
    synonym_mask,
    "USDA_accepted_symbol"
] = (
    dictionary.loc[
        synonym_mask,
        "observed_code"
    ]
    .map(synonym_to_accepted)
)

dictionary.loc[
    synonym_mask,
    "USDA_match_method"
] = "synonym_symbol"


# -------------------------------------------------------------------------
# Summary before metadata join
# -------------------------------------------------------------------------

print(
    "\nExact accepted-symbol matches:",
    f"{exact_mask.sum():,}"
)

print(
    "Synonym-symbol matches:",
    f"{synonym_mask.sum():,}"
)

print(
    "Total USDA-resolved rows:",
    f"{dictionary['USDA_accepted_symbol'].notna().sum():,}"
)

print(
    "Unique accepted USDA taxa represented:",
    f"{dictionary['USDA_accepted_symbol'].nunique():,}"
)

Taxonomy-eligible codes: 10,937

Exact accepted-symbol matches: 7,160
Synonym-symbol matches: 415
Total USDA-resolved rows: 7,575
Unique accepted USDA taxa represented: 7,264


In [10]:
# =========================================================================
# 9. ATTACH USDA TAXONOMIC METADATA
# =========================================================================

dictionary = dictionary.merge(
    usda_accepted_lookup,
    on="USDA_accepted_symbol",
    how="left",
    validate="many_to_one",
)


# -------------------------------------------------------------------------
# Any successful USDA taxonomic resolution is a plant
# -------------------------------------------------------------------------

usda_plant_mask = (
    dictionary["USDA_accepted_symbol"].notna()
    & dictionary["code_class"].isna()
)

dictionary.loc[
    usda_plant_mask,
    "code_class"
] = "Plant"


# -------------------------------------------------------------------------
# USDA scientific name becomes label where protocol label is absent
# -------------------------------------------------------------------------

dictionary["resolved_label"] = (
    dictionary["resolved_label"]
    .combine_first(
        dictionary["USDA_scientific_name"]
    )
)


# -------------------------------------------------------------------------
# Summary
# -------------------------------------------------------------------------

display(
    dictionary["code_class"]
    .value_counts(dropna=False)
    .rename_axis("code_class")
    .reset_index(name="n_codes")
)

,code_class,n_codes
0,Plant,7575
1,<NA>,3362
2,NonPlant,17
3,ContextDependent,3
4,ProtocolUnresolved,2
5,NoCanopy,1


In [11]:
# =========================================================================
# 10. CRITICAL SENTINEL / USDA NONE QA
# =========================================================================

critical = dictionary[
    dictionary["observed_code"].isin(
        [
            "__NO_CANOPY__",
            "NONE",
        ]
    )
].copy()

display(
    critical[
        [
            "observed_code",
            "observed_code_raw",
            "source_code_was_blank",
            "code_class",
            "resolved_label",
            "USDA_match_method",
            "USDA_accepted_symbol",
            "USDA_scientific_name",
            "USDA_common_name",
            "USDA_family",
            "n_records",
            "n_top",
            "n_lower",
            "n_soil_surface",
        ]
    ]
)


no_canopy = dictionary[
    dictionary["observed_code"].eq(
        "__NO_CANOPY__"
    )
]

literal_none = dictionary[
    dictionary["observed_code"].eq(
        "NONE"
    )
]


# -------------------------------------------------------------------------
# These are now legitimate hard invariants
# -------------------------------------------------------------------------

assert len(no_canopy) == 1

assert (
    no_canopy[
        "source_code_was_blank"
    ]
    .all()
)

assert (
    no_canopy[
        "code_class"
    ]
    .eq("NoCanopy")
    .all()
)

assert (
    no_canopy[
        "USDA_accepted_symbol"
    ]
    .isna()
    .all()
)


assert len(literal_none) == 1

assert (
    ~literal_none[
        "source_code_was_blank"
    ]
    .all()
)

assert (
    literal_none[
        "USDA_accepted_symbol"
    ]
    .eq("NONE")
    .all()
)

assert (
    literal_none[
        "code_class"
    ]
    .eq("Plant")
    .all()
)


print(
    "\nCritical QA passed:"
    "\n  __NO_CANOPY__ is excluded from USDA taxonomy"
    "\n  literal NONE is retained as a USDA plant"
)

,observed_code,observed_code_raw,source_code_was_blank,code_class,resolved_label,USDA_match_method,USDA_accepted_symbol,USDA_scientific_name,USDA_common_name,USDA_family,n_records,n_top,n_lower,n_soil_surface
15,__NO_CANOPY__,<NA>,True,NoCanopy,No top-canopy contact,<NA>,<NA>,<NA>,<NA>,<NA>,442612,442612,0,0
2527,NONE,NONE,False,Plant,Notholaena neglecta Maxon,accepted_symbol,NONE,Notholaena neglecta Maxon,Maxon's cloak fern,Pteridaceae,398,394,4,0



Critical QA passed:
  __NO_CANOPY__ is excluded from USDA taxonomy
  literal NONE is retained as a USDA plant


C:\Users\scottfordham\AppData\Local\Temp\ipykernel_40900\2614387556.py:82: DeprecationWarning: Bitwise inversion '~' on bool is deprecated and will be removed in Python 3.16. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  ~literal_none[


In [12]:
# =========================================================================
# 11. INSPECT USDA-UNRESOLVED CODES
# =========================================================================

unresolved = dictionary[
    dictionary["code_class"].isna()
].copy()

print("Unresolved codes:", f"{len(unresolved):,}")

print(
    "Records represented by unresolved codes:",
    f"{unresolved['n_records'].sum():,}"
)

print("\nHighest-frequency unresolved codes:")

display(
    unresolved[
        [
            "observed_code",
            "n_records",
            "n_plot_visits",
            "n_top",
            "n_lower",
            "n_soil_surface",
            "first_year",
            "last_year",
        ]
    ]
    .sort_values(
        "n_records",
        ascending=False
    )
    .head(100)
)

Unresolved codes: 3,362
Records represented by unresolved codes: 529,463

Highest-frequency unresolved codes:


,observed_code,n_records,n_plot_visits,n_top,n_lower,n_soil_surface,first_year,last_year
14,RF,177392,9091,0,0,177392,2018,2024
494,LM,68581,157,0,0,68581,2018,2023
75,Perennial grasses,61012,1814,47784,13228,0,2018,2024
42,CY,56462,2957,0,0,56462,2018,2025
83,Annual plants,35750,1507,23705,12045,0,2018,2024
101,Shrubs,21122,1221,16657,4465,0,2018,2024
96,Sub-shrubs and perennial forbs,20140,1330,13449,6691,0,2018,2024
111,Plant base,17980,1105,0,0,17980,2018,2024
138,Trees,14631,841,13379,1252,0,2018,2024
128,SH00,4187,903,3145,691,351,2018,2023


In [13]:
# =========================================================================
# 11. RESOLVE GENERIC NUMBERED VEGETATION CODES
# =========================================================================

generic_rules = {
    "AF": ("Annual forb", "AnnualForb"),
    "PF": ("Perennial forb", "PerennialForb"),
    "AG": ("Annual graminoid", "AnnualGraminoid"),
    "PG": ("Perennial graminoid", "PerennialGraminoid"),
    "SH": ("Shrub", "Shrub"),
    "TR": ("Tree", "Tree"),
}

dictionary["protocol_plant_group"] = pd.Series(
    pd.NA,
    index=dictionary.index,
    dtype="string"
)

dictionary["generic_code_prefix"] = pd.Series(
    pd.NA,
    index=dictionary.index,
    dtype="string"
)

for prefix, (label, group) in generic_rules.items():

    # Must have at least one digit after the prefix.
    # Therefore bare "AG" is NOT captured.
    pattern = rf"^{prefix}\d+$"

    mask = (
        dictionary["code_class"].isna()
        & dictionary["observed_code"].str.match(
            pattern,
            na=False
        )
    )

    dictionary.loc[mask, "code_class"] = "Plant"
    dictionary.loc[mask, "resolved_label"] = label
    dictionary.loc[mask, "protocol_plant_group"] = group
    dictionary.loc[mask, "generic_code_prefix"] = prefix


generic_resolved = dictionary[
    dictionary["generic_code_prefix"].notna()
].copy()

print(
    "Generic numbered plant codes resolved:",
    f"{len(generic_resolved):,}"
)

print(
    "Records represented:",
    f"{generic_resolved['n_records'].sum():,}"
)

display(
    generic_resolved[
        [
            "observed_code",
            "resolved_label",
            "protocol_plant_group",
            "n_records",
            "n_top",
            "n_lower",
            "n_soil_surface",
        ]
    ]
    .sort_values("n_records", ascending=False)
    .head(100)
)

Generic numbered plant codes resolved: 2,954
Records represented: 40,570


,observed_code,resolved_label,protocol_plant_group,n_records,n_top,n_lower,n_soil_surface
128,SH00,Shrub,Shrub,4187,3145,691,351
156,AF00,Annual forb,AnnualForb,2058,1266,786,6
214,PF00,Perennial forb,PerennialForb,1268,825,435,8
724,TR00,Tree,Tree,972,878,67,27
459,PG00,Perennial graminoid,PerennialGraminoid,687,408,173,106
762,AG04001,Annual graminoid,AnnualGraminoid,567,546,21,0
532,AF01000,Annual forb,AnnualForb,528,386,141,1
579,AF02000,Annual forb,AnnualForb,498,347,149,2
791,PG01,Perennial graminoid,PerennialGraminoid,456,269,98,89
661,SH03000,Shrub,Shrub,319,254,40,25


In [14]:
# =========================================================================
# 12. RESOLVE EXPLICIT DESCRIPTIVE VEGETATION CLASSES
# =========================================================================

descriptive_plant_rules = {
    "Perennial grasses": (
        "Perennial grasses",
        "PerennialGraminoid"
    ),

    "Annual plants": (
        "Annual plants",
        "AnnualPlant"
    ),

    "Shrubs": (
        "Shrubs",
        "Shrub"
    ),

    "Sub-shrubs and perennial forbs": (
        "Sub-shrubs and perennial forbs",
        "SubshrubOrPerennialForb"
    ),

    "Trees": (
        "Trees",
        "Tree"
    ),
}

for code, (label, group) in descriptive_plant_rules.items():

    mask = (
        dictionary["code_class"].isna()
        & dictionary["observed_code"].eq(code)
    )

    dictionary.loc[mask, "code_class"] = "Plant"
    dictionary.loc[mask, "resolved_label"] = label
    dictionary.loc[mask, "protocol_plant_group"] = group

In [15]:
# =========================================================================
# 13. REMAINING UNRESOLVED AFTER GENERIC VEGETATION RESOLUTION
# =========================================================================

remaining = dictionary[
    dictionary["code_class"].isna()
].copy()

print(
    "Remaining unresolved codes:",
    f"{len(remaining):,}"
)

print(
    "Records represented:",
    f"{remaining['n_records'].sum():,}"
)

display(
    remaining[
        [
            "observed_code",
            "n_records",
            "n_plot_visits",
            "n_top",
            "n_lower",
            "n_soil_surface",
            "first_year",
            "last_year",
        ]
    ]
    .sort_values("n_records", ascending=False)
    .head(100)
)

Remaining unresolved codes: 403
Records represented: 336,238


,observed_code,n_records,n_plot_visits,n_top,n_lower,n_soil_surface,first_year,last_year
14,RF,177392,9091,0,0,177392,2018,2024
494,LM,68581,157,0,0,68581,2018,2023
42,CY,56462,2957,0,0,56462,2018,2025
111,Plant base,17980,1105,0,0,17980,2018,2024
509,GUTCFSAR,1894,149,1659,193,42,2019,2021
1980,PPGG,853,21,420,61,372,2019,2023
340,2FA1,849,250,524,276,49,2018,2024
1266,FG,751,43,0,1,750,2018,2023
678,CHJUMG,711,102,530,177,4,2018,2019
2184,ER,536,17,0,536,0,2018,2020


In [16]:
# ============================================================
# CONSOLIDATED LPI CODE RESOLUTION
#
# Run AFTER:
#   1. raw code canonicalization
#   2. core protocol classification
#   3. USDA accepted-symbol + synonym resolution
#
# This cell:
#   - preserves observed_code exactly
#   - only modifies currently unresolved codes
#   - handles protocol/material codes
#   - handles functional/project codes
#   - handles project-code regex families
#   - handles manual taxonomic aliases
#   - preserves uncertainty/provenance
# ============================================================

import re
import pandas as pd


# ============================================================
# 0. ENSURE REQUIRED OUTPUT COLUMNS EXIST
# ============================================================

required_resolution_columns = [
    "resolved_label",
    "resolution_source",
    "taxonomic_level",
    "protocol_plant_group",
    "USDA_match_method",
    "USDA_accepted_symbol",
    "USDA_scientific_name",
    "USDA_common_name",
    "USDA_family",
]

for col in required_resolution_columns:
    if col not in dictionary.columns:
        dictionary[col] = pd.NA


# ============================================================
# 1. FIXED PROTOCOL / MATERIAL / FUNCTIONAL RESOLUTIONS
#
# Format:
# observed_code:
#     resolved_label,
#     code_class,
#     protocol_plant_group,
#     resolution_source
# ============================================================

fixed_rules = {

    # --------------------------------------------------------
    # No-canopy sentinel
    # --------------------------------------------------------
    "__NO_CANOPY__": (
        "No canopy contact",
        "NoCanopy",
        pd.NA,
        "source_blank_sentinel",
    ),

    # --------------------------------------------------------
    # Standard / well-supported non-plant protocol codes
    # --------------------------------------------------------
    "S": (
        "Soil",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "R": (
        "Rock",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "BR": (
        "Bedrock",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "GR": (
        "Gravel",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "CB": (
        "Cobble",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "ST": (
        "Stone",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "BY": (
        "Boulder",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "RF": (
        "Rock fragment",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "FG": (
        "Fine gravel",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "HL": (
        "Herbaceous litter",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "WL": (
        "Woody litter",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "EL": (
        "Embedded litter",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "D": (
        "Duff",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "Duff": (
        "Duff",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    # IMPORTANT:
    # L = lichen, NOT litter
    "L": (
        "Lichen",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "LC": (
        "Lichen / biological crust",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "M": (
        "Moss",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "CY": (
        "Cyanobacterial crust",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "LM": (
        "Loose erodible mineral soil",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "DN": (
        "Dung",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "AM": (
        "Unattached animal material",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "HT": (
        "Human trash",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    # --------------------------------------------------------
    # Additional project/material aliases
    # --------------------------------------------------------
    "2MOSS": (
        "Moss",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    "MOSS": (
        "Moss",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    "2LW": (
        "Woody litter",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    "2LICHN": (
        "Lichen",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    "LIVR86": (
        "Liverwort",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    "LVRWORT": (
        "Liverwort",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    "2LVRWRT": (
        "Liverwort",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    "2FUNGI": (
        "Fungi",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    "2ALGA": (
        "Algae",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    "ALGAE86": (
        "Algae",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    # Deposited soil interpretation
    "DS": (
        "Deposited soil",
        "NonPlant",
        pd.NA,
        "manual_protocol_interpretation",
    ),

    # Unknown material codes whose observed vertical placement
    # strongly suggests non-vegetation rather than plant taxa.
    "ER": (
        "Unknown lower-layer material",
        "NonPlant",
        pd.NA,
        "manual_context_resolution",
    ),

    "CM": (
        "Unknown surface material",
        "NonPlant",
        pd.NA,
        "manual_context_resolution",
    ),

    "P": (
        "Unknown surface material",
        "NonPlant",
        pd.NA,
        "manual_context_resolution",
    ),

    "NT": (
        "Unknown lower-layer material",
        "NonPlant",
        pd.NA,
        "manual_context_resolution",
    ),

    # --------------------------------------------------------
    # Basal vegetation
    # --------------------------------------------------------
    # SoilSurface DOES NOT imply non-plant.
    "Plant base": (
        "Plant base",
        "Plant",
        "UnknownPlant",
        "protocol_basal_vegetation",
    ),

    # --------------------------------------------------------
    # Explicit functional/project vegetation codes
    # --------------------------------------------------------
    "PPGG": (
        "Perennial graminoid",
        "Plant",
        "PerennialGraminoid",
        "manual_functional_resolution",
    ),

    "PPFF": (
        "Perennial forb",
        "Plant",
        "PerennialForb",
        "manual_functional_resolution",
    ),

    "AAFF": (
        "Annual forb",
        "Plant",
        "AnnualForb",
        "manual_functional_resolution",
    ),

    "AAGG": (
        "Annual graminoid",
        "Plant",
        "AnnualGraminoid",
        "manual_functional_resolution",
    ),

    "SHRUB": (
        "Shrub",
        "Plant",
        "Shrub",
        "manual_functional_resolution",
    ),

    "Sh01999": (
        "Shrub",
        "Plant",
        "Shrub",
        "manual_functional_resolution",
    ),

    # SU00 interpreted as Shrub Unknown
    "SU00": (
        "Unknown shrub",
        "Plant",
        "Shrub",
        "manual_project_code_inferred",
    ),

    "UKGR": (
        "Unknown grass",
        "Plant",
        "Graminoid",
        "manual_functional_resolution",
    ),

    "ASTERA": (
        "Aster / unknown forb",
        "Plant",
        "Forb",
        "manual_functional_resolution",
    ),

    "Af02": (
        "Annual forb",
        "Plant",
        "AnnualForb",
        "manual_functional_resolution",
    ),

    "LOTUSAF": (
        "Annual forb",
        "Plant",
        "AnnualForb",
        "manual_functional_resolution",
    ),

    "BRASS2PF": (
        "Perennial Brassicaceae",
        "Plant",
        "PerennialForb",
        "manual_functional_resolution",
    ),

    "POA*": (
        "Graminoid",
        "Plant",
        "Graminoid",
        "manual_functional_resolution",
    ),

    # Explicitly unknown taxonomically but clearly vegetation
    "BOETRII": (
        "Unknown plant",
        "Plant",
        "UnknownPlant",
        "manual_unknown_plant",
    ),

    "PINSCO": (
        "Unknown plant",
        "Plant",
        "UnknownPlant",
        "manual_unknown_plant",
    ),

    "Unk": (
        "Unknown plant",
        "Plant",
        "UnknownPlant",
        "manual_unknown_plant",
    ),

    "UNK": (
        "Unknown plant",
        "Plant",
        "UnknownPlant",
        "manual_unknown_plant",
    ),
}


# ============================================================
# 2. APPLY FIXED RULES
# ============================================================

for code, (
    label,
    code_class,
    plant_group,
    source,
) in fixed_rules.items():

    mask = (
        dictionary["code_class"].isna()
        & dictionary["observed_code"].eq(code)
    )

    if not mask.any():
        continue

    dictionary.loc[mask, "code_class"] = code_class
    dictionary.loc[mask, "resolved_label"] = label
    dictionary.loc[mask, "protocol_plant_group"] = plant_group
    dictionary.loc[mask, "resolution_source"] = source


# ============================================================
# 3. GENERIC NUMBERED AIM / PROJECT FUNCTIONAL CODES
#
# Strict patterns only.
# Bare AG is intentionally NOT handled here because AG can
# represent a SoilSurface aggregate code in some protocols.
# ============================================================

project_functional_patterns = [
    (r"^AF\d+$",      "Annual forb",         "AnnualForb"),
    (r"^PF\d+$",      "Perennial forb",      "PerennialForb"),
    (r"^AG\d+$",      "Annual graminoid",    "AnnualGraminoid"),
    (r"^PG\d+$",      "Perennial graminoid", "PerennialGraminoid"),
    (r"^SH\d+$",      "Shrub",               "Shrub"),
    (r"^TR\d+$",      "Tree",                "Tree"),

    # Project-prefixed variants
    (r"^2FA\d+$",     "Annual forb",         "AnnualForb"),
    (r"^2FP\d+$",     "Perennial forb",      "PerennialForb"),
    (r"^2GA\d+$",     "Annual graminoid",    "AnnualGraminoid"),
    (r"^2GP\d+$",     "Perennial graminoid", "PerennialGraminoid"),
    (r"^2SHRUB\d+$",  "Shrub",               "Shrub"),
    (r"^2TREE\d+$",   "Tree",                "Tree"),
    (r"^2SUBS\d+$",   "Subshrub",            "Subshrub"),
]


for pattern, label, group in project_functional_patterns:

    mask = (
        dictionary["code_class"].isna()
        & dictionary["observed_code"].str.match(pattern, na=False)
    )

    dictionary.loc[mask, "code_class"] = "Plant"
    dictionary.loc[mask, "resolved_label"] = label
    dictionary.loc[mask, "protocol_plant_group"] = group
    dictionary.loc[mask, "resolution_source"] = "project_code_pattern"


# ============================================================
# 4. SPACED PG PROJECT CODES
#
# PG 04, PG 05, PG 06, etc.
# ============================================================

mask = (
    dictionary["code_class"].isna()
    & dictionary["observed_code"].str.match(
        r"^PG\s+\d+$",
        na=False,
    )
)

dictionary.loc[mask, "code_class"] = "Plant"
dictionary.loc[mask, "resolved_label"] = "Perennial graminoid"
dictionary.loc[mask, "protocol_plant_group"] = "PerennialGraminoid"
dictionary.loc[mask, "resolution_source"] = "project_code_pattern"


# ============================================================
# 5. ALL UN... CODES = UNKNOWN PLANT
#
# Preserve the original observed code.
# ============================================================

mask = (
    dictionary["code_class"].isna()
    & dictionary["observed_code"].str.match(
        r"^UN",
        case=False,
        na=False,
    )
)

dictionary.loc[mask, "code_class"] = "Plant"
dictionary.loc[mask, "resolved_label"] = "Unknown plant"
dictionary.loc[mask, "protocol_plant_group"] = "UnknownPlant"
dictionary.loc[mask, "resolution_source"] = "project_unknown_code"


# ============================================================
# 6. ELELLA
#
# Observer ambiguity between Elymus elymoides and E. lanceolatus.
# Resolve only to Elymus / perennial graminoid.
# ============================================================

mask = (
    dictionary["code_class"].isna()
    & dictionary["observed_code"].eq("ELELLA")
)

dictionary.loc[mask, "code_class"] = "Plant"
dictionary.loc[mask, "resolved_label"] = "Elymus"
dictionary.loc[mask, "taxonomic_level"] = "genus"
dictionary.loc[mask, "protocol_plant_group"] = "PerennialGraminoid"
dictionary.loc[mask, "resolution_source"] = (
    "manual_ambiguous_species_to_genus"
)


# ============================================================
# 7. MANUAL TAXONOMIC ALIASES
#
# target = canonical USDA symbol we want to query/join.
#
# The original observed_code is NEVER overwritten.
# ============================================================

manual_alias_candidates = {

    # --------------------------------------------------------
    # Cryptantha variants
    # --------------------------------------------------------
    "CRYPTA": {
        "target": "CRYPT",
        "label": "Cryptantha",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "CRYPT1": {
        "target": "CRYPT",
        "label": "Cryptantha",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "CRYPTa": {
        "target": "CRYPT",
        "label": "Cryptantha",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    # --------------------------------------------------------
    # Genus aliases
    # --------------------------------------------------------
    "ALLIUM": {
        "target": "ALLIU",
        "label": "Allium",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "ASTRAAF": {
        "target": "ASTRA",
        "label": "Astragalus",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "PHACEA": {
        "target": "PHACE",
        "label": "Phacelia",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "PHACEAF": {
        "target": "PHACE",
        "label": "Phacelia",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "PHACEa": {
        "target": "PHACE",
        "label": "Phacelia",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "CYPERA": {
        "target": "CYPER",
        "label": "Cyperus",
        "taxonomic_level": "genus",
        "resolution_source": "manual_typo_genus",
    },

    "ALCOC": {
        "target": "CALOC",
        "label": "Calochortus",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "ERITRA": {
        "target": "ERITR",
        "label": "Eritrichium",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "CERASTIUM": {
        "target": "CERAS",
        "label": "Cerastium",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    # --------------------------------------------------------
    # Gutierrezia cf. sarothrae
    # Preserve cf. uncertainty in resolved label.
    # --------------------------------------------------------
    "GUTCFSAR": {
        "target": "GUSA2",
        "label": "Gutierrezia cf. sarothrae",
        "taxonomic_level": "species_cf",
        "resolution_source": "manual_cf_taxonomic_alias",
    },

    "GUTcfSAR": {
        "target": "GUSA2",
        "label": "Gutierrezia cf. sarothrae",
        "taxonomic_level": "species_cf",
        "resolution_source": "manual_cf_taxonomic_alias",
    },

    # --------------------------------------------------------
    # CHJU project variants
    # --------------------------------------------------------
    "CHJUMG": {
        "target": "CHJU",
        "label": "CHJU",
        "taxonomic_level": "project_base_code",
        "resolution_source": "manual_project_code_alias",
    },

    "CHJUR": {
        "target": "CHJU",
        "label": "CHJU",
        "taxonomic_level": "project_base_code",
        "resolution_source": "manual_project_code_alias",
    },

    "CHJUMT": {
        "target": "CHJU",
        "label": "CHJU",
        "taxonomic_level": "project_base_code",
        "resolution_source": "manual_project_code_alias",
    },

    # --------------------------------------------------------
    # Species / typo / capitalization corrections
    # --------------------------------------------------------
    "CHIV8": {
        "target": "CHVI8",
        "label": "Chrysothamnus viscidiflorus",
        "taxonomic_level": "species",
        "resolution_source": "manual_typo_correction",
    },

    "Elel5": {
        "target": "ELEL5",
        "label": "Elymus elymoides",
        "taxonomic_level": "species",
        "resolution_source": "manual_case_correction",
    },

    "DRAR86": {
        "target": "POARA4",
        "label": "Potentilla arguta",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_alias",
    },

    "SYOR22": {
        "target": "SYOR",
        "label": "Symphoricarpos orbiculatus",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_alias",
    },

    "ANDI": {
        "target": "ANDI2",
        "label": "Antennaria dimorpha",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_resolution",
    },

    "AMAL22": {
        "target": "AMAL2",
        "label": "Amelanchier alnifolia",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_resolution",
    },

    "ERNA1": {
        "target": "ERNA10",
        "label": "ERNA10",
        "taxonomic_level": "species",
        "resolution_source": "manual_typo_correction",
    },

    "Koma": {
        "target": "KOMA",
        "label": "Koeleria macrantha",
        "taxonomic_level": "species",
        "resolution_source": "manual_case_correction",
    },

    "BAPRV": {
        "target": "BAPR",
        "label": "Bassia prostrata",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_alias",
    },

    "ARTW8": {
        "target": "ARTRW8",
        "label": "Artemisia tridentata ssp. wyomingensis",
        "taxonomic_level": "subspecies",
        "resolution_source": "manual_typo_correction",
    },

    "POAR2R2": {
        "target": "PONIN",
        "label": "Potentilla nivea var. nivea",
        "taxonomic_level": "variety",
        "resolution_source": "manual_USDA_synonym_resolution",
    },

    "SPAL86": {
        "target": "SPAL",
        "label": "Spartina",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "SPAL_01": {
        "target": "SPAL",
        "label": "Spartina",
        "taxonomic_level": "genus",
        "resolution_source": "manual_project_code_alias",

    # Sphaeralcea coccinea
    "SPCO86": {
        "target": "SPCO",
        "label": "Sphaeralcea coccinea",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_alias",
    },

    # Bassia prostrata variants
    "BAPRG": {
        "target": "BAPR5",
        "label": "Bassia prostrata",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_alias",
    },

    "BAPRV": {
        "target": "BAPR5",
        "label": "Bassia prostrata",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_alias",
    },

    # Ribes genus
    "RIBE5": {
        "target": "RIBES",
        "label": "Ribes",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    # Linum lewisii
    # observed LILEL3 -> likely intended LILE3
    "LILEL3": {
        "target": "LILE3",
        "label": "Linum lewisii",
        "taxonomic_level": "species",
        "resolution_source": "manual_typo_correction",
    },

    # Lepidium genus
    "LEPEAS": {
        "target": "LEPE",
        "label": "Lepidium",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    # Comandra umbellata
    "COUM3": {
        "target": "COUM3",
        "label": "Comandra umbellata",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_resolution",
    },

    # Phlox longifolia
    "PHLO22": {
        "target": "PHLO22",
        "label": "Phlox longifolia",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_resolution",
    },

    # Penstemon genus
    "PENST4": {
        "target": "PENST",
        "label": "Penstemon",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },
    },
}


# ============================================================
# 8. HELPER: ROBUST USDA TARGET LOOKUP
#
# Supports the column naming convention used in the current
# usda_accepted_lookup table.
# ============================================================

def get_usda_target_row(target):
    """
    Return exactly one USDA accepted-symbol row if available.
    Otherwise return None.
    """

    if "usda_accepted_lookup" not in globals():
        return None

    if "USDA_accepted_symbol" not in usda_accepted_lookup.columns:
        return None

    hit = usda_accepted_lookup[
        usda_accepted_lookup["USDA_accepted_symbol"].eq(target)
    ]

    if len(hit) == 1:
        return hit.iloc[0]

    return None


# ============================================================
# 9. APPLY MANUAL TAXONOMIC ALIASES
# ============================================================

alias_warnings = []

for observed_code, info in manual_alias_candidates.items():

    mask = (
        dictionary["code_class"].isna()
        & dictionary["observed_code"].eq(observed_code)
    )

    if not mask.any():
        continue

    target = info["target"]

    # We know these represent plants even if USDA target lookup fails.
    dictionary.loc[mask, "code_class"] = "Plant"
    dictionary.loc[mask, "resolved_label"] = info["label"]
    dictionary.loc[mask, "resolution_source"] = info["resolution_source"]
    dictionary.loc[mask, "taxonomic_level"] = info["taxonomic_level"]
    dictionary.loc[mask, "USDA_accepted_symbol"] = target
    dictionary.loc[mask, "USDA_match_method"] = "manual_alias"

    if "protocol_plant_group" in info:
        dictionary.loc[
            mask,
            "protocol_plant_group"
        ] = info["protocol_plant_group"]

    # Attach USDA metadata when target is found locally
    target_row = get_usda_target_row(target)

    if target_row is not None:

        metadata_cols = [
            "USDA_scientific_name",
            "USDA_common_name",
            "USDA_family",
        ]

        for col in metadata_cols:
            if col in target_row.index:
                dictionary.loc[mask, col] = target_row[col]

    else:
        alias_warnings.append(
            (observed_code, target)
        )


# ============================================================
# 10. OPTIONAL MANUAL FUNCTIONAL OVERRIDES FOR ALIASED TAXA
#
# These encode ecological information that is defensible even
# when the taxonomic alias itself is genus-level or uncertain.
# ============================================================

functional_overrides = {

    # Elymus ambiguity resolved structurally
    "ELELLA": "PerennialGraminoid",

    # Unknown grasses / graminoids
    "UKGR": "Graminoid",

    # Aster only known as forb
    "ASTERA": "Forb",

    # Explicit project functional encodings
    "LOTUSAF": "AnnualForb",
    "BRASS2PF": "PerennialForb",

    # Wyoming big sagebrush
    "ARTW8": "Shrub",

    # Koeleria
    "Koma": "PerennialGraminoid",

    # Cordgrass / Spartina
    "SPAL86": "PerennialGraminoid",
    "SPAL_01": "PerennialGraminoid",

    # Bassia prostrata:
    # treat as exotic forb in this framework, NOT shrub
    "BAPRV": "Forb",
    "BAPRG": "Forb",

    # Sphaeralcea coccinea
    "SPCO86": "Forb",

    # Ribes
    "RIBE5": "Shrub",

    # Linum lewisii
    "LILEL3": "Forb",

    # Lepidium
    "LEPEAS": "Forb",

    # Comandra umbellata
    "COUM3": "Forb",

    # Phlox longifolia
    "PHLO22": "Forb",

    # Penstemon
    "PENST4": "Forb",
}


for code, group in functional_overrides.items():

    mask = dictionary["observed_code"].eq(code)

    dictionary.loc[
        mask,
        "protocol_plant_group"
    ] = group


# ============================================================
# 11. QA: MANUAL RULES THAT FAILED USDA TARGET LOOKUP
#
# This does NOT undo their classification.
# It simply tells us which canonical targets were not found
# in the local accepted-symbol lookup.
# ============================================================

if alias_warnings:

    alias_warning_df = pd.DataFrame(
        alias_warnings,
        columns=[
            "observed_code",
            "requested_USDA_target",
        ],
    ).drop_duplicates()

    print(
        f"Manual alias targets not found in "
        f"usda_accepted_lookup: {len(alias_warning_df):,}"
    )

    display(alias_warning_df)

else:
    print(
        "All applied manual alias targets found "
        "in usda_accepted_lookup."
    )


# ============================================================
# 12. QA: DISPLAY ALL CODES TOUCHED BY THIS CONSOLIDATED CELL
# ============================================================

manual_codes = (
    set(fixed_rules.keys())
    | set(manual_alias_candidates.keys())
)

qa_manual = (
    dictionary[
        dictionary["observed_code"].isin(manual_codes)
        | dictionary["resolution_source"].isin(
            [
                "project_code_pattern",
                "project_unknown_code",
                "manual_ambiguous_species_to_genus",
            ]
        )
    ][
        [
            "observed_code",
            "n_records",
            "code_class",
            "resolved_label",
            "USDA_accepted_symbol",
            "protocol_plant_group",
            "taxonomic_level",
            "resolution_source",
        ]
    ]
    .sort_values(
        ["n_records", "observed_code"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

print(
    f"Codes represented in consolidated resolution QA: "
    f"{len(qa_manual):,}"
)

display(qa_manual)


# ============================================================
# 13. FINAL UNRESOLVED TABLE
# ============================================================

unresolved = (
    dictionary[
        dictionary["code_class"].isna()
    ][
        [
            "observed_code",
            "n_records",
            "n_plot_visits",
            "n_top",
            "n_lower",
            "n_soil_surface",
            "first_year",
            "last_year",
        ]
    ]
    .sort_values(
        "n_records",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    f"Remaining unresolved codes: "
    f"{len(unresolved):,}"
)

print(
    f"Records represented: "
    f"{unresolved['n_records'].sum():,}"
)

display(unresolved.head(100))


# ============================================================
# 14. STOPPING-RULE QA
#
# Our manual-review threshold is 50 observations.
# Anything remaining below this threshold can be retained as
# unresolved rather than forcing increasingly speculative IDs.
# ============================================================

needs_review = unresolved[
    unresolved["n_records"] >= 50
].copy()

below_manual_threshold = unresolved[
    unresolved["n_records"] < 50
].copy()

print()
print(
    f"Still >= 50 records and requiring review: "
    f"{len(needs_review):,}"
)

print(
    f"Remaining < 50-record codes: "
    f"{len(below_manual_threshold):,}"
)

print(
    f"Records represented by <50 tail: "
    f"{below_manual_threshold['n_records'].sum():,}"
)

display(needs_review)

All applied manual alias targets found in usda_accepted_lookup.
Codes represented in consolidated resolution QA: 246


,observed_code,n_records,code_class,resolved_label,USDA_accepted_symbol,protocol_plant_group,taxonomic_level,resolution_source
0,S,4908062,NonPlant,Soil,<NA>,<NA>,<NA>,<NA>
1,HL,2420983,NonPlant,Herbaceous litter,<NA>,<NA>,<NA>,<NA>
2,L,853273,NonPlant,Lichen,<NA>,<NA>,<NA>,<NA>
3,GR,721055,NonPlant,Gravel,<NA>,<NA>,<NA>,<NA>
4,__NO_CANOPY__,442612,NoCanopy,No top-canopy contact,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...
241,UN23070,1,Plant,Unknown plant,<NA>,UnknownPlant,<NA>,project_unknown_code
242,UN25037,1,Plant,Unknown plant,<NA>,UnknownPlant,<NA>,project_unknown_code
243,UN33009,1,Plant,Unknown plant,<NA>,UnknownPlant,<NA>,project_unknown_code
244,UNK10,1,Plant,Unknown plant,<NA>,UnknownPlant,<NA>,project_unknown_code


Remaining unresolved codes: 172
Records represented: 2,186


,observed_code,n_records,n_plot_visits,n_top,n_lower,n_soil_surface,first_year,last_year
0,PHYGOR,365,50,271,89,5,2018,2023
1,PHYFEN,290,66,193,90,7,2018,2023
2,SIDTEN,246,25,227,17,2,2018,2024
3,SPOCFCRY,220,38,173,29,18,2018,2021
4,BOERHAF,75,10,72,3,0,2022,2023
5,PS09009,63,1,45,12,6,2023,2023
6,PS91006,57,2,41,15,1,2021,2021
7,SPHPUM,39,24,24,13,2,2018,2023
8,PL,33,10,1,21,11,2018,2018
9,DF,25,13,2,23,0,2021,2022



Still >= 50 records and requiring review: 7
Remaining < 50-record codes: 165
Records represented by <50 tail: 870


,observed_code,n_records,n_plot_visits,n_top,n_lower,n_soil_surface,first_year,last_year
0,PHYGOR,365,50,271,89,5,2018,2023
1,PHYFEN,290,66,193,90,7,2018,2023
2,SIDTEN,246,25,227,17,2,2018,2024
3,SPOCFCRY,220,38,173,29,18,2018,2021
4,BOERHAF,75,10,72,3,0,2022,2023
5,PS09009,63,1,45,12,6,2023,2023
6,PS91006,57,2,41,15,1,2021,2021


In [17]:
# ============================================================
# FINALIZE REMAINING HIGH-FREQUENCY UNRESOLVED PLANT CODES
#
# These have been manually reviewed and cannot be resolved
# defensibly beyond unknown vegetation.
# ============================================================

reviewed_unknown_plant_codes = [
    "PHYGOR",
    "PHYFEN",
    "SIDTEN",
    "SPOCFCRY",
    "BOERHAF",
    "PS09009",
    "PS91006",
]

mask = (
    dictionary["code_class"].isna()
    & dictionary["observed_code"].isin(reviewed_unknown_plant_codes)
)

dictionary.loc[mask, "code_class"] = "Plant"
dictionary.loc[mask, "resolved_label"] = "Unknown plant"
dictionary.loc[mask, "protocol_plant_group"] = "UnknownPlant"
dictionary.loc[mask, "resolution_source"] = "manual_review_unresolvable"

In [18]:
# ============================================================
# TOTAL SPECIES-DICTIONARY METADATA + SUMMARY
#
# Purpose:
#   Audit the completed pre-functional-group dictionary at both:
#       1. unique observed-code level
#       2. LPI record-weighted level
#
# Assumes:
#   dictionary exists
#   n_records is present
# ============================================================

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)


# ============================================================
# 0. BASIC INTEGRITY
# ============================================================

N_CODES = len(dictionary)
N_RECORDS = int(dictionary["n_records"].sum())

print("=" * 80)
print("LPI SPECIES / PROTOCOL DICTIONARY — TOTAL SUMMARY")
print("=" * 80)

print(f"Unique observed codes:             {N_CODES:,}")
print(f"Total LPI records represented:     {N_RECORDS:,}")

if "n_plot_visits" in dictionary.columns:
    print(
        f"Sum of code-level plot occurrences:{dictionary['n_plot_visits'].sum():,}"
    )

print()


# ============================================================
# 1. CORE COMPLETENESS
# ============================================================

summary_fields = [
    "code_class",
    "resolved_label",
    "resolution_source",
    "taxonomic_level",
    "protocol_plant_group",
    "USDA_accepted_symbol",
    "USDA_scientific_name",
    "USDA_common_name",
    "USDA_family",
]

completeness_rows = []

for col in summary_fields:

    if col not in dictionary.columns:
        continue

    resolved = dictionary[col].notna()

    n_codes = int(resolved.sum())
    records = int(dictionary.loc[resolved, "n_records"].sum())

    completeness_rows.append({
        "field": col,
        "codes_present": n_codes,
        "codes_pct": 100 * n_codes / N_CODES,
        "records_present": records,
        "records_pct": 100 * records / N_RECORDS,
    })

completeness = pd.DataFrame(completeness_rows)

print("FIELD COMPLETENESS")
display(
    completeness.style.format({
        "codes_pct": "{:.2f}",
        "records_pct": "{:.4f}",
    })
)


# ============================================================
# 2. CODE CLASS — UNIQUE CODES + RECORD-WEIGHTED
# ============================================================

def categorical_summary(df, column):
    """
    Summarize a categorical dictionary field by:
      - number of unique observed codes
      - percent of dictionary codes
      - represented LPI records
      - percent of all represented records
    """

    x = df.copy()

    x["_category"] = (
        x[column]
        .astype("string")
        .fillna("<UNRESOLVED>")
    )

    out = (
        x.groupby("_category", dropna=False)
        .agg(
            n_codes=("observed_code", "size"),
            n_records=("n_records", "sum"),
        )
        .reset_index()
        .rename(columns={"_category": column})
    )

    out["codes_pct"] = 100 * out["n_codes"] / N_CODES
    out["records_pct"] = 100 * out["n_records"] / N_RECORDS

    return (
        out.sort_values(
            ["n_records", "n_codes"],
            ascending=False,
        )
        .reset_index(drop=True)
    )


print()
print("CODE CLASS")
code_class_summary = categorical_summary(
    dictionary,
    "code_class",
)

display(
    code_class_summary.style.format({
        "codes_pct": "{:.2f}",
        "records_pct": "{:.4f}",
    })
)


# ============================================================
# 3. RESOLUTION SOURCE
# ============================================================

print()
print("RESOLUTION SOURCE")

resolution_summary = categorical_summary(
    dictionary,
    "resolution_source",
)

display(
    resolution_summary.style.format({
        "codes_pct": "{:.2f}",
        "records_pct": "{:.4f}",
    })
)


# ============================================================
# 4. TAXONOMIC LEVEL
# ============================================================

print()
print("TAXONOMIC LEVEL")

taxonomic_level_summary = categorical_summary(
    dictionary,
    "taxonomic_level",
)

display(
    taxonomic_level_summary.style.format({
        "codes_pct": "{:.2f}",
        "records_pct": "{:.4f}",
    })
)


# ============================================================
# 5. SOURCE-LEVEL FUNCTIONAL INFORMATION
#
# This is NOT the final MOSAIC functional-group derivation.
# It only summarizes protocol_plant_group already known from
# source/project information.
# ============================================================

print()
print("PROTOCOL / SOURCE-LEVEL PLANT GROUP")

protocol_group_summary = categorical_summary(
    dictionary,
    "protocol_plant_group",
)

display(
    protocol_group_summary.style.format({
        "codes_pct": "{:.2f}",
        "records_pct": "{:.4f}",
    })
)


# ============================================================
# 6. PLANT-ONLY SUMMARY
# ============================================================

plants = dictionary[
    dictionary["code_class"].eq("Plant")
].copy()

N_PLANT_CODES = len(plants)
N_PLANT_RECORDS = int(plants["n_records"].sum())

print()
print("=" * 80)
print("PLANT-CODE SUMMARY")
print("=" * 80)

print(f"Plant codes:                       {N_PLANT_CODES:,}")
print(f"Plant-contact records:             {N_PLANT_RECORDS:,}")
print(f"Plant codes / all codes:           {100*N_PLANT_CODES/N_CODES:.2f}%")
print(f"Plant records / all records:       {100*N_PLANT_RECORDS/N_RECORDS:.2f}%")

print()


# ============================================================
# 7. USDA TAXONOMY COVERAGE AMONG PLANTS
# ============================================================

if N_PLANT_CODES > 0:

    usda_resolved = plants[
        plants["USDA_accepted_symbol"].notna()
    ].copy()

    n_usda_codes = len(usda_resolved)
    n_usda_records = int(usda_resolved["n_records"].sum())

    print("USDA TAXONOMIC COVERAGE AMONG PLANTS")
    print(
        f"Plant codes with USDA target:      "
        f"{n_usda_codes:,} "
        f"({100*n_usda_codes/N_PLANT_CODES:.2f}%)"
    )

    print(
        f"Plant records with USDA target:    "
        f"{n_usda_records:,} "
        f"({100*n_usda_records/N_PLANT_RECORDS:.2f}%)"
    )

    print(
        f"Distinct USDA accepted symbols:    "
        f"{usda_resolved['USDA_accepted_symbol'].nunique():,}"
    )

    if "USDA_scientific_name" in usda_resolved.columns:
        print(
            f"Distinct scientific names:         "
            f"{usda_resolved['USDA_scientific_name'].nunique():,}"
        )


# ============================================================
# 8. UNKNOWN / UNRESOLVED VEGETATION
# ============================================================

unknown_plant_mask = (
    dictionary["code_class"].eq("Plant")
    & (
        dictionary["protocol_plant_group"].eq("UnknownPlant")
        | dictionary["resolved_label"].eq("Unknown plant")
    )
)

unknown_plants = dictionary[
    unknown_plant_mask
].copy()

unresolved = dictionary[
    dictionary["code_class"].isna()
].copy()

unknown_plant_records = int(
    unknown_plants["n_records"].sum()
)

unresolved_records = int(
    unresolved["n_records"].sum()
)

print()
print("=" * 80)
print("UNKNOWN / UNRESOLVED IMPACT")
print("=" * 80)

print(
    f"Explicit UnknownPlant codes:       "
    f"{len(unknown_plants):,}"
)

print(
    f"Explicit UnknownPlant records:     "
    f"{unknown_plant_records:,} "
    f"({100*unknown_plant_records/N_RECORDS:.5f}% of all records)"
)

print(
    f"Still-unresolved codes:            "
    f"{len(unresolved):,}"
)

print(
    f"Still-unresolved records:          "
    f"{unresolved_records:,} "
    f"({100*unresolved_records/N_RECORDS:.5f}% of all records)"
)

if N_PLANT_RECORDS > 0:
    print(
        f"UnknownPlant share of plant hits:  "
        f"{100*unknown_plant_records/N_PLANT_RECORDS:.4f}%"
    )


# ============================================================
# 9. TOP EXPLICIT UNKNOWN PLANT CODES
# ============================================================

print()
print("TOP UNKNOWN-PLANT CODES")

display(
    unknown_plants[
        [
            "observed_code",
            "n_records",
            "n_plot_visits",
            "n_top",
            "n_lower",
            "n_soil_surface",
            "resolved_label",
            "resolution_source",
        ]
    ]
    .sort_values(
        "n_records",
        ascending=False,
    )
    .head(30)
    .reset_index(drop=True)
)


# ============================================================
# 10. TOP REMAINING COMPLETELY UNRESOLVED CODES
# ============================================================

print()
print("TOP COMPLETELY UNRESOLVED CODES")

display(
    unresolved[
        [
            "observed_code",
            "n_records",
            "n_plot_visits",
            "n_top",
            "n_lower",
            "n_soil_surface",
            "first_year",
            "last_year",
        ]
    ]
    .sort_values(
        "n_records",
        ascending=False,
    )
    .head(50)
    .reset_index(drop=True)
)


# ============================================================
# 11. MOST ABUNDANT OBSERVED CODES
# ============================================================

print()
print("=" * 80)
print("MOST ABUNDANT OBSERVED CODES")
print("=" * 80)

display(
    dictionary[
        [
            "observed_code",
            "n_records",
            "n_plot_visits",
            "code_class",
            "resolved_label",
            "USDA_accepted_symbol",
            "protocol_plant_group",
            "resolution_source",
        ]
    ]
    .sort_values(
        "n_records",
        ascending=False,
    )
    .head(50)
    .reset_index(drop=True)
)


# ============================================================
# 12. MOST ABUNDANT PLANT CODES
# ============================================================

print()
print("MOST ABUNDANT PLANT CODES")

display(
    plants[
        [
            "observed_code",
            "n_records",
            "n_plot_visits",
            "resolved_label",
            "USDA_accepted_symbol",
            "USDA_scientific_name",
            "protocol_plant_group",
            "resolution_source",
        ]
    ]
    .sort_values(
        "n_records",
        ascending=False,
    )
    .head(50)
    .reset_index(drop=True)
)


# ============================================================
# 13. VERTICAL-CONTACT SUMMARY BY CODE CLASS
#
# Useful QA because TopCanopy / Lower / SoilSurface have very
# different ecological meanings.
# ============================================================

vertical_cols = [
    "n_top",
    "n_lower",
    "n_soil_surface",
]

if all(
    col in dictionary.columns
    for col in vertical_cols
):

    vertical_summary = (
        dictionary
        .assign(
            code_class_display=
                dictionary["code_class"]
                .astype("string")
                .fillna("<UNRESOLVED>")
        )
        .groupby(
            "code_class_display"
        )[vertical_cols]
        .sum()
        .reset_index()
    )

    vertical_summary["total_contacts"] = (
        vertical_summary[vertical_cols]
        .sum(axis=1)
    )

    for col in vertical_cols:
        vertical_summary[f"{col}_pct"] = (
            100
            * vertical_summary[col]
            / vertical_summary["total_contacts"]
        )

    print()
    print("VERTICAL CONTACT DISTRIBUTION BY CODE CLASS")

    display(
        vertical_summary.style.format({
            "n_top_pct": "{:.2f}",
            "n_lower_pct": "{:.2f}",
            "n_soil_surface_pct": "{:.2f}",
        })
    )


# ============================================================
# 14. TEMPORAL COVERAGE
# ============================================================

if {
    "first_year",
    "last_year",
}.issubset(dictionary.columns):

    print()
    print("TEMPORAL COVERAGE")

    print(
        f"Earliest code occurrence:          "
        f"{dictionary['first_year'].min()}"
    )

    print(
        f"Latest code occurrence:            "
        f"{dictionary['last_year'].max()}"
    )

    dictionary["_years_present_span"] = (
        dictionary["last_year"]
        - dictionary["first_year"]
        + 1
    )

    print(
        f"Codes observed in >1-year span:    "
        f"{(dictionary['_years_present_span'] > 1).sum():,}"
    )


# ============================================================
# 15. DUPLICATE / UNIQUENESS QA
# ============================================================

print()
print("=" * 80)
print("UNIQUENESS QA")
print("=" * 80)

n_duplicate_codes = int(
    dictionary["observed_code"].duplicated().sum()
)

print(
    f"Duplicate observed_code rows:      "
    f"{n_duplicate_codes:,}"
)

if n_duplicate_codes > 0:

    display(
        dictionary[
            dictionary["observed_code"].duplicated(
                keep=False
            )
        ].sort_values("observed_code")
    )


# ============================================================
# 16. USDA ACCEPTED SYMBOL CONVERGENCE
#
# How many raw/project codes converge onto the same accepted
# taxon? Useful for seeing the benefit of synonym + alias work.
# ============================================================

usda_convergence = (
    dictionary[
        dictionary["USDA_accepted_symbol"].notna()
    ]
    .groupby(
        "USDA_accepted_symbol"
    )
    .agg(
        n_observed_codes=("observed_code", "nunique"),
        n_records=("n_records", "sum"),
        scientific_name=(
            "USDA_scientific_name",
            "first",
        ),
    )
    .reset_index()
    .sort_values(
        ["n_observed_codes", "n_records"],
        ascending=False,
    )
)

print()
print("USDA SYMBOL CONVERGENCE")

print(
    f"Accepted taxa represented by >1 observed code: "
    f"{(usda_convergence['n_observed_codes'] > 1).sum():,}"
)

display(
    usda_convergence[
        usda_convergence["n_observed_codes"] > 1
    ]
    .head(50)
    .reset_index(drop=True)
)


# ============================================================
# 17. COMPACT FINAL AUDIT
# ============================================================

classified_mask = dictionary["code_class"].notna()

classified_codes = int(classified_mask.sum())
classified_records = int(
    dictionary.loc[
        classified_mask,
        "n_records",
    ].sum()
)

print()
print("=" * 80)
print("FINAL AUDIT")
print("=" * 80)

print(
    f"Classified codes:                  "
    f"{classified_codes:,} / {N_CODES:,} "
    f"({100*classified_codes/N_CODES:.2f}%)"
)

print(
    f"Classified records represented:    "
    f"{classified_records:,} / {N_RECORDS:,} "
    f"({100*classified_records/N_RECORDS:.5f}%)"
)

print(
    f"Unclassified record fraction:      "
    f"{100*(N_RECORDS-classified_records)/N_RECORDS:.5f}%"
)

print("=" * 80)

LPI SPECIES / PROTOCOL DICTIONARY — TOTAL SUMMARY
Unique observed codes:             10,960
Total LPI records represented:     16,150,101
Sum of code-level plot occurrences:812,524

FIELD COMPLETENESS


,field,codes_present,codes_pct,records_present,records_pct
0,code_class,10795,98.49,16149231,99.9946
1,resolved_label,10795,98.49,16149231,99.9946
2,resolution_source,238,2.17,335368,2.0766
3,taxonomic_level,31,0.28,4023,0.0249
4,protocol_plant_group,3157,28.80,219371,1.3583
5,USDA_accepted_symbol,7605,69.39,5217035,32.3034
6,USDA_scientific_name,7605,69.39,5217035,32.3034
7,USDA_common_name,7547,68.86,5215423,32.2934
8,USDA_family,7605,69.39,5217035,32.3034



CODE CLASS


,code_class,n_codes,n_records,codes_pct,records_pct
0,NonPlant,40,10177190,0.36,63.0163
1,Plant,10749,5436341,98.07,33.6613
2,NoCanopy,1,442612,0.01,2.7406
3,ContextDependent,3,89313,0.03,0.5530
4,ProtocolUnresolved,2,3775,0.02,0.0234
5,,165,870,1.51,0.0054



RESOLUTION SOURCE


...


VERTICAL CONTACT DISTRIBUTION BY CODE CLASS


,code_class_display,n_top,n_lower,n_soil_surface,total_contacts,n_top_pct,n_lower_pct,n_soil_surface_pct
0,,512,324,34,870,58.85,37.24,3.91
1,ContextDependent,0,1402,87911,89313,0.00,1.57,98.43
2,NoCanopy,442612,0,0,442612,100.00,0.00,0.00
3,NonPlant,130,3586943,6590117,10177190,0.00,35.24,64.75
4,Plant,3862958,1285519,287864,5436341,71.06,23.65,5.30
5,ProtocolUnresolved,0,0,3775,3775,0.00,0.00,100.00



TEMPORAL COVERAGE
Earliest code occurrence:          2018
Latest code occurrence:            2025
Codes observed in >1-year span:    5,604

UNIQUENESS QA
Duplicate observed_code rows:      0

USDA SYMBOL CONVERGENCE
Accepted taxa represented by >1 observed code: 292


,USDA_accepted_symbol,n_observed_codes,n_records,scientific_name
0,KOMA,6,41779,Koeleria macrantha (Ledeb.) Schult.
1,POSE,5,333033,Poa secunda J. Presl
2,THIN6,5,11859,Thinopyrum intermedium (Host) Barkworth & D.R....
3,GUSA2,4,30001,Gutierrezia sarothrae (Pursh) Britton & Rusby
4,CHJU,4,3398,Chondrilla juncea L.
5,CRYPT,4,1568,Cryptantha Lehm. ex G. Don
6,BRCA5,4,643,Bromus carinatus Hook. & Arn.
7,PHACE,4,287,Phacelia Juss.
8,EUGL19,4,119,Eurybia glauca (Nutt.) G.L. Nesom
9,BRTE,3,429829,Bromus tectorum L.



FINAL AUDIT
Classified codes:                  10,795 / 10,960 (98.49%)
Classified records represented:    16,149,231 / 16,150,101 (99.99461%)
Unclassified record fraction:      0.00539%


In [19]:
# ============================================================
# BACKFILL RESOLUTION PROVENANCE
#
# Earlier pipeline stages resolved many codes before
# resolution_source was introduced. This reconstructs
# provenance without overwriting later/manual decisions.
# ============================================================

source_missing = (
    dictionary["resolution_source"].isna()
    | dictionary["resolution_source"].astype("string").str.strip().eq("")
)


# ------------------------------------------------------------
# 1. No-canopy sentinel
# ------------------------------------------------------------

mask = (
    source_missing
    & dictionary["observed_code"].eq("__NO_CANOPY__")
)

dictionary.loc[
    mask,
    "resolution_source"
] = "source_blank_sentinel"


# ------------------------------------------------------------
# 2. USDA taxonomy resolution
#
# Prefer USDA_match_method because it preserves whether the
# original resolution was accepted-symbol vs synonym-symbol.
# ------------------------------------------------------------

if "USDA_match_method" in dictionary.columns:

    usda_exact = (
        source_missing
        & dictionary["USDA_match_method"].astype("string").isin([
            "accepted",
            "accepted_symbol",
            "exact",
            "exact_accepted_symbol",
        ])
    )

    dictionary.loc[
        usda_exact,
        "resolution_source"
    ] = "USDA_accepted_symbol"


    usda_synonym = (
        source_missing
        & dictionary["USDA_match_method"].astype("string").isin([
            "synonym",
            "synonym_symbol",
            "USDA_synonym",
        ])
    )

    dictionary.loc[
        usda_synonym,
        "resolution_source"
    ] = "USDA_synonym_symbol"


# Catch USDA-resolved rows whose older match-method naming
# does not match the strings above.
source_missing = dictionary["resolution_source"].isna()

mask = (
    source_missing
    & dictionary["USDA_accepted_symbol"].notna()
)

dictionary.loc[
    mask,
    "resolution_source"
] = "USDA_taxonomic_resolution"


# ------------------------------------------------------------
# 3. Previously resolved protocol/nonplant codes
#
# Only fills remaining missing provenance.
# ------------------------------------------------------------

source_missing = dictionary["resolution_source"].isna()

core_protocol_codes = {
    "S", "R", "BR", "GR", "CB", "ST", "BY",
    "RF", "FG",
    "HL", "WL", "EL", "D",
    "L", "LC", "M", "CY",
    "LM", "DN", "AM", "HT",
}

mask = (
    source_missing
    & dictionary["observed_code"].isin(core_protocol_codes)
)

dictionary.loc[
    mask,
    "resolution_source"
] = "protocol_code"


# ------------------------------------------------------------
# 4. Existing resolved rows lacking finer provenance
#
# This is deliberately generic. It prevents resolved rows from
# appearing as "unresolved" while retaining the fact that the
# exact historical resolution pathway was not recorded.
# ------------------------------------------------------------

source_missing = dictionary["resolution_source"].isna()

mask = (
    source_missing
    & dictionary["code_class"].notna()
)

dictionary.loc[
    mask,
    "resolution_source"
] = "legacy_resolved_pre_provenance"


# ------------------------------------------------------------
# QA
# ------------------------------------------------------------

print(
    dictionary["resolution_source"]
    .value_counts(dropna=False)
)

still_missing = dictionary[
    dictionary["resolution_source"].isna()
]

print(
    f"\nRows still lacking resolution_source: "
    f"{len(still_missing):,}"
)

resolution_source
USDA_accepted_symbol                 7160
legacy_resolved_pre_provenance       2967
USDA_synonym_symbol                   415
<NA>                                  165
project_unknown_code                  130
project_code_pattern                   29
protocol_code                          22
manual_functional_resolution           12
manual_genus_alias                     12
manual_protocol_alias                  10
manual_review_unresolvable              7
manual_project_code_alias               4
manual_unknown_plant                    4
manual_context_resolution               4
manual_typo_correction                  3
manual_taxonomic_alias                  3
manual_cf_taxonomic_alias               2
manual_taxonomic_resolution             2
manual_case_correction                  2
source_blank_sentinel                   1
protocol_basal_vegetation               1
manual_project_code_inferred            1
manual_protocol_interpretation          1
manual_ambiguous

In [20]:
# ============================================================
# USDA ECOLOGICAL ATTRIBUTES -> MOSAIC FUNCTIONAL GROUPS
#
# FINAL CONSERVATIVE CROSSWALK
#
# PRINCIPLES
# ----------
# 1. Preserve explicit source/project functional information.
#
# 2. Preserve PBG wherever PBG is already known.
#
# 3. USDA:
#       Perennial + Graminoid
#            -> PerennialGraminoid
#
#    NOT automatically:
#       Perennial + Graminoid
#            -> PBG
#
# 4. Explicit PBG therefore has precedence over the USDA
#    PerennialGraminoid fallback.
#
# 5. USDA nativity is encoded as:
#       N = Native
#       I = Introduced
#
# 6. Missing USDA traits are distinguished from genuinely
#    unresolved plant identity.
#
# 7. Raw dictionary taxonomy/provenance is preserved.
#
#
# EXPECTED INPUTS
# ---------------
# dictionary
# USDA_ATTRIBUTE_CACHE
#
#
# OUTPUTS
# -------
# dictionary_fg
#
# Important new columns:
#   USDA_duration
#   USDA_growth_habit
#   USDA_native_status_L48
#   USDA_trait_status
#   USDA_derived_group
#   resolved_plant_group
#   MOSAIC_FG
#   FG_resolution_source
#
# QA objects:
#   fg_summary
#   plant_fg_summary
#   trait_recovery_targets
#   ambiguous_fg
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 0. LOAD USDA ECOLOGICAL ATTRIBUTE CACHE
# ============================================================

usda_attributes = pd.read_csv(
    USDA_ATTRIBUTE_CACHE,
    dtype="string",
    low_memory=False,
)

print(
    f"Loaded USDA ecological attribute cache: "
    f"{len(usda_attributes):,} rows"
)


# ============================================================
# 1. NORMALIZE USDA CACHE COLUMN NAMES
#
# Handles both older and current cache schemas.
# ============================================================

usda_attributes = usda_attributes.copy()

column_aliases = {
    # accepted symbol
    "accepted_symbol": "USDA_accepted_symbol",
    "Symbol": "USDA_accepted_symbol",

    # duration
    "USDA_duration_api": "USDA_duration",
    "duration": "USDA_duration",
    "Duration": "USDA_duration",

    # growth habit
    "USDA_growth_habit_api": "USDA_growth_habit",
    "growth_habit": "USDA_growth_habit",
    "GrowthHabit": "USDA_growth_habit",
    "Growth Habit": "USDA_growth_habit",

    # Lower-48 native status
    "USDA_L48_native_status_api": "USDA_native_status_L48",
    "native_status_l48": "USDA_native_status_L48",
    "NativeStatus_L48": "USDA_native_status_L48",
    "L48_native_status": "USDA_native_status_L48",
    "L48_status": "USDA_native_status_L48",
}

for old, new in column_aliases.items():
    if old in usda_attributes.columns and new not in usda_attributes.columns:
        usda_attributes = usda_attributes.rename(columns={old: new})

required_usda_columns = [
    "USDA_accepted_symbol",
    "USDA_duration",
    "USDA_growth_habit",
    "USDA_native_status_L48",
]

missing_usda_columns = [
    col
    for col in required_usda_columns
    if col not in usda_attributes.columns
]

if missing_usda_columns:
    raise ValueError(
        "USDA attribute cache is missing required columns:\n"
        + "\n".join(missing_usda_columns)
        + "\n\nAvailable columns:\n"
        + "\n".join(usda_attributes.columns.astype(str))
    )

print("USDA attribute columns normalized successfully.")
# ============================================================
# 2. CLEAN TEXT VALUES
# ============================================================

def clean_text(x):

    if pd.isna(x):
        return pd.NA

    x = str(x).strip()

    if x == "":
        return pd.NA

    if x.lower() in {
        "nan",
        "<na>",
        "none",
    }:
        return pd.NA

    return x


for col in required_usda_columns:

    usda_attributes[col] = (
        usda_attributes[col]
        .map(clean_text)
        .astype("string")
    )


# ============================================================
# 3. NORMALIZE LOWER-48 NATIVITY
#
# USDA API cache uses primarily:
#
#   N = Native
#   I = Introduced
#
# Convert these to explicit labels before FG classification.
#
# Multi-valued values remain multi-valued.
# ============================================================

def normalize_native_status(x):

    if pd.isna(x):
        return pd.NA

    raw_values = {
        value.strip()
        for value in str(x).split("|")
        if value.strip()
    }

    if not raw_values:
        return pd.NA

    normalized = set()

    for value in raw_values:

        value_upper = value.upper()

        if value_upper in {
            "N",
            "NATIVE",
        }:
            normalized.add("Native")

        elif value_upper in {
            "I",
            "INTRODUCED",
        }:
            normalized.add("Introduced")

        else:
            normalized.add(value)

    return "|".join(
        sorted(normalized)
    )


usda_attributes[
    "USDA_native_status_L48"
] = (
    usda_attributes[
        "USDA_native_status_L48"
    ]
    .map(normalize_native_status)
    .astype("string")
)


# ============================================================
# 4. BUILD ONE USDA TRAIT ROW PER ACCEPTED SYMBOL
# ============================================================

usda_fg_attributes = (
    usda_attributes[
        required_usda_columns
    ]
    .dropna(
        subset=["USDA_accepted_symbol"]
    )
    .copy()
)


# ------------------------------------------------------------
# Verify duplicated accepted symbols do not disagree.
# ------------------------------------------------------------

trait_conflicts = (
    usda_fg_attributes
    .groupby(
        "USDA_accepted_symbol",
        dropna=False,
    )
    .agg(
        n_duration=(
            "USDA_duration",
            lambda x: x.dropna().nunique()
        ),
        n_growth_habit=(
            "USDA_growth_habit",
            lambda x: x.dropna().nunique()
        ),
        n_native_status=(
            "USDA_native_status_L48",
            lambda x: x.dropna().nunique()
        ),
    )
    .reset_index()
)

trait_conflicts = trait_conflicts[
    (trait_conflicts["n_duration"] > 1)
    | (trait_conflicts["n_growth_habit"] > 1)
    | (trait_conflicts["n_native_status"] > 1)
]

if len(trait_conflicts) > 0:

    display(
        trait_conflicts.head(100)
    )

    raise ValueError(
        f"{len(trait_conflicts):,} USDA accepted symbols "
        "have conflicting ecological attributes."
    )


usda_fg_attributes = (
    usda_fg_attributes
    .drop_duplicates(
        subset=["USDA_accepted_symbol"]
    )
    .reset_index(drop=True)
)

print(
    f"Unique USDA accepted taxa available for join: "
    f"{len(usda_fg_attributes):,}"
)


# ============================================================
# 5. START FROM COMPLETED TAXONOMIC DICTIONARY
# ============================================================

dictionary_fg = dictionary.copy()


# ------------------------------------------------------------
# Remove outputs from an earlier run of this cell.
#
# Do NOT remove protocol_plant_group.
# That is precisely where explicit PBG or other known source
# functional information may already exist.
# ------------------------------------------------------------

stale_columns = [
    "USDA_duration",
    "USDA_growth_habit",
    "USDA_native_status_L48",
    "USDA_trait_status",
    "USDA_derived_group",
    "resolved_plant_group",
    "MOSAIC_FG",
    "FG_resolution_source",
]

dictionary_fg = dictionary_fg.drop(
    columns=[
        col
        for col in stale_columns
        if col in dictionary_fg.columns
    ],
    errors="ignore",
)


# ============================================================
# 6. JOIN USDA ECOLOGICAL ATTRIBUTES
# ============================================================

dictionary_fg = dictionary_fg.merge(
    usda_fg_attributes,
    on="USDA_accepted_symbol",
    how="left",
    validate="m:1",
)

print(
    f"Dictionary rows after USDA join: "
    f"{len(dictionary_fg):,}"
)


# ============================================================
# 7. TRAIT SET HELPER
#
# Examples:
#
#   Annual
#       -> {"Annual"}
#
#   Annual|Perennial
#       -> {"Annual", "Perennial"}
#
#   <NA>
#       -> set()
# ============================================================

def trait_set(x):

    if pd.isna(x):
        return set()

    return {
        value.strip()
        for value in str(x).split("|")
        if value.strip()
    }


# ============================================================
# 8. USDA TRAIT COMPLETENESS STATUS
#
# Important distinction:
#
#   No_USDA_taxon
#       taxonomy itself was not USDA-resolved
#
#   Missing_traits
#       USDA taxonomy exists, but ecological attribute
#       retrieval is incomplete
#
# These should NOT be treated as equivalent.
# ============================================================

def derive_trait_status(row):

    accepted = row[
        "USDA_accepted_symbol"
    ]

    if pd.isna(accepted):
        return "No_USDA_taxon"

    duration = pd.notna(
        row["USDA_duration"]
    )

    habit = pd.notna(
        row["USDA_growth_habit"]
    )

    native = pd.notna(
        row["USDA_native_status_L48"]
    )

    if (
        duration
        and habit
        and native
    ):
        return "Complete"

    missing_parts = []

    if not duration:
        missing_parts.append("duration")

    if not habit:
        missing_parts.append("growth_habit")

    if not native:
        missing_parts.append("nativity")

    return (
        "Missing_"
        + "_".join(missing_parts)
    )


dictionary_fg[
    "USDA_trait_status"
] = dictionary_fg.apply(
    derive_trait_status,
    axis=1,
)


# ============================================================
# 9. DERIVE STRUCTURAL/LIFE-HISTORY GROUP FROM USDA
#
# CONSERVATIVE RULE:
#
#   Perennial + Graminoid
#       -> PerennialGraminoid
#
# It does NOT become PBG here.
#
# Explicit PBG, if already present in protocol_plant_group,
# will be preserved later.
# ============================================================

def derive_usda_group(row):

    code_class = row[
        "code_class"
    ]

    # --------------------------------------------------------
    # NA-safe plant check
    # --------------------------------------------------------

    if pd.isna(code_class):
        return pd.NA

    if code_class != "Plant":
        return pd.NA


    duration = trait_set(
        row["USDA_duration"]
    )

    habit = trait_set(
        row["USDA_growth_habit"]
    )


    # --------------------------------------------------------
    # Cannot infer structural group without habit.
    # --------------------------------------------------------

    if not habit:
        return pd.NA


    # ========================================================
    # GRAMINOIDS
    # ========================================================

    if habit == {"Graminoid"}:

        if duration == {"Annual"}:
            return "AnnualGraminoid"

        if duration == {"Perennial"}:
            return "PerennialGraminoid"

        # Duration ambiguous or absent, but graminoid
        # structure itself is known.
        return "Graminoid"


    # ========================================================
    # FORBS / HERBS
    # ========================================================

    forb_terms = {
        "Forb/herb",
        "Forb",
        "Herb",
    }

    if habit.issubset(
        forb_terms
    ):

        if duration == {"Annual"}:
            return "AnnualForb"

        if duration == {"Perennial"}:
            return "PerennialForb"

        return "Forb"


    # ========================================================
    # SHRUB
    # ========================================================

    if habit == {"Shrub"}:
        return "Shrub"


    # ========================================================
    # SUBSHRUB
    # ========================================================

    if habit == {"Subshrub"}:
        return "Subshrub"


    # ========================================================
    # SHRUB / SUBSHRUB
    #
    # Both remain within shrub-domain vegetation for the
    # current broad MOSAIC FG representation.
    # ========================================================

    if habit and habit.issubset(
        {
            "Shrub",
            "Subshrub",
        }
    ):
        return "Shrub"


    # ========================================================
    # TREE
    # ========================================================

    if habit == {"Tree"}:
        return "Tree"


    # ========================================================
    # SHRUB / TREE
    #
    # Preserve ambiguity rather than arbitrarily selecting one.
    # ========================================================

    if habit and habit.issubset(
        {
            "Shrub",
            "Tree",
        }
    ):
        return "Woody"


    # ========================================================
    # FORB / SUBSHRUB
    # ========================================================

    if habit and habit.issubset(
        {
            "Forb/herb",
            "Forb",
            "Herb",
            "Subshrub",
        }
    ):
        return "SubshrubOrForb"


    # ========================================================
    # More complex combinations stay unresolved.
    # ========================================================

    return pd.NA


dictionary_fg[
    "USDA_derived_group"
] = dictionary_fg.apply(
    derive_usda_group,
    axis=1,
)


# ============================================================
# 10. RESOLVE SOURCE + USDA FUNCTIONAL INFORMATION
#
# PRECEDENCE
# ----------
#
#   explicit protocol/project functional group
#                >
#   USDA-derived group
#
#
# Therefore:
#
# protocol_plant_group = PBG
# USDA_derived_group   = PerennialGraminoid
#
# resolves to:
#
#   PBG
#
#
# But:
#
# protocol_plant_group = NA
# USDA_derived_group   = PerennialGraminoid
#
# resolves to:
#
#   PerennialGraminoid
#
# This is the intended asymmetry.
# ============================================================

dictionary_fg[
    "resolved_plant_group"
] = (
    dictionary_fg[
        "protocol_plant_group"
    ]
    .combine_first(
        dictionary_fg[
            "USDA_derived_group"
        ]
    )
)


# ============================================================
# 11. FINAL MOSAIC FUNCTIONAL GROUP
#
# Current working crosswalk:
#
#   PBG                         -> PBG
#
#   AnnualGraminoid + I         -> EAG
#   AnnualGraminoid + N         -> NAG
#
#   PerennialGraminoid          -> PerennialGraminoid
#
#   AnnualForb                  -> AF
#   PerennialForb               -> PF
#
#   Shrub                       -> SHR
#   Subshrub                    -> SUBSHR
#   Tree                        -> TREE
#
# USDA-resolved taxa missing traits are explicitly marked
# TraitUnresolvedPlant rather than collapsed into UnknownPlant.
# ============================================================

def derive_mosaic_fg(row):

    code_class = row[
        "code_class"
    ]

    # --------------------------------------------------------
    # NA-safe plant check
    # --------------------------------------------------------

    if pd.isna(code_class):
        return pd.NA

    if code_class != "Plant":
        return pd.NA


    group = row[
        "resolved_plant_group"
    ]

    native = trait_set(
        row[
            "USDA_native_status_L48"
        ]
    )


    # ========================================================
    # PLANT WITH NO RESOLVED FUNCTIONAL GROUP
    # ========================================================

    if pd.isna(group):

        # Taxonomy is known, traits are incomplete.
        if pd.notna(
            row["USDA_accepted_symbol"]
        ):
            return "TraitUnresolvedPlant"

        # Taxonomy itself is unresolved.
        return "UnknownPlant"


    # ========================================================
    # PRESERVE KNOWN PBG
    # ========================================================

    if group == "PBG":
        return "PBG"


    # Also tolerate expanded source labels if any occur.
    if group in {
        "PerennialBunchgrass",
        "Perennial bunchgrass",
    }:
        return "PBG"


    # ========================================================
    # ANNUAL GRAMINOIDS
    # ========================================================

    if group == "AnnualGraminoid":

        if native == {"Introduced"}:
            return "EAG"

        if native == {"Native"}:
            return "NAG"

        return "AnnualGraminoid_UnknownNativity"


    # ========================================================
    # PERENNIAL GRAMINOIDS
    #
    # IMPORTANT:
    # these remain PerennialGraminoid unless independently
    # identified as PBG above.
    # ========================================================

    if group == "PerennialGraminoid":
        return "PerennialGraminoid"


    if group == "Graminoid":
        return "Graminoid"


    # ========================================================
    # FORBS
    # ========================================================

    if group == "AnnualForb":
        return "AF"

    if group == "PerennialForb":
        return "PF"

    if group == "Forb":
        return "Forb"


    # ========================================================
    # SHRUB / WOODY
    # ========================================================

    if group == "Shrub":
        return "SHR"

    if group == "Subshrub":
        return "SUBSHR"

    if group == "Tree":
        return "TREE"

    if group == "Woody":
        return "Woody"


    # ========================================================
    # EXISTING BROAD SOURCE CLASSES
    # ========================================================

    if group == "SubshrubOrPerennialForb":
        return "SubshrubOrPF"

    if group == "SubshrubOrForb":
        return "SubshrubOrForb"

    if group == "AnnualPlant":
        return "AnnualPlant"

    if group == "UnknownPlant":
        return "UnknownPlant"


    # ========================================================
    # Preserve any unanticipated explicit project/source group.
    #
    # Do not silently destroy information.
    # ========================================================

    return group


dictionary_fg[
    "MOSAIC_FG"
] = dictionary_fg.apply(
    derive_mosaic_fg,
    axis=1,
)


# ============================================================
# 12. FUNCTIONAL-GROUP PROVENANCE
# ============================================================

dictionary_fg[
    "FG_resolution_source"
] = pd.NA


# ------------------------------------------------------------
# Explicit source/project group
# ------------------------------------------------------------

source_group_mask = (
    dictionary_fg[
        "protocol_plant_group"
    ].notna()
)

dictionary_fg.loc[
    source_group_mask,
    "FG_resolution_source"
] = "protocol_or_project_group"


# ------------------------------------------------------------
# Specifically flag PBG provenance
# ------------------------------------------------------------

pbg_source_mask = (
    dictionary_fg[
        "protocol_plant_group"
    ].isin(
        {
            "PBG",
            "PerennialBunchgrass",
            "Perennial bunchgrass",
        }
    )
)

dictionary_fg.loc[
    pbg_source_mask,
    "FG_resolution_source"
] = "explicit_PBG_source"


# ------------------------------------------------------------
# USDA traits supplied the functional group
# ------------------------------------------------------------

usda_group_mask = (
    dictionary_fg[
        "protocol_plant_group"
    ].isna()
    & dictionary_fg[
        "USDA_derived_group"
    ].notna()
)

dictionary_fg.loc[
    usda_group_mask,
    "FG_resolution_source"
] = "USDA_traits"


# ------------------------------------------------------------
# USDA taxonomy exists, but traits are incomplete
# ------------------------------------------------------------

trait_unresolved_mask = (
    dictionary_fg[
        "code_class"
    ].eq("Plant")
    & dictionary_fg[
        "USDA_accepted_symbol"
    ].notna()
    & dictionary_fg[
        "resolved_plant_group"
    ].isna()
)

dictionary_fg.loc[
    trait_unresolved_mask,
    "FG_resolution_source"
] = "USDA_taxon_missing_traits"


# ------------------------------------------------------------
# Plant identity/function genuinely unresolved
# ------------------------------------------------------------

unknown_plant_mask = (
    dictionary_fg[
        "code_class"
    ].eq("Plant")
    & dictionary_fg[
        "USDA_accepted_symbol"
    ].isna()
    & dictionary_fg[
        "resolved_plant_group"
    ].isna()
)

dictionary_fg.loc[
    unknown_plant_mask,
    "FG_resolution_source"
] = "unresolved_plant_group"


# ============================================================
# 13. QA — BRTE
#
# Should now be:
#
#   Annual
#   Graminoid
#   Introduced
#   AnnualGraminoid
#   EAG
# ============================================================

print()
print("=" * 90)
print("BRTE QA")
print("=" * 90)

qa_columns = [
    "observed_code",
    "n_records",
    "USDA_accepted_symbol",
    "USDA_scientific_name",
    "USDA_duration",
    "USDA_growth_habit",
    "USDA_native_status_L48",
    "protocol_plant_group",
    "USDA_derived_group",
    "resolved_plant_group",
    "MOSAIC_FG",
    "FG_resolution_source",
]

qa_columns = [
    col
    for col in qa_columns
    if col in dictionary_fg.columns
]

display(
    dictionary_fg.loc[
        dictionary_fg[
            "observed_code"
        ].eq("BRTE"),
        qa_columns,
    ]
)


# ============================================================
# 14. QA — EXISTING PBG
#
# Verify no known PBG was degraded to PerennialGraminoid.
# ============================================================

print()
print("=" * 90)
print("EXPLICIT PBG PRESERVATION QA")
print("=" * 90)

pbg_qa = (
    dictionary_fg.loc[
        dictionary_fg[
            "protocol_plant_group"
        ].isin(
            {
                "PBG",
                "PerennialBunchgrass",
                "Perennial bunchgrass",
            }
        )
    ]
    .copy()
)

print(
    f"Explicit PBG codes found: "
    f"{len(pbg_qa):,}"
)

if len(pbg_qa) > 0:

    print(
        f"Records represented: "
        f"{pbg_qa['n_records'].sum():,}"
    )

    display(
        pbg_qa[
            qa_columns
        ]
        .sort_values(
            "n_records",
            ascending=False,
        )
        .head(100)
    )

    bad_pbg = (
        pbg_qa[
            "MOSAIC_FG"
        ] != "PBG"
    )

    if bad_pbg.any():

        raise ValueError(
            "At least one explicitly identified PBG was "
            "not preserved as MOSAIC_FG='PBG'."
        )

    print(
        "PASS: all explicit PBG classifications were preserved."
    )

else:

    print(
        "No explicit PBG values currently occur in "
        "protocol_plant_group."
    )


# ============================================================
# 15. QA — USDA PERENNIAL GRAMINOID FALLBACK
#
# These should remain PerennialGraminoid unless source data
# independently identified them as PBG.
# ============================================================

print()
print("=" * 90)
print("PERENNIAL GRAMINOID FALLBACK QA")
print("=" * 90)

perennial_graminoid_qa = (
    dictionary_fg.loc[
        dictionary_fg[
            "MOSAIC_FG"
        ].eq("PerennialGraminoid")
    ]
    .sort_values(
        "n_records",
        ascending=False,
    )
)

print(
    f"PerennialGraminoid codes: "
    f"{len(perennial_graminoid_qa):,}"
)

print(
    f"Records represented: "
    f"{perennial_graminoid_qa['n_records'].sum():,}"
)

display(
    perennial_graminoid_qa[
        qa_columns
    ].head(50)
)


# ============================================================
# 16. QA — BAPR VARIANTS
#
# Preserve whatever manually reviewed protocol/project group
# has already been assigned upstream.
#
# This cell DOES NOT overwrite the user's existing BAPR
# functional decision.
# ============================================================

print()
print("=" * 90)
print("BAPR VARIANT QA")
print("=" * 90)

display(
    dictionary_fg.loc[
        dictionary_fg[
            "observed_code"
        ].isin(
            [
                "BAPR",
                "BAPRV",
                "BAPRG",
            ]
        ),
        qa_columns,
    ]
    .sort_values(
        "n_records",
        ascending=False,
    )
)


# ============================================================
# 17. TRAIT RECOVERY TARGETS
#
# These are USDA-resolved plant taxa for which the cached
# ecological attributes are incomplete.
#
# Sort by record weight so recovery effort targets the taxa
# that matter most to the national dataset.
# ============================================================

trait_recovery_targets = (
    dictionary_fg.loc[
        dictionary_fg[
            "code_class"
        ].eq("Plant")
        & dictionary_fg[
            "USDA_accepted_symbol"
        ].notna()
        & (
            dictionary_fg[
                "USDA_duration"
            ].isna()
            | dictionary_fg[
                "USDA_growth_habit"
            ].isna()
            | dictionary_fg[
                "USDA_native_status_L48"
            ].isna()
        )
    ]
    .copy()
)

trait_recovery_targets = (
    trait_recovery_targets
    .sort_values(
        "n_records",
        ascending=False,
    )
    .reset_index(drop=True)
)


print()
print("=" * 90)
print("USDA TRAIT RECOVERY TARGETS")
print("=" * 90)

print(
    f"Dictionary codes requiring >=1 USDA trait: "
    f"{len(trait_recovery_targets):,}"
)

print(
    f"Records represented: "
    f"{trait_recovery_targets['n_records'].sum():,}"
)

recovery_columns = [
    "observed_code",
    "n_records",
    "n_plot_visits",
    "USDA_accepted_symbol",
    "USDA_scientific_name",
    "USDA_duration",
    "USDA_growth_habit",
    "USDA_native_status_L48",
    "USDA_trait_status",
    "protocol_plant_group",
    "MOSAIC_FG",
    "FG_resolution_source",
]

recovery_columns = [
    col
    for col in recovery_columns
    if col in trait_recovery_targets.columns
]

display(
    trait_recovery_targets[
        recovery_columns
    ].head(100)
)


# ============================================================
# 18. UNIQUE USDA TAXA REQUIRING TRAIT RECOVERY
#
# This is the actual lookup target list.
# ============================================================

trait_recovery_taxa = (
    trait_recovery_targets[
        [
            "USDA_accepted_symbol",
            "USDA_scientific_name",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "USDA_accepted_symbol"
    )
    .reset_index(drop=True)
)

print(
    f"\nUnique USDA taxa requiring supplemental trait lookup: "
    f"{len(trait_recovery_taxa):,}"
)


# ============================================================
# 19. FUNCTIONAL-GROUP SUMMARY — ALL DICTIONARY RECORDS
# ============================================================

N_TOTAL_RECORDS = int(
    dictionary_fg[
        "n_records"
    ].sum()
)

fg_summary = (
    dictionary_fg
    .assign(
        MOSAIC_FG_display=(
            dictionary_fg[
                "MOSAIC_FG"
            ]
            .astype("string")
            .fillna("<NONPLANT_OR_UNCLASSIFIED>")
        )
    )
    .groupby(
        "MOSAIC_FG_display",
        dropna=False,
    )
    .agg(
        n_codes=(
            "observed_code",
            "size",
        ),
        n_records=(
            "n_records",
            "sum",
        ),
    )
    .reset_index()
)

fg_summary[
    "codes_pct"
] = (
    100
    * fg_summary["n_codes"]
    / len(dictionary_fg)
)

fg_summary[
    "records_pct"
] = (
    100
    * fg_summary["n_records"]
    / N_TOTAL_RECORDS
)

fg_summary = (
    fg_summary
    .sort_values(
        "n_records",
        ascending=False,
    )
    .reset_index(drop=True)
)

print()
print("=" * 90)
print("MOSAIC FUNCTIONAL-GROUP SUMMARY — ALL RECORDS")
print("=" * 90)

display(
    fg_summary.style.format(
        {
            "codes_pct": "{:.2f}",
            "records_pct": "{:.4f}",
        }
    )
)


# ============================================================
# 20. PLANT-ONLY FUNCTIONAL-GROUP SUMMARY
# ============================================================

plant_fg = (
    dictionary_fg.loc[
        dictionary_fg[
            "code_class"
        ].eq("Plant")
    ]
    .copy()
)

N_PLANT_RECORDS = int(
    plant_fg[
        "n_records"
    ].sum()
)

plant_fg_summary = (
    plant_fg
    .assign(
        MOSAIC_FG_display=(
            plant_fg[
                "MOSAIC_FG"
            ]
            .astype("string")
            .fillna("<UNRESOLVED>")
        )
    )
    .groupby(
        "MOSAIC_FG_display",
        dropna=False,
    )
    .agg(
        n_codes=(
            "observed_code",
            "size",
        ),
        n_records=(
            "n_records",
            "sum",
        ),
    )
    .reset_index()
)

plant_fg_summary[
    "plant_codes_pct"
] = (
    100
    * plant_fg_summary["n_codes"]
    / len(plant_fg)
)

plant_fg_summary[
    "plant_records_pct"
] = (
    100
    * plant_fg_summary["n_records"]
    / N_PLANT_RECORDS
)

plant_fg_summary = (
    plant_fg_summary
    .sort_values(
        "n_records",
        ascending=False,
    )
    .reset_index(drop=True)
)

print()
print("=" * 90)
print("PLANT-ONLY FUNCTIONAL-GROUP SUMMARY")
print("=" * 90)

display(
    plant_fg_summary.style.format(
        {
            "plant_codes_pct": "{:.2f}",
            "plant_records_pct": "{:.4f}",
        }
    )
)


# ============================================================
# 21. FUNCTIONAL-GROUP PROVENANCE SUMMARY
# ============================================================

fg_source_summary = (
    plant_fg
    .assign(
        FG_source_display=(
            plant_fg[
                "FG_resolution_source"
            ]
            .astype("string")
            .fillna("<UNRESOLVED>")
        )
    )
    .groupby(
        "FG_source_display",
        dropna=False,
    )
    .agg(
        n_codes=(
            "observed_code",
            "size",
        ),
        n_records=(
            "n_records",
            "sum",
        ),
    )
    .reset_index()
)

fg_source_summary[
    "plant_records_pct"
] = (
    100
    * fg_source_summary["n_records"]
    / N_PLANT_RECORDS
)

fg_source_summary = (
    fg_source_summary
    .sort_values(
        "n_records",
        ascending=False,
    )
    .reset_index(drop=True)
)

print()
print("=" * 90)
print("FUNCTIONAL-GROUP RESOLUTION SOURCE")
print("=" * 90)

display(
    fg_source_summary.style.format(
        {
            "plant_records_pct": "{:.4f}",
        }
    )
)


# ============================================================
# 22. AMBIGUOUS BUT INFORMATIVE GROUPS
#
# These are intentionally not forced into narrower classes.
# ============================================================

ambiguous_groups = {
    "Graminoid",
    "Forb",
    "Woody",
    "SubshrubOrForb",
    "SubshrubOrPF",
    "AnnualPlant",
    "AnnualGraminoid_UnknownNativity",
    "TraitUnresolvedPlant",
}

ambiguous_fg = (
    dictionary_fg.loc[
        dictionary_fg[
            "MOSAIC_FG"
        ].isin(
            ambiguous_groups
        )
    ]
    .sort_values(
        "n_records",
        ascending=False,
    )
    .reset_index(drop=True)
)

print()
print("=" * 90)
print("AMBIGUOUS / TRAIT-INCOMPLETE FUNCTIONAL GROUPS")
print("=" * 90)

print(
    f"Codes:   "
    f"{len(ambiguous_fg):,}"
)

print(
    f"Records: "
    f"{ambiguous_fg['n_records'].sum():,}"
)

display(
    ambiguous_fg[
        [
            col
            for col in [
                "observed_code",
                "n_records",
                "USDA_accepted_symbol",
                "USDA_scientific_name",
                "USDA_duration",
                "USDA_growth_habit",
                "USDA_native_status_L48",
                "USDA_trait_status",
                "protocol_plant_group",
                "USDA_derived_group",
                "resolved_plant_group",
                "MOSAIC_FG",
                "FG_resolution_source",
            ]
            if col in ambiguous_fg.columns
        ]
    ].head(100)
)


# ============================================================
# 23. TRUE UNKNOWN PLANTS
#
# These lack both a resolved source group and usable USDA
# taxonomy/traits.
# ============================================================

unknown_plants = (
    dictionary_fg.loc[
        dictionary_fg[
            "code_class"
        ].eq("Plant")
        & dictionary_fg[
            "MOSAIC_FG"
        ].eq("UnknownPlant")
    ]
    .sort_values(
        "n_records",
        ascending=False,
    )
    .reset_index(drop=True)
)

print()
print("=" * 90)
print("TRUE UNKNOWN PLANTS")
print("=" * 90)

print(
    f"UnknownPlant codes: "
    f"{len(unknown_plants):,}"
)

print(
    f"Records represented: "
    f"{unknown_plants['n_records'].sum():,}"
)


# ============================================================
# 24. FINAL PBG/PERENNIAL-GRAMINOID AUDIT
# ============================================================

n_pbg_codes = int(
    plant_fg[
        "MOSAIC_FG"
    ].eq("PBG").sum()
)

n_pbg_records = int(
    plant_fg.loc[
        plant_fg[
            "MOSAIC_FG"
        ].eq("PBG"),
        "n_records",
    ].sum()
)

n_pg_codes = int(
    plant_fg[
        "MOSAIC_FG"
    ].eq(
        "PerennialGraminoid"
    ).sum()
)

n_pg_records = int(
    plant_fg.loc[
        plant_fg[
            "MOSAIC_FG"
        ].eq(
            "PerennialGraminoid"
        ),
        "n_records",
    ].sum()
)

print()
print("=" * 90)
print("PBG / PERENNIAL GRAMINOID AUDIT")
print("=" * 90)

print(
    f"Explicit PBG:             "
    f"{n_pbg_codes:,} codes | "
    f"{n_pbg_records:,} records"
)

print(
    f"PerennialGraminoid:       "
    f"{n_pg_codes:,} codes | "
    f"{n_pg_records:,} records"
)

print()
print(
    "Interpretation:"
)

print(
    "  PBG = bunchgrass identity already known from an "
    "explicit source/project classification."
)

print(
    "  PerennialGraminoid = perennial graminoid supported "
    "by available information, but bunchgrass architecture "
    "has not yet been independently established."
)


# ============================================================
# 25. FINAL COVERAGE AUDIT
# ============================================================

resolved_mask = (
    plant_fg[
        "MOSAIC_FG"
    ].notna()
)

n_resolved_codes = int(
    resolved_mask.sum()
)

n_resolved_records = int(
    plant_fg.loc[
        resolved_mask,
        "n_records",
    ].sum()
)

print()
print("=" * 90)
print("FINAL FUNCTIONAL-GROUP COVERAGE AUDIT")
print("=" * 90)

print(
    f"Plant codes with MOSAIC_FG:    "
    f"{n_resolved_codes:,} / {len(plant_fg):,} "
    f"({100*n_resolved_codes/len(plant_fg):.2f}%)"
)

print(
    f"Plant records with MOSAIC_FG:  "
    f"{n_resolved_records:,} / {N_PLANT_RECORDS:,} "
    f"({100*n_resolved_records/N_PLANT_RECORDS:.5f}%)"
)

print(
    f"Plant records lacking any FG:  "
    f"{N_PLANT_RECORDS - n_resolved_records:,}"
)

print("=" * 90)

Loaded USDA ecological attribute cache: 7,268 rows
USDA attribute columns normalized successfully.
Unique USDA accepted taxa available for join: 7,267
Dictionary rows after USDA join: 10,960

BRTE QA


,observed_code,n_records,USDA_accepted_symbol,USDA_scientific_name,USDA_duration,USDA_growth_habit,USDA_native_status_L48,protocol_plant_group,USDA_derived_group,resolved_plant_group,MOSAIC_FG,FG_resolution_source
5,BRTE,429816,BRTE,Bromus tectorum L.,Annual,Graminoid,Introduced,<NA>,AnnualGraminoid,AnnualGraminoid,EAG,USDA_traits



EXPLICIT PBG PRESERVATION QA
Explicit PBG codes found: 0
No explicit PBG values currently occur in protocol_plant_group.

PERENNIAL GRAMINOID FALLBACK QA
PerennialGraminoid codes: 1,492
Records represented: 2,267,084


,observed_code,n_records,USDA_accepted_symbol,USDA_scientific_name,USDA_duration,USDA_growth_habit,USDA_native_status_L48,protocol_plant_group,USDA_derived_group,resolved_plant_group,MOSAIC_FG,FG_resolution_source
4,POSE,332740,POSE,Poa secunda J. Presl,Perennial,Graminoid,Native,<NA>,PerennialGraminoid,PerennialGraminoid,PerennialGraminoid,USDA_traits
10,PSSP6,163393,PSSP6,Pseudoroegneria spicata (Pursh) Á. Löve,Perennial,Graminoid,Native,<NA>,PerennialGraminoid,PerennialGraminoid,PerennialGraminoid,USDA_traits
16,BOGR2,159606,BOGR2,Bouteloua gracilis (Willd. ex Kunth) Lag. ex G...,Perennial,Graminoid,Native,<NA>,PerennialGraminoid,PerennialGraminoid,PerennialGraminoid,USDA_traits
17,PASM,136002,PASM,Pascopyrum smithii (Rydb.) Á. Löve,Perennial,Graminoid,Native,<NA>,PerennialGraminoid,PerennialGraminoid,PerennialGraminoid,USDA_traits
29,POPR,115614,POPR,Poa pratensis L.,Perennial,Graminoid,Introduced,<NA>,PerennialGraminoid,PerennialGraminoid,PerennialGraminoid,USDA_traits
31,AGCR,86637,AGCR,Agropyron cristatum (L.) Gaertn.,Perennial,Graminoid,Introduced,<NA>,PerennialGraminoid,PerennialGraminoid,PerennialGraminoid,USDA_traits
19,HECO26,80033,HECO26,Hesperostipa comata (Trin. & Rupr.) Barkworth,Perennial,Graminoid,Native,<NA>,PerennialGraminoid,PerennialGraminoid,PerennialGraminoid,USDA_traits
34,FEID,78604,FEID,Festuca idahoensis Elmer,Perennial,Graminoid,Native,<NA>,PerennialGraminoid,PerennialGraminoid,PerennialGraminoid,USDA_traits
75,Perennial grasses,61012,<NA>,<NA>,<NA>,<NA>,<NA>,PerennialGraminoid,NaN,PerennialGraminoid,PerennialGraminoid,protocol_or_project_group
9,ELEL5,58454,ELEL5,Elymus elymoides (Raf.) Swezey,Perennial,Graminoid,Native,<NA>,PerennialGraminoid,PerennialGraminoid,PerennialGraminoid,USDA_traits



BAPR VARIANT QA


...


FUNCTIONAL-GROUP RESOLUTION SOURCE


,FG_source_display,n_codes,n_records,plant_records_pct
0,USDA_traits,6880,5095277,93.7262
1,protocol_or_project_group,3149,219350,4.0349
2,USDA_taxon_missing_traits,720,121714,2.2389



AMBIGUOUS / TRAIT-INCOMPLETE FUNCTIONAL GROUPS
Codes:   2,718
Records: 1,105,011


,observed_code,n_records,USDA_accepted_symbol,USDA_scientific_name,USDA_duration,USDA_growth_habit,USDA_native_status_L48,USDA_trait_status,protocol_plant_group,USDA_derived_group,resolved_plant_group,MOSAIC_FG,FG_resolution_source
0,ARTRW8,232161,ARTRW8,Artemisia tridentata Nutt. ssp. wyomingensis B...,Perennial,Shrub|Tree,Native,Complete,<NA>,Woody,Woody,Woody,USDA_traits
1,ARTRV,79491,ARTRV,Artemisia tridentata Nutt. ssp. vaseyana (Rydb...,Perennial,Shrub|Tree,Native,Complete,<NA>,Woody,Woody,Woody,USDA_traits
2,Annual plants,35750,<NA>,<NA>,<NA>,<NA>,<NA>,No_USDA_taxon,AnnualPlant,NaN,AnnualPlant,AnnualPlant,protocol_or_project_group
3,PRGL2,32583,PRGL2,Prosopis glandulosa Torr.,Perennial,Shrub|Tree,Native,Complete,<NA>,Woody,Woody,Woody,USDA_traits
4,ARTRT,30220,ARTRT,Artemisia tridentata Nutt. ssp. tridentata,Perennial,Shrub|Tree,Native,Complete,<NA>,Woody,Woody,Woody,USDA_traits
5,ARTR2,27981,ARTR2,Artemisia tridentata Nutt.,Perennial,Shrub|Tree,Native,Complete,<NA>,Woody,Woody,Woody,USDA_traits
6,GUSA2,27951,GUSA2,Gutierrezia sarothrae (Pursh) Britton & Rusby,Perennial,Forb/herb|Shrub|Subshrub,Native,Complete,<NA>,NaN,<NA>,TraitUnresolvedPlant,USDA_taxon_missing_traits
7,QUGA,24748,QUGA,Quercus gambelii Nutt.,Perennial,Shrub|Tree,Native,Complete,<NA>,Woody,Woody,Woody,USDA_traits
8,ARPU9,21232,ARPU9,Aristida purpurea Nutt.,Annual|Perennial,Graminoid,Native,Complete,<NA>,Graminoid,Graminoid,Graminoid,USDA_traits
9,Sub-shrubs and perennial forbs,20140,<NA>,<NA>,<NA>,<NA>,<NA>,No_USDA_taxon,SubshrubOrPerennialForb,NaN,SubshrubOrPerennialForb,SubshrubOrPF,protocol_or_project_group



TRUE UNKNOWN PLANTS
UnknownPlant codes: 142
Records represented: 21,114

PBG / PERENNIAL GRAMINOID AUDIT
Explicit PBG:             0 codes | 0 records
PerennialGraminoid:       1,492 codes | 2,267,084 records

Interpretation:
  PBG = bunchgrass identity already known from an explicit source/project classification.
  PerennialGraminoid = perennial graminoid supported by available information, but bunchgrass architecture has not yet been independently established.

FINAL FUNCTIONAL-GROUP COVERAGE AUDIT
Plant codes with MOSAIC_FG:    10,749 / 10,749 (100.00%)
Plant records with MOSAIC_FG:  5,436,341 / 5,436,341 (100.00000%)
Plant records lacking any FG:  0


In [21]:
# ============================================================
# DIAGNOSE USDA TRAIT-MISSINGNESS
#
# Separates:
#
#   1. accepted USDA symbol absent from trait cache
#   2. symbol present in cache but trait fields are blank
#
# This tells us whether we need:
#   - additional taxa queried
#   - or failed/incomplete taxa re-queried
# ============================================================

# Normalize cache symbol column if needed
usda_cache_check = pd.read_csv(
    USDA_ATTRIBUTE_CACHE,
    dtype="string",
    low_memory=False,
).copy()

if (
    "accepted_symbol" in usda_cache_check.columns
    and "USDA_accepted_symbol" not in usda_cache_check.columns
):
    usda_cache_check = usda_cache_check.rename(
        columns={"accepted_symbol": "USDA_accepted_symbol"}
    )


# ------------------------------------------------------------
# Basic cache membership
# ------------------------------------------------------------

cache_symbols = set(
    usda_cache_check[
        "USDA_accepted_symbol"
    ]
    .dropna()
    .astype(str)
    .str.strip()
)


dictionary_fg["USDA_symbol_in_trait_cache"] = (
    dictionary_fg[
        "USDA_accepted_symbol"
    ]
    .astype("string")
    .isin(cache_symbols)
)


# ------------------------------------------------------------
# Examine the important unresolved taxa
# ------------------------------------------------------------

check_codes = [
    "TRDU",
    "PSSP6",
    "POPR",
    "AGCR",
    "HECO26",
    "FEID",
    "ELEL5",
    "BRIN2",
    "CHVI8",
    "SCAR7",
    "TACA8",
    "ALDE",
    "ARTR2",
    "SIAL2",
    "ERCI6",
]


diagnostic = (
    dictionary_fg.loc[
        dictionary_fg["observed_code"].isin(check_codes),
        [
            "observed_code",
            "n_records",
            "USDA_accepted_symbol",
            "USDA_scientific_name",
            "USDA_symbol_in_trait_cache",
            "USDA_duration",
            "USDA_growth_habit",
            "USDA_native_status_L48",
            "USDA_trait_status",
        ],
    ]
    .sort_values(
        "n_records",
        ascending=False,
    )
)

display(diagnostic)

,observed_code,n_records,USDA_accepted_symbol,USDA_scientific_name,USDA_symbol_in_trait_cache,USDA_duration,USDA_growth_habit,USDA_native_status_L48,USDA_trait_status
10,PSSP6,163393,PSSP6,Pseudoroegneria spicata (Pursh) Á. Löve,True,Perennial,Graminoid,Native,Complete
29,POPR,115614,POPR,Poa pratensis L.,True,Perennial,Graminoid,Introduced,Complete
31,AGCR,86637,AGCR,Agropyron cristatum (L.) Gaertn.,True,Perennial,Graminoid,Introduced,Complete
19,HECO26,80033,HECO26,Hesperostipa comata (Trin. & Rupr.) Barkworth,True,Perennial,Graminoid,Native,Complete
34,FEID,78604,FEID,Festuca idahoensis Elmer,True,Perennial,Graminoid,Native,Complete
9,ELEL5,58454,ELEL5,Elymus elymoides (Raf.) Swezey,True,Perennial,Graminoid,Native,Complete
63,BRIN2,56601,BRIN2,Bromus inermis Leyss.,True,Perennial,Graminoid,Introduced,Complete
12,CHVI8,52275,CHVI8,Chrysothamnus viscidiflorus (Hook.) Nutt.,True,Perennial,Shrub,Native,Complete
109,SCAR7,46213,SCAR7,"Schedonorus arundinaceus (Schreb.) Dumort., no...",True,Perennial,Graminoid,Introduced,Complete
82,ARTR2,27981,ARTR2,Artemisia tridentata Nutt.,True,Perennial,Shrub|Tree,Native,Complete


In [22]:
display(
    usda_cache_check.loc[
        usda_cache_check[
            "USDA_accepted_symbol"
        ].eq("TRDU")
    ]
)

,USDA_accepted_symbol,USDA_plant_id,USDA_rank,USDA_duration_api,USDA_growth_habit_api,USDA_L48_native_status_api,USDA_L48_native_type_api,retrieved_utc
6890,TRDU,41535.0,Species,Annual|Biennial,Forb/herb,I,Introduced,2026-09-01T20:26:28.390058+00:00


In [23]:
# ============================================================
# FIND USDA-RESOLVED TAXA MISSING FROM ECOLOGICAL TRAIT CACHE
# ============================================================

# Current cache symbols
cache_symbols = set(
    usda_attributes["USDA_accepted_symbol"]
    .dropna()
    .astype(str)
    .str.strip()
)

# Current dictionary USDA symbols
dictionary_symbols = set(
    dictionary_fg["USDA_accepted_symbol"]
    .dropna()
    .astype(str)
    .str.strip()
)

missing_trait_symbols = sorted(
    dictionary_symbols - cache_symbols
)

print(f"USDA taxa represented in dictionary: {len(dictionary_symbols):,}")
print(f"USDA taxa already in trait cache:      {len(cache_symbols):,}")
print(f"USDA taxa missing from trait cache:    {len(missing_trait_symbols):,}")

USDA taxa represented in dictionary: 7,268
USDA taxa already in trait cache:      7,267
USDA taxa missing from trait cache:    2


In [24]:
missing_trait_taxa = (
    dictionary_fg.loc[
        dictionary_fg["USDA_accepted_symbol"].isin(missing_trait_symbols),
        [
            "USDA_accepted_symbol",
            "USDA_scientific_name",
            "n_records",
        ],
    ]
    .groupby(
        ["USDA_accepted_symbol", "USDA_scientific_name"],
        dropna=False,
        as_index=False,
    )
    ["n_records"]
    .sum()
    .sort_values(
        "n_records",
        ascending=False,
    )
)

display(missing_trait_taxa.head(100))

print(
    "\nRecords represented by taxa missing from trait cache:",
    f"{missing_trait_taxa['n_records'].sum():,}"
)

,USDA_accepted_symbol,USDA_scientific_name,n_records
0,NONE,Notholaena neglecta Maxon,398
1,PONIN,Potentilla nivea L. var. nivea,10



Records represented by taxa missing from trait cache: 408


In [25]:
# ============================================================
# TRAIT CACHE COVERAGE GATE
# ============================================================

plant_usda = dictionary_fg.loc[
    dictionary_fg["code_class"].eq("Plant")
    & dictionary_fg["USDA_accepted_symbol"].notna()
].copy()

plant_usda["has_trait_cache"] = (
    plant_usda["USDA_accepted_symbol"]
    .isin(
        set(
            usda_attributes["USDA_accepted_symbol"]
            .dropna()
            .astype(str)
            .str.strip()
        )
    )
)

code_coverage = plant_usda["has_trait_cache"].mean()

record_coverage = (
    plant_usda.loc[
        plant_usda["has_trait_cache"],
        "n_records"
    ].sum()
    /
    plant_usda["n_records"].sum()
)

print(
    f"USDA trait-cache coverage by plant code: "
    f"{code_coverage:.2%}"
)

print(
    f"USDA trait-cache coverage by plant records: "
    f"{record_coverage:.2%}"
)

if record_coverage < 0.99:
    print(
        "\nWARNING: <99% of USDA-resolved plant records "
        "are represented in the ecological trait cache."
    )

USDA trait-cache coverage by plant code: 99.97%
USDA trait-cache coverage by plant records: 99.99%


In [26]:
missing_trait_taxa = (
    dictionary_fg.loc[
        dictionary_fg["code_class"].eq("Plant")
        & dictionary_fg["USDA_accepted_symbol"].notna()
        & ~dictionary_fg["USDA_accepted_symbol"].isin(
            set(
                usda_attributes["USDA_accepted_symbol"]
                .dropna()
                .astype(str)
                .str.strip()
            )
        ),
        [
            "USDA_accepted_symbol",
            "USDA_scientific_name",
            "n_records",
        ],
    ]
    .groupby(
        ["USDA_accepted_symbol", "USDA_scientific_name"],
        dropna=False,
        as_index=False,
    )
    ["n_records"]
    .sum()
    .sort_values("n_records", ascending=False)
)

print(f"Missing USDA taxa: {len(missing_trait_taxa):,}")
print(
    "Plant records represented by missing taxa:",
    f"{missing_trait_taxa['n_records'].sum():,}"
)

display(missing_trait_taxa.head(100))

Missing USDA taxa: 2
Plant records represented by missing taxa: 408


,USDA_accepted_symbol,USDA_scientific_name,n_records
0,NONE,Notholaena neglecta Maxon,398
1,PONIN,Potentilla nivea L. var. nivea,10


In [27]:
# ============================================================
# INSPECT RAW USDA PlantProfile JSON FOR ONE KNOWN TAXON
# ============================================================

import json
import requests

symbol = "PSSP6"

url = f"{USDA_SERVICE_BASE}/PlantProfile"

response = requests.get(
    url,
    params={"symbol": symbol},
    timeout=REQUEST_TIMEOUT,
    headers={
        "Accept": "application/json",
        "User-Agent": "MOSAIC-USDA-trait-debug/1.0",
    },
)

print("HTTP status:", response.status_code)
print("Final URL:", response.url)
print("Content-Type:", response.headers.get("content-type"))

response.raise_for_status()

payload = response.json()

print("\nTOP-LEVEL TYPE:")
print(type(payload))

print("\nTOP-LEVEL KEYS:")
if isinstance(payload, dict):
    print(list(payload.keys()))
else:
    print("Not a dictionary")

print("\nRAW JSON:")
print(
    json.dumps(
        payload,
        indent=2,
        ensure_ascii=False,
    )
)

NameError: name 'USDA_SERVICE_BASE' is not defined

In [ ]:
# ============================================================
# INVENTORY ALL KEYS / VALUES IN THE PSSP6 RESPONSE
# ============================================================

def inspect_json(obj, path="root"):

    if isinstance(obj, dict):

        for key, value in obj.items():

            current_path = f"{path}.{key}"

            if isinstance(value, (dict, list)):
                print(
                    f"{current_path:<80} "
                    f"[{type(value).__name__}]"
                )
                inspect_json(
                    value,
                    current_path,
                )

            else:
                print(
                    f"{current_path:<80} "
                    f"= {value!r}"
                )

    elif isinstance(obj, list):

        for i, value in enumerate(obj):

            inspect_json(
                value,
                f"{path}[{i}]",
            )


inspect_json(payload)

root.Id                                                                          = 25163
root.Symbol                                                                      = 'PSSP6'
root.ScientificName                                                              = '<i>Pseudoroegneria spicata</i> (Pursh) Á. Löve'
root.ScientificNameComponents                                                    = None
root.CommonName                                                                  = 'bluebunch wheatgrass'
root.Group                                                                       = 'Monocot'
root.RankId                                                                      = 180
root.Rank                                                                        = 'Species'
root.Durations                                                                   [list]
root.GrowthHabits                                                                [list]
root.NativeStatuses                          

In [ ]:
# ============================================================
# RECOVER THE 478 USDA TAXA MISSING FROM THE TRAIT CACHE
#
# Purpose:
#   Query ecological traits ONLY for accepted USDA symbols
#   absent from the existing cache.
#
# Recovers:
#   - duration
#   - growth habit
#   - Lower-48 native status
#
# Behavior:
#   - resumable
#   - checkpointed
#   - never re-queries existing cached taxa
#   - appends recovered rows to the existing cache
#   - writes failures separately
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import time
import re
import pandas as pd
import requests


# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

USDA_SERVICE_BASE = "https://plantsservices.sc.egov.usda.gov/api"

REQUEST_TIMEOUT = 30
REQUEST_DELAY_SECONDS = 0.25
MAX_RETRIES = 3
SAVE_EVERY = 20

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RECOVERY_PROGRESS_FILE = (
    OUTPUT_DIR /
    "USDA_PLANTS_ecological_attributes_missing_taxa_progress.csv"
)

RECOVERY_FAILURE_FILE = (
    OUTPUT_DIR /
    "USDA_PLANTS_ecological_attributes_missing_taxa_failures.csv"
)


# ------------------------------------------------------------
# GENERAL CLEANING
# ------------------------------------------------------------

def clean_text(x):
    if x is None:
        return None

    x = str(x).strip()

    if (
        x == ""
        or x.lower() in {
            "nan",
            "none",
            "null",
            "<na>",
        }
    ):
        return None

    return x


def unique_pipe(values):
    """
    Collapse repeated values while preserving multiple legitimate
    trait states such as Annual|Biennial|Perennial.
    """
    seen = []

    for x in values:
        x = clean_text(x)

        if x is not None and x not in seen:
            seen.append(x)

    return "|".join(seen) if seen else None


# ------------------------------------------------------------
# WALK ARBITRARY USDA JSON
# ------------------------------------------------------------

def walk_json(obj):
    """
    Recursively yield every dictionary in a nested JSON object.
    """

    if isinstance(obj, dict):

        yield obj

        for value in obj.values():
            yield from walk_json(value)

    elif isinstance(obj, list):

        for value in obj:
            yield from walk_json(value)


def values_for_keys(dicts, possible_keys):
    """
    Search nested dictionaries for candidate field names.
    """

    values = []

    for d in dicts:

        for key in possible_keys:

            if key in d:

                value = d[key]

                # Scalar
                if not isinstance(
                    value,
                    (dict, list),
                ):

                    value = clean_text(value)

                    if value is not None:
                        values.append(value)

    return unique_pipe(values)


# ------------------------------------------------------------
# NORMALIZE LOWER-48 NATIVITY
# ------------------------------------------------------------

def normalize_l48_nativity(raw):
    """
    Normalize USDA nativity into:

        compact:
            N
            I
            N|I
            None

        text:
            Native
            Introduced
            Native|Introduced
            None
    """

    raw = clean_text(raw)

    if raw is None:
        return None, None

    text = raw.upper()

    native = bool(
        re.search(r"\bL48\s*N\b", text)
        or text == "N"
        or "NATIVE" in text
    )

    introduced = bool(
        re.search(r"\bL48\s*I\b", text)
        or text == "I"
        or "INTRODUCED" in text
    )

    # Avoid "Introduced" accidentally triggering the word "Native"
    if "INTRODUCED" in text and "NATIVE" not in text:
        native = False

    if native and introduced:
        return "N|I", "Native|Introduced"

    if native:
        return "N", "Native"

    if introduced:
        return "I", "Introduced"

    return raw, None


# ------------------------------------------------------------
# PARSE USDA PROFILE RESPONSE
# ------------------------------------------------------------

def parse_profile_payload(symbol, payload):

    dicts = list(
        walk_json(payload)
    )

    if not dicts:
        raise ValueError(
            "USDA response contained no JSON dictionaries."
        )


    # --------------------------------------------------------
    # Plant ID
    # --------------------------------------------------------

    plant_id = values_for_keys(
        dicts,
        [
            "PlantID",
            "PlantId",
            "plantID",
            "plantId",
        ],
    )


    # --------------------------------------------------------
    # Taxonomic rank
    # --------------------------------------------------------

    rank = values_for_keys(
        dicts,
        [
            "Rank",
            "rank",
            "TaxonRank",
            "taxonRank",
        ],
    )


    # --------------------------------------------------------
    # Duration
    # --------------------------------------------------------

    duration = values_for_keys(
        dicts,
        [
            "Duration",
            "duration",
            "DurationName",
            "durationName",
        ],
    )


    # --------------------------------------------------------
    # Growth habit
    # --------------------------------------------------------

    growth_habit = values_for_keys(
        dicts,
        [
            "GrowthHabit",
            "growthHabit",
            "GrowthHabitName",
            "growthHabitName",
        ],
    )


    # --------------------------------------------------------
    # Native status
    #
    # We specifically want Lower-48 status.
    # --------------------------------------------------------

    native_raw_values = []

    for d in dicts:

        for key, value in d.items():

            if isinstance(
                value,
                (dict, list),
            ):
                continue

            value = clean_text(value)

            if value is None:
                continue

            key_upper = str(key).upper()
            value_upper = value.upper()

            # Explicit L48-named property
            if "L48" in key_upper:
                native_raw_values.append(value)

            # Generic native-status field that itself says L48
            elif (
                "NATIVE" in key_upper
                and "L48" in value_upper
            ):
                native_raw_values.append(value)

            elif (
                "STATUS" in key_upper
                and "L48" in value_upper
            ):
                native_raw_values.append(value)


    native_raw = unique_pipe(
        native_raw_values
    )

    native_status, native_type = (
        normalize_l48_nativity(
            native_raw
        )
    )


    return {
        "USDA_accepted_symbol":
            symbol,

        "USDA_plant_id":
            plant_id,

        "USDA_rank":
            rank,

        "USDA_duration_api":
            duration,

        "USDA_growth_habit_api":
            growth_habit,

        "USDA_L48_native_status_api":
            native_status,

        "USDA_L48_native_type_api":
            native_type,

        "retrieved_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }


# ------------------------------------------------------------
# FETCH ONE USDA SYMBOL
# ------------------------------------------------------------

def fetch_usda_traits(symbol):

    url = (
        f"{USDA_SERVICE_BASE}/PlantProfile"
    )

    last_error = None

    # The service has historically accepted symbol.
    # Try capitalization variant only if needed.
    param_options = [
        {"symbol": symbol},
        {"Symbol": symbol},
    ]

    for params in param_options:

        for attempt in range(
            1,
            MAX_RETRIES + 1,
        ):

            try:

                response = requests.get(
                    url,
                    params=params,
                    timeout=REQUEST_TIMEOUT,
                    headers={
                        "Accept":
                            "application/json",

                        "User-Agent":
                            "MOSAIC-USDA-trait-recovery/1.0",
                    },
                )

                response.raise_for_status()

                payload = response.json()

                result = parse_profile_payload(
                    symbol,
                    payload,
                )

                # Successful HTTP response is not enough.
                # Require at least one useful profile field.
                useful = any(
                    result[x] is not None
                    for x in [
                        "USDA_plant_id",
                        "USDA_rank",
                        "USDA_duration_api",
                        "USDA_growth_habit_api",
                        "USDA_L48_native_status_api",
                    ]
                )

                if useful:
                    return result

                last_error = (
                    "Profile returned but no usable "
                    "ecological/taxonomic fields were found."
                )

            except Exception as exc:

                last_error = repr(exc)


            if attempt < MAX_RETRIES:

                wait = (
                    REQUEST_DELAY_SECONDS
                    * (2 ** (attempt - 1))
                )

                time.sleep(wait)


    raise RuntimeError(
        last_error
    )


# ------------------------------------------------------------
# LOAD CURRENT CACHE
# ------------------------------------------------------------

cache = pd.read_csv(
    USDA_ATTRIBUTE_CACHE,
    dtype="string",
    low_memory=False,
).copy()


# Normalize historical schema
if (
    "accepted_symbol" in cache.columns
    and
    "USDA_accepted_symbol"
    not in cache.columns
):

    cache = cache.rename(
        columns={
            "accepted_symbol":
                "USDA_accepted_symbol"
        }
    )


if "USDA_accepted_symbol" not in cache.columns:

    raise ValueError(
        "Could not locate accepted-symbol field "
        "in USDA ecological trait cache."
    )


cache["USDA_accepted_symbol"] = (
    cache["USDA_accepted_symbol"]
    .astype("string")
    .str.strip()
)


cached_symbols = set(
    cache[
        "USDA_accepted_symbol"
    ]
    .dropna()
)


print(
    f"Existing USDA cache taxa: "
    f"{len(cached_symbols):,}"
)


# ------------------------------------------------------------
# DEFINE TARGETS DIRECTLY FROM CURRENT MISSING TABLE
# ------------------------------------------------------------

targets = (
    missing_trait_taxa[
        "USDA_accepted_symbol"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)


# Safety check:
# remove anything already cached
targets = [
    x
    for x in targets
    if x not in cached_symbols
]


print(
    f"Taxa requiring recovery: "
    f"{len(targets):,}"
)


# Expected right now:
# 478
if len(targets) != 478:

    print(
        "\nNOTE:"
        f" expected 478 based on current audit, "
        f"but found {len(targets):,}."
    )


# ------------------------------------------------------------
# RESUME PRIOR RECOVERY IF ONE EXISTS
# ------------------------------------------------------------

if RECOVERY_PROGRESS_FILE.exists():

    recovered = pd.read_csv(
        RECOVERY_PROGRESS_FILE,
        dtype="string",
        low_memory=False,
    )

    recovered_symbols = set(
        recovered[
            "USDA_accepted_symbol"
        ]
        .dropna()
        .astype(str)
        .str.strip()
    )

else:

    recovered = pd.DataFrame()

    recovered_symbols = set()


remaining = [
    x
    for x in targets
    if x not in recovered_symbols
]


print(
    f"Previously recovered: "
    f"{len(recovered_symbols):,}"
)

print(
    f"Remaining this run: "
    f"{len(remaining):,}"
)


# ------------------------------------------------------------
# RECOVERY LOOP
# ------------------------------------------------------------

pending_rows = []
pending_failures = []


for i, symbol in enumerate(
    remaining,
    start=1,
):

    try:

        result = fetch_usda_traits(
            symbol
        )

        pending_rows.append(
            result
        )

        print(
            f"[{i:03d}/{len(remaining):03d}] "
            f"{symbol:<10} "
            f"D={str(result['USDA_duration_api']):<24} "
            f"H={str(result['USDA_growth_habit_api']):<28} "
            f"L48={result['USDA_L48_native_type_api']}"
        )

    except Exception as exc:

        pending_failures.append(
            {
                "USDA_accepted_symbol":
                    symbol,

                "error":
                    repr(exc),

                "failed_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
            }
        )

        print(
            f"[{i:03d}/{len(remaining):03d}] "
            f"{symbol:<10} FAILED | {exc}"
        )


    # --------------------------------------------------------
    # CHECKPOINT
    # --------------------------------------------------------

    if (
        i % SAVE_EVERY == 0
        or i == len(remaining)
    ):

        if pending_rows:

            new_df = pd.DataFrame(
                pending_rows
            )

            if recovered.empty:

                recovered = (
                    new_df.copy()
                )

            else:

                recovered = pd.concat(
                    [
                        recovered,
                        new_df,
                    ],
                    ignore_index=True,
                )


            recovered = (
                recovered
                .drop_duplicates(
                    subset=[
                        "USDA_accepted_symbol"
                    ],
                    keep="last",
                )
                .sort_values(
                    "USDA_accepted_symbol"
                )
                .reset_index(
                    drop=True
                )
            )


            recovered.to_csv(
                RECOVERY_PROGRESS_FILE,
                index=False,
            )

            pending_rows = []


        if pending_failures:

            fail_df = pd.DataFrame(
                pending_failures
            )


            if RECOVERY_FAILURE_FILE.exists():

                old_fail = pd.read_csv(
                    RECOVERY_FAILURE_FILE,
                    dtype="string",
                    low_memory=False,
                )

                fail_df = pd.concat(
                    [
                        old_fail,
                        fail_df,
                    ],
                    ignore_index=True,
                )


            fail_df.to_csv(
                RECOVERY_FAILURE_FILE,
                index=False,
            )

            pending_failures = []


    time.sleep(
        REQUEST_DELAY_SECONDS
    )


# ------------------------------------------------------------
# RELOAD RECOVERED RESULTS
# ------------------------------------------------------------

if RECOVERY_PROGRESS_FILE.exists():

    recovered = pd.read_csv(
        RECOVERY_PROGRESS_FILE,
        dtype="string",
        low_memory=False,
    )

else:

    recovered = pd.DataFrame()


print()
print(
    f"Successfully recovered taxa: "
    f"{len(recovered):,}"
)


# ------------------------------------------------------------
# APPEND TO MAIN CACHE
# ------------------------------------------------------------

if len(recovered):

    updated_cache = pd.concat(
        [
            cache,
            recovered,
        ],
        ignore_index=True,
        sort=False,
    )

else:

    updated_cache = cache.copy()


# Existing rows and recovered rows should have
# unique USDA accepted symbols.
updated_cache = (
    updated_cache
    .drop_duplicates(
        subset=[
            "USDA_accepted_symbol"
        ],
        keep="first",
    )
    .sort_values(
        "USDA_accepted_symbol"
    )
    .reset_index(
        drop=True
    )
)


updated_cache.to_csv(
    USDA_ATTRIBUTE_CACHE,
    index=False,
)


print(
    f"Updated cache taxa: "
    f"{len(updated_cache):,}"
)

print(
    f"Updated cache written to:\n"
    f"{USDA_ATTRIBUTE_CACHE}"
)


# ------------------------------------------------------------
# MEMBERSHIP QA
# ------------------------------------------------------------

updated_symbols = set(
    updated_cache[
        "USDA_accepted_symbol"
    ]
    .dropna()
    .astype(str)
    .str.strip()
)


still_absent = sorted(
    set(targets)
    - updated_symbols
)


print()
print(
    f"Recovery targets still absent "
    f"from cache: {len(still_absent):,}"
)


if still_absent:

    print(
        "First 50 still absent:"
    )

    print(
        still_absent[:50]
    )


# ------------------------------------------------------------
# TRAIT COMPLETENESS AMONG RECOVERED TAXA
# ------------------------------------------------------------

if len(recovered):

    for c in [
        "USDA_duration_api",
        "USDA_growth_habit_api",
        "USDA_L48_native_status_api",
    ]:

        n = (
            recovered[c]
            .notna()
            .sum()
            if c in recovered.columns
            else 0
        )

        print(
            f"{c}: "
            f"{n:,} / {len(recovered):,} "
            f"({n / len(recovered):.1%})"
        )


# ------------------------------------------------------------
# HIGH-PRIORITY SPECIES QA
# ------------------------------------------------------------

priority_symbols = [
    "PSSP6",
    "POPR",
    "AGCR",
    "HECO26",
    "FEID",
    "ELEL5",
    "BRIN2",
    "CHVI8",
    "ARTR2",
    "TACA8",
    "SIAL2",
    "ERCI6",
    "TRDU",
]


qa_columns = [
    x
    for x in [
        "USDA_accepted_symbol",
        "USDA_duration_api",
        "USDA_growth_habit_api",
        "USDA_L48_native_status_api",
        "USDA_L48_native_type_api",
        "USDA_rank",
        "retrieved_utc",
    ]
    if x in updated_cache.columns
]


display(
    updated_cache.loc[
        updated_cache[
            "USDA_accepted_symbol"
        ].isin(
            priority_symbols
        ),
        qa_columns,
    ]
    .sort_values(
        "USDA_accepted_symbol"
    )
)


print(
    "\nUSDA trait recovery finished."
)

Existing USDA cache taxa: 6,790
Taxa requiring recovery: 478
Previously recovered: 0
Remaining this run: 478
[001/478] PSSP6      D=None                     H=None                         L48=None
[002/478] POPR       D=None                     H=None                         L48=None
[003/478] AGCR       D=None                     H=None                         L48=None
[004/478] HECO26     D=None                     H=None                         L48=None
[005/478] FEID       D=None                     H=None                         L48=None
[006/478] ELEL5      D=None                     H=None                         L48=None
[007/478] BRIN2      D=None                     H=None                         L48=None
[008/478] CHVI8      D=None                     H=None                         L48=None
[009/478] SCAR7      D=None                     H=None                         L48=None
[010/478] ARAR8      D=None                     H=None                         L48=None
[011/478] A

,USDA_accepted_symbol,USDA_duration_api,USDA_growth_habit_api,USDA_L48_native_status_api,USDA_L48_native_type_api,USDA_rank,retrieved_utc
131,AGCR,<NA>,<NA>,<NA>,<NA>,Species|Kingdom|Subkingdom|Superdivision|Divis...,2026-09-01T20:01:37.103993+00:00
642,ARTR2,<NA>,<NA>,<NA>,<NA>,Species|Kingdom|Subkingdom|Superdivision|Divis...,2026-09-01T20:01:45.389504+00:00
1085,BRIN2,<NA>,<NA>,<NA>,<NA>,Species|Kingdom|Subkingdom|Superdivision|Divis...,2026-09-01T20:01:41.258874+00:00
1755,CHVI8,<NA>,<NA>,<NA>,<NA>,Species|Kingdom|Subkingdom|Superdivision|Divis...,2026-09-01T20:01:42.285878+00:00
2543,ELEL5,<NA>,<NA>,<NA>,<NA>,Species|Kingdom|Subkingdom|Superdivision|Divis...,2026-09-01T20:01:40.214832+00:00
2711,ERCI6,<NA>,<NA>,<NA>,<NA>,Species|Kingdom|Subkingdom|Superdivision|Divis...,2026-09-01T20:02:00.927287+00:00
3093,FEID,<NA>,<NA>,<NA>,<NA>,Species|Kingdom|Subkingdom|Superdivision|Divis...,2026-09-01T20:01:39.154428+00:00
3411,HECO26,<NA>,<NA>,<NA>,<NA>,Species|Kingdom|Subkingdom|Superdivision|Divis...,2026-09-01T20:01:38.110862+00:00
5591,POPR,<NA>,<NA>,<NA>,<NA>,Species|Kingdom|Subkingdom|Superdivision|Divis...,2026-09-01T20:01:36.096027+00:00
5715,PSSP6,<NA>,<NA>,<NA>,<NA>,Species|Kingdom|Subkingdom|Superdivision|Divis...,2026-09-01T20:01:34.948611+00:00



USDA trait recovery finished.


In [ ]:
# ============================================================
# CORRECT USDA PlantProfile PARSER
# ============================================================

from datetime import datetime, timezone


def clean_text(x):
    if x is None:
        return None

    x = str(x).strip()

    if x == "" or x.lower() in {
        "nan", "none", "null", "<na>"
    }:
        return None

    return x


def unique_pipe(values):
    if values is None:
        return None

    if not isinstance(values, (list, tuple, set)):
        values = [values]

    out = []

    for value in values:
        value = clean_text(value)

        if value is not None and value not in out:
            out.append(value)

    return "|".join(out) if out else None


def parse_profile_payload(symbol, payload):
    """
    Parse ecological traits directly from the ROOT USDA PlantProfile object.

    Important:
        Do NOT recursively search the JSON.
        Ancestors contain the same field names with null values.
    """

    if not isinstance(payload, dict):
        raise TypeError(
            f"{symbol}: expected dictionary response, "
            f"got {type(payload).__name__}"
        )

    # --------------------------------------------------------
    # Verify that the returned profile corresponds to symbol
    # --------------------------------------------------------

    returned_symbol = clean_text(payload.get("Symbol"))

    if returned_symbol is not None and returned_symbol != symbol:
        raise ValueError(
            f"Requested {symbol}, USDA returned {returned_symbol}"
        )

    # --------------------------------------------------------
    # Identity
    # --------------------------------------------------------

    plant_id = payload.get("Id")
    rank = clean_text(payload.get("Rank"))

    # --------------------------------------------------------
    # Duration
    #
    # USDA JSON:
    #   "Durations": ["Perennial"]
    # --------------------------------------------------------

    duration = unique_pipe(
        payload.get("Durations")
    )

    # --------------------------------------------------------
    # Growth habit
    #
    # USDA JSON:
    #   "GrowthHabits": ["Graminoid"]
    # --------------------------------------------------------

    growth_habit = unique_pipe(
        payload.get("GrowthHabits")
    )

    # --------------------------------------------------------
    # Lower-48 native status
    #
    # USDA JSON:
    #   "NativeStatuses": [
    #       {"Region": "AK",  ...},
    #       {"Region": "CAN", ...},
    #       {"Region": "L48", "Status": "N", "Type": "Native"}
    #   ]
    # --------------------------------------------------------

    l48_status = None
    l48_type = None

    native_statuses = payload.get("NativeStatuses") or []

    for item in native_statuses:

        if not isinstance(item, dict):
            continue

        region = clean_text(
            item.get("Region")
        )

        if region == "L48":

            l48_status = clean_text(
                item.get("Status")
            )

            l48_type = clean_text(
                item.get("Type")
            )

            break

    return {
        "USDA_accepted_symbol":
            symbol,

        "USDA_plant_id":
            plant_id,

        "USDA_rank":
            rank,

        "USDA_duration_api":
            duration,

        "USDA_growth_habit_api":
            growth_habit,

        "USDA_L48_native_status_api":
            l48_status,

        "USDA_L48_native_type_api":
            l48_type,

        "retrieved_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

In [ ]:
test = parse_profile_payload(
    "PSSP6",
    payload
)

display(
    pd.DataFrame([test])
)

,USDA_accepted_symbol,USDA_plant_id,USDA_rank,USDA_duration_api,USDA_growth_habit_api,USDA_L48_native_status_api,USDA_L48_native_type_api,retrieved_utc
0,PSSP6,25163,Species,Perennial,Graminoid,N,Native,2026-09-01T20:12:06.349353+00:00


In [ ]:
# ============================================================
# CHECK / REMOVE BAD RECOVERY CHECKPOINT
# ============================================================

if RECOVERY_PROGRESS_FILE.exists():

    old_recovery = pd.read_csv(
        RECOVERY_PROGRESS_FILE,
        dtype="string",
        low_memory=False,
    )

    trait_cols = [
        "USDA_duration_api",
        "USDA_growth_habit_api",
        "USDA_L48_native_status_api",
    ]

    existing_trait_cols = [
        c for c in trait_cols
        if c in old_recovery.columns
    ]

    if existing_trait_cols:

        all_traits_blank = (
            old_recovery[
                existing_trait_cols
            ]
            .isna()
            .all(axis=1)
        )

        print(
            "Rows in recovery checkpoint:",
            len(old_recovery)
        )

        print(
            "Rows with all ecological traits blank:",
            int(all_traits_blank.sum())
        )

        if all_traits_blank.any():

            backup = (
                RECOVERY_PROGRESS_FILE
                .with_name(
                    RECOVERY_PROGRESS_FILE.stem
                    + "_BAD_PARSER_BACKUP.csv"
                )
            )

            old_recovery.to_csv(
                backup,
                index=False
            )

            RECOVERY_PROGRESS_FILE.unlink()

            print(
                "\nRemoved contaminated recovery checkpoint."
            )

            print(
                "Backup preserved at:"
            )

            print(backup)

        else:

            print(
                "\nExisting recovery checkpoint appears usable."
            )

else:

    print(
        "No recovery checkpoint exists. "
        "Safe to start from 0."
    )

Rows in recovery checkpoint: 478
Rows with all ecological traits blank: 478

Removed contaminated recovery checkpoint.
Backup preserved at:
C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\species_dictionary_outputs\USDA_PLANTS_ecological_attributes_missing_taxa_progress_BAD_PARSER_BACKUP.csv


In [28]:
# ============================================================
# RECOVER ALL USDA TAXA MISSING FROM THE TRAIT CACHE
# USING CORRECT ROOT-LEVEL PlantProfile PARSER
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import time
import pandas as pd
import requests


# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

USDA_SERVICE_BASE = "https://plantsservices.sc.egov.usda.gov/api"

REQUEST_TIMEOUT = 30
REQUEST_DELAY_SECONDS = 0.25
MAX_RETRIES = 3
SAVE_EVERY = 20

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RECOVERY_PROGRESS_FILE = (
    OUTPUT_DIR /
    "USDA_PLANTS_ecological_attributes_missing_taxa_progress.csv"
)

RECOVERY_FAILURE_FILE = (
    OUTPUT_DIR /
    "USDA_PLANTS_ecological_attributes_missing_taxa_failures.csv"
)


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def clean_text(x):
    if x is None:
        return None

    x = str(x).strip()

    if x == "" or x.lower() in {
        "nan",
        "none",
        "null",
        "<na>",
    }:
        return None

    return x


def unique_pipe(values):
    """
    Preserve multiple USDA values such as:
        Annual|Biennial|Perennial
    """

    if values is None:
        return None

    if not isinstance(values, (list, tuple, set)):
        values = [values]

    out = []

    for value in values:
        value = clean_text(value)

        if value is not None and value not in out:
            out.append(value)

    return "|".join(out) if out else None


# ------------------------------------------------------------
# CORRECT USDA PlantProfile PARSER
# ------------------------------------------------------------

def parse_profile_payload(symbol, payload):
    """
    Parse ONLY the root PlantProfile object.

    USDA root fields:
        Id
        Symbol
        Rank
        Durations
        GrowthHabits
        NativeStatuses

    Do not recursively traverse Ancestors because those objects
    repeat these field names with null values.
    """

    if not isinstance(payload, dict):
        raise TypeError(
            f"{symbol}: expected dict response, "
            f"got {type(payload).__name__}"
        )

    returned_symbol = clean_text(
        payload.get("Symbol")
    )

    if (
        returned_symbol is not None
        and returned_symbol != symbol
    ):
        raise ValueError(
            f"Requested {symbol}, USDA returned "
            f"{returned_symbol}"
        )

    plant_id = payload.get("Id")
    rank = clean_text(
        payload.get("Rank")
    )

    duration = unique_pipe(
        payload.get("Durations")
    )

    growth_habit = unique_pipe(
        payload.get("GrowthHabits")
    )

    l48_status = None
    l48_type = None

    native_statuses = (
        payload.get("NativeStatuses")
        or []
    )

    for item in native_statuses:

        if not isinstance(item, dict):
            continue

        region = clean_text(
            item.get("Region")
        )

        if region == "L48":

            l48_status = clean_text(
                item.get("Status")
            )

            l48_type = clean_text(
                item.get("Type")
            )

            break

    return {
        "USDA_accepted_symbol":
            symbol,

        "USDA_plant_id":
            plant_id,

        "USDA_rank":
            rank,

        "USDA_duration_api":
            duration,

        "USDA_growth_habit_api":
            growth_habit,

        "USDA_L48_native_status_api":
            l48_status,

        "USDA_L48_native_type_api":
            l48_type,

        "retrieved_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }


# ------------------------------------------------------------
# FETCH ONE USDA PROFILE
# ------------------------------------------------------------

def fetch_usda_traits(symbol):

    url = (
        f"{USDA_SERVICE_BASE}/PlantProfile"
    )

    last_error = None

    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):

        try:

            response = requests.get(
                url,
                params={
                    "symbol": symbol
                },
                timeout=REQUEST_TIMEOUT,
                headers={
                    "Accept":
                        "application/json",

                    "User-Agent":
                        "MOSAIC-USDA-trait-recovery/1.1",
                },
            )

            response.raise_for_status()

            payload = response.json()

            result = parse_profile_payload(
                symbol,
                payload,
            )

            # A profile is considered successfully recovered
            # if USDA returned the requested symbol/profile.
            # Traits may legitimately be blank for some taxa.
            if (
                result["USDA_accepted_symbol"]
                == symbol
            ):
                return result

        except Exception as exc:

            last_error = exc


        if attempt < MAX_RETRIES:

            wait = (
                REQUEST_DELAY_SECONDS
                * (2 ** (attempt - 1))
            )

            time.sleep(wait)

    raise RuntimeError(
        f"{symbol}: {repr(last_error)}"
    )


# ------------------------------------------------------------
# LOAD EXISTING MAIN CACHE
# ------------------------------------------------------------

cache = pd.read_csv(
    USDA_ATTRIBUTE_CACHE,
    dtype="string",
    low_memory=False,
).copy()


# Normalize historical cache schema
column_aliases = {
    "accepted_symbol":
        "USDA_accepted_symbol",

    "USDA_duration":
        "USDA_duration_api",

    "USDA_growth_habit":
        "USDA_growth_habit_api",

    "USDA_native_status_L48":
        "USDA_L48_native_status_api",
}


for old, new in column_aliases.items():

    if (
        old in cache.columns
        and new not in cache.columns
    ):

        cache = cache.rename(
            columns={
                old: new
            }
        )


if (
    "USDA_accepted_symbol"
    not in cache.columns
):

    raise ValueError(
        "No USDA accepted-symbol field "
        "found in ecological trait cache."
    )


cache[
    "USDA_accepted_symbol"
] = (
    cache[
        "USDA_accepted_symbol"
    ]
    .astype("string")
    .str.strip()
)


cached_symbols = set(
    cache[
        "USDA_accepted_symbol"
    ]
    .dropna()
)


print(
    f"Existing USDA cache taxa: "
    f"{len(cached_symbols):,}"
)


# ------------------------------------------------------------
# DEFINE RECOVERY TARGETS FROM CURRENT DICTIONARY
# ------------------------------------------------------------

targets = (
    missing_trait_taxa[
        "USDA_accepted_symbol"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)


# Never query anything already in main cache
targets = [
    symbol
    for symbol in targets
    if symbol not in cached_symbols
]


print(
    f"Taxa requiring recovery: "
    f"{len(targets):,}"
)


# ------------------------------------------------------------
# LOAD PRIOR GOOD RECOVERY PROGRESS, IF ANY
# ------------------------------------------------------------

if RECOVERY_PROGRESS_FILE.exists():

    recovered = pd.read_csv(
        RECOVERY_PROGRESS_FILE,
        dtype="string",
        low_memory=False,
    )

    if (
        "USDA_accepted_symbol"
        not in recovered.columns
    ):
        raise ValueError(
            "Recovery progress file does not "
            "contain USDA_accepted_symbol."
        )

    recovered[
        "USDA_accepted_symbol"
    ] = (
        recovered[
            "USDA_accepted_symbol"
        ]
        .astype("string")
        .str.strip()
    )

    recovered_symbols = set(
        recovered[
            "USDA_accepted_symbol"
        ]
        .dropna()
    )

else:

    recovered = pd.DataFrame()

    recovered_symbols = set()


remaining = [
    symbol
    for symbol in targets
    if symbol not in recovered_symbols
]


print(
    f"Previously recovered: "
    f"{len(recovered_symbols):,}"
)

print(
    f"Remaining this run: "
    f"{len(remaining):,}"
)


# ------------------------------------------------------------
# RECOVERY LOOP
# ------------------------------------------------------------

pending_rows = []
pending_failures = []


for i, symbol in enumerate(
    remaining,
    start=1,
):

    try:

        row = fetch_usda_traits(
            symbol
        )

        pending_rows.append(
            row
        )

        print(
            f"[{i:03d}/{len(remaining):03d}] "
            f"{symbol:<10} "
            f"D={str(row['USDA_duration_api']):<24} "
            f"H={str(row['USDA_growth_habit_api']):<28} "
            f"L48={str(row['USDA_L48_native_type_api'])}"
        )

    except Exception as exc:

        pending_failures.append(
            {
                "USDA_accepted_symbol":
                    symbol,

                "error":
                    repr(exc),

                "failed_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
            }
        )

        print(
            f"[{i:03d}/{len(remaining):03d}] "
            f"{symbol:<10} FAILED | "
            f"{repr(exc)}"
        )


    # --------------------------------------------------------
    # PERIODIC CHECKPOINT
    # --------------------------------------------------------

    if (
        i % SAVE_EVERY == 0
        or i == len(remaining)
    ):

        # Save successful recoveries
        if pending_rows:

            batch = pd.DataFrame(
                pending_rows
            )

            if recovered.empty:

                recovered = (
                    batch.copy()
                )

            else:

                recovered = pd.concat(
                    [
                        recovered,
                        batch,
                    ],
                    ignore_index=True,
                    sort=False,
                )


            recovered = (
                recovered
                .drop_duplicates(
                    subset=[
                        "USDA_accepted_symbol"
                    ],
                    keep="last",
                )
                .sort_values(
                    "USDA_accepted_symbol"
                )
                .reset_index(
                    drop=True
                )
            )


            recovered.to_csv(
                RECOVERY_PROGRESS_FILE,
                index=False,
            )

            pending_rows = []


        # Save failures
        if pending_failures:

            fail_batch = pd.DataFrame(
                pending_failures
            )

            if RECOVERY_FAILURE_FILE.exists():

                prior_failures = pd.read_csv(
                    RECOVERY_FAILURE_FILE,
                    dtype="string",
                    low_memory=False,
                )

                fail_batch = pd.concat(
                    [
                        prior_failures,
                        fail_batch,
                    ],
                    ignore_index=True,
                    sort=False,
                )


            fail_batch = (
                fail_batch
                .drop_duplicates(
                    subset=[
                        "USDA_accepted_symbol",
                        "error",
                    ],
                    keep="last",
                )
                .reset_index(
                    drop=True
                )
            )


            fail_batch.to_csv(
                RECOVERY_FAILURE_FILE,
                index=False,
            )

            pending_failures = []


    time.sleep(
        REQUEST_DELAY_SECONDS
    )


# ------------------------------------------------------------
# RELOAD COMPLETE RECOVERY TABLE
# ------------------------------------------------------------

if RECOVERY_PROGRESS_FILE.exists():

    recovered = pd.read_csv(
        RECOVERY_PROGRESS_FILE,
        dtype="string",
        low_memory=False,
    )

else:

    recovered = pd.DataFrame()


print()
print(
    f"Successfully recovered taxa: "
    f"{len(recovered):,}"
)


# ------------------------------------------------------------
# APPEND RECOVERED TAXA TO MAIN CACHE
# ------------------------------------------------------------

if not recovered.empty:

    updated_cache = pd.concat(
        [
            cache,
            recovered,
        ],
        ignore_index=True,
        sort=False,
    )

else:

    updated_cache = cache.copy()


# Existing cached symbols should never be overwritten.
# Since recovery targets excluded existing symbols, duplicates
# should not normally exist.
updated_cache = (
    updated_cache
    .drop_duplicates(
        subset=[
            "USDA_accepted_symbol"
        ],
        keep="first",
    )
    .sort_values(
        "USDA_accepted_symbol"
    )
    .reset_index(
        drop=True
    )
)


updated_cache.to_csv(
    USDA_ATTRIBUTE_CACHE,
    index=False,
)


print(
    f"Updated USDA cache taxa: "
    f"{len(updated_cache):,}"
)

print(
    f"Updated cache written to:\n"
    f"{USDA_ATTRIBUTE_CACHE}"
)


# ------------------------------------------------------------
# QA 1 — ARE ANY TARGET SYMBOLS STILL ABSENT?
# ------------------------------------------------------------

updated_symbols = set(
    updated_cache[
        "USDA_accepted_symbol"
    ]
    .dropna()
    .astype(str)
    .str.strip()
)


still_absent = sorted(
    set(targets)
    - updated_symbols
)


print()
print(
    f"Recovery targets still absent "
    f"from cache: {len(still_absent):,}"
)


if still_absent:

    print(
        "First 50 absent symbols:"
    )

    print(
        still_absent[:50]
    )


# ------------------------------------------------------------
# QA 2 — TRAIT COMPLETENESS IN RECOVERED TAXA
# ------------------------------------------------------------

if len(recovered):

    print(
        "\nRecovered-taxon trait completeness:"
    )

    for col in [
        "USDA_duration_api",
        "USDA_growth_habit_api",
        "USDA_L48_native_status_api",
    ]:

        n_present = (
            recovered[col]
            .notna()
            .sum()
            if col in recovered.columns
            else 0
        )

        print(
            f"{col}: "
            f"{n_present:,} / {len(recovered):,} "
            f"({n_present / len(recovered):.2%})"
        )


# ------------------------------------------------------------
# QA 3 — HIGH-PRIORITY SPECIES
# ------------------------------------------------------------

priority_symbols = [
    "PSSP6",
    "POPR",
    "AGCR",
    "HECO26",
    "FEID",
    "ELEL5",
    "BRIN2",
    "CHVI8",
    "SCAR7",
    "ARTR2",
    "TACA8",
    "ALDE",
    "SIAL2",
    "ERCI6",
    "TRDU",
]


qa_cols = [
    col
    for col in [
        "USDA_accepted_symbol",
        "USDA_plant_id",
        "USDA_rank",
        "USDA_duration_api",
        "USDA_growth_habit_api",
        "USDA_L48_native_status_api",
        "USDA_L48_native_type_api",
        "retrieved_utc",
    ]
    if col in updated_cache.columns
]


display(
    updated_cache.loc[
        updated_cache[
            "USDA_accepted_symbol"
        ].isin(
            priority_symbols
        ),
        qa_cols,
    ]
    .sort_values(
        "USDA_accepted_symbol"
    )
)


print(
    "\nUSDA missing-taxon recovery complete."
)

print(
    "Next: reload this updated cache into dictionary_fg "
    "and rerun record-weighted cache coverage."
)

Existing USDA cache taxa: 7,268
Taxa requiring recovery: 1
Previously recovered: 0
Remaining this run: 1
[001/001] PONIN      D=Perennial                H=Forb/herb|Subshrub           L48=Native

Successfully recovered taxa: 1
Updated USDA cache taxa: 7,269
Updated cache written to:
C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\species_dictionary_outputs\USDA_PLANTS_ecological_attributes.csv

Recovery targets still absent from cache: 0

Recovered-taxon trait completeness:
USDA_duration_api: 1 / 1 (100.00%)
USDA_growth_habit_api: 1 / 1 (100.00%)
USDA_L48_native_status_api: 1 / 1 (100.00%)


,USDA_accepted_symbol,USDA_plant_id,USDA_rank,USDA_duration_api,USDA_growth_habit_api,USDA_L48_native_status_api,USDA_L48_native_type_api,retrieved_utc
131,AGCR,20068.0,Species,Perennial,Graminoid,I,Introduced,2026-09-01T20:19:14.115727+00:00
207,ALDE,61805.0,Species,Annual,Forb/herb,I,Introduced,2026-09-01T20:19:23.118227+00:00
642,ARTR2,32390.0,Species,Perennial,Shrub|Tree,N,Native,2026-09-01T20:19:59.612017+00:00
1085,BRIN2,20954.0,Species,Perennial,Graminoid,I,Introduced,2026-09-01T20:20:20.233090+00:00
1755,CHVI8,33492.0,Species,Perennial,Shrub,N,Native,2026-09-01T20:20:52.319662+00:00
2543,ELEL5,22420.0,Species,Perennial,Graminoid,N,Native,2026-09-01T20:21:38.980060+00:00
2711,ERCI6,83665.0,Species,Annual|Biennial,Forb/herb,I,Introduced,2026-09-01T20:21:59.522310+00:00
3093,FEID,23094.0,Species,Perennial,Graminoid,N,Native,2026-09-01T20:22:28.108292+00:00
3411,HECO26,23323.0,Species,Perennial,Graminoid,N,Native,2026-09-01T20:22:52.765229+00:00
5592,POPR,25028.0,Species,Perennial,Graminoid,I,Introduced,2026-09-01T20:25:09.294033+00:00



USDA missing-taxon recovery complete.
Next: reload this updated cache into dictionary_fg and rerun record-weighted cache coverage.


In [30]:
# ============================================================================
# FINAL REPAIR — REJOIN REPAIRED USDA ECOLOGICAL ATTRIBUTES
# AND OVERWRITE LDC_LPI_species_dictionary_FINAL_MASTER.csv
# ============================================================================

from pathlib import Path
import shutil
import pandas as pd
import numpy as np
from datetime import datetime


# ============================================================================
# PATHS
# ============================================================================

FINAL_MASTER_PATH = (
    Path(r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present")
    / "species_dictionary_outputs"
    / "LDC_LPI_species_dictionary_FINAL_MASTER.csv"
)

USDA_TRAIT_PATH = (
    Path(r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present")
    / "species_dictionary_outputs"
    / "USDA_PLANTS_ecological_attributes.csv"
)

assert FINAL_MASTER_PATH.exists(), FINAL_MASTER_PATH
assert USDA_TRAIT_PATH.exists(), USDA_TRAIT_PATH

print("FINAL MASTER:")
print(FINAL_MASTER_PATH)

print("\nUSDA trait cache:")
print(USDA_TRAIT_PATH)


# ============================================================================
# 1. READ CURRENT FINAL MASTER + REPAIRED USDA TRAIT CACHE
# ============================================================================

master = pd.read_csv(
    FINAL_MASTER_PATH,
    dtype="string",
    keep_default_na=True
)

traits = pd.read_csv(
    USDA_TRAIT_PATH,
    dtype="string",
    keep_default_na=True
)

print("\nLoaded:")
print(f"  final master: {len(master):,} rows")
print(f"  USDA cache:   {len(traits):,} rows")


# ============================================================================
# 2. BASIC FINAL-MASTER INVARIANTS BEFORE TOUCHING ANYTHING
# ============================================================================

assert len(master) == 10960, (
    f"Unexpected FINAL_MASTER row count: {len(master):,}"
)

assert master["observed_code"].nunique(dropna=False) == 10960, (
    "FINAL_MASTER observed_code is no longer unique."
)

assert "USDA_accepted_symbol" in master.columns
assert "MOSAIC_FG" in master.columns


# ============================================================================
# 3. NORMALIZATION HELPERS
# ============================================================================

def clean_string(s):
    """
    Normalize blanks/whitespace to pandas NA while preserving string content.
    """
    s = s.astype("string").str.strip()
    return s.mask(s.eq(""))


def first_existing(df, candidates, required=False):
    """
    Return first existing column name from candidates.
    """
    for c in candidates:
        if c in df.columns:
            return c

    if required:
        raise KeyError(
            "None of these expected columns were found:\n"
            + "\n".join(candidates)
        )

    return None


# Normalize master key.
master["USDA_accepted_symbol"] = (
    clean_string(master["USDA_accepted_symbol"])
    .str.upper()
)


# ============================================================================
# 4. DETECT ACTUAL USDA CACHE SCHEMA
#
# The repaired cache has changed slightly during development, so explicitly
# map the fields rather than assuming a single historical column spelling.
# ============================================================================

trait_symbol_col = first_existing(
    traits,
    [
        "USDA_accepted_symbol",
        "USDA_symbol",
        "AcceptedSymbol",
        "accepted_symbol",
        "symbol",
    ],
    required=True,
)

duration_col = first_existing(
    traits,
    [
        "USDA_duration",
        "USDA_duration_api",
        "Duration",
        "duration",
    ],
    required=True,
)

growth_col = first_existing(
    traits,
    [
        "USDA_growth_habit",
        "USDA_growth_habit_api",
        "GrowthHabit",
        "GrowthHabitName",
        "growth_habit",
    ],
    required=True,
)

native_col = first_existing(
    traits,
    [
        "USDA_native_status_L48",
        "USDA_L48_native_status_api",
        "USDA_native_status",
        "NativeStatus_L48",
        "native_status_L48",
    ],
    required=True,
)

print("\nDetected USDA-cache fields:")
print("  symbol:      ", trait_symbol_col)
print("  duration:    ", duration_col)
print("  growth habit:", growth_col)
print("  L48 nativity:", native_col)


# ============================================================================
# 5. BUILD ONE-ROW-PER-USDA-SYMBOL TRAIT LOOKUP
# ============================================================================

trait_lookup = traits[
    [
        trait_symbol_col,
        duration_col,
        growth_col,
        native_col,
    ]
].copy()

trait_lookup = trait_lookup.rename(
    columns={
        trait_symbol_col: "_USDA_JOIN_SYMBOL",
        duration_col: "_repair_duration",
        growth_col: "_repair_growth_habit",
        native_col: "_repair_native_L48",
    }
)

trait_lookup["_USDA_JOIN_SYMBOL"] = (
    clean_string(trait_lookup["_USDA_JOIN_SYMBOL"])
    .str.upper()
)

for c in [
    "_repair_duration",
    "_repair_growth_habit",
    "_repair_native_L48",
]:
    trait_lookup[c] = clean_string(trait_lookup[c])


# Remove empty symbols.
trait_lookup = trait_lookup.loc[
    trait_lookup["_USDA_JOIN_SYMBOL"].notna()
].copy()


# We must have at most one ecological-trait row per USDA symbol.
duplicate_symbols = (
    trait_lookup[
        trait_lookup["_USDA_JOIN_SYMBOL"].duplicated(keep=False)
    ]
    .sort_values("_USDA_JOIN_SYMBOL")
)

if len(duplicate_symbols):
    print("\nWARNING: duplicate USDA symbols in ecological cache:")
    display(duplicate_symbols.head(50))

    # Collapse only when duplicate rows are ecologically identical.
    conflict_check = (
        trait_lookup
        .groupby("_USDA_JOIN_SYMBOL", dropna=False)
        .agg(
            n_duration=("_repair_duration", "nunique"),
            n_growth=("_repair_growth_habit", "nunique"),
            n_native=("_repair_native_L48", "nunique"),
        )
        .reset_index()
    )

    true_conflicts = conflict_check.loc[
        (conflict_check["n_duration"] > 1)
        | (conflict_check["n_growth"] > 1)
        | (conflict_check["n_native"] > 1)
    ]

    assert len(true_conflicts) == 0, (
        "USDA ecological cache contains conflicting duplicate symbols. "
        "Do not collapse them automatically."
    )

trait_lookup = (
    trait_lookup
    .drop_duplicates("_USDA_JOIN_SYMBOL", keep="first")
    .reset_index(drop=True)
)

print(
    "\nUnique USDA symbols available for trait repair:",
    f"{len(trait_lookup):,}"
)


# ============================================================================
# 6. JOIN TRAITS ONTO CURRENT FINAL MASTER
# ============================================================================

before_n = len(master)

master = master.merge(
    trait_lookup,
    left_on="USDA_accepted_symbol",
    right_on="_USDA_JOIN_SYMBOL",
    how="left",
    validate="many_to_one",
)

assert len(master) == before_n


# ============================================================================
# 7. ENSURE TARGET COLUMNS EXIST
# ============================================================================

for c in [
    "USDA_duration",
    "USDA_growth_habit",
    "USDA_native_status_L48",
    "USDA_trait_status",
    "USDA_derived_group",
    "resolved_plant_group",
    "MOSAIC_FG",
    "FG_resolution_source",
]:
    if c not in master.columns:
        master[c] = pd.NA


for c in [
    "USDA_duration",
    "USDA_growth_habit",
    "USDA_native_status_L48",
]:
    master[c] = clean_string(master[c])


# ============================================================================
# 8. FILL ONLY MISSING ECOLOGICAL TRAITS
#
# Existing valid values win.
# Repaired cache is used only where FINAL_MASTER is blank.
# ============================================================================

missing_duration_before = master["USDA_duration"].isna()
missing_growth_before = master["USDA_growth_habit"].isna()
missing_native_before = master["USDA_native_status_L48"].isna()

master["USDA_duration"] = (
    master["USDA_duration"]
    .fillna(master["_repair_duration"])
)

master["USDA_growth_habit"] = (
    master["USDA_growth_habit"]
    .fillna(master["_repair_growth_habit"])
)

master["USDA_native_status_L48"] = (
    master["USDA_native_status_L48"]
    .fillna(master["_repair_native_L48"])
)

duration_repaired = (
    missing_duration_before
    & master["USDA_duration"].notna()
)

growth_repaired = (
    missing_growth_before
    & master["USDA_growth_habit"].notna()
)

native_repaired = (
    missing_native_before
    & master["USDA_native_status_L48"].notna()
)

print("\nTrait values repaired:")
print(f"  duration:     {duration_repaired.sum():,}")
print(f"  growth habit: {growth_repaired.sum():,}")
print(f"  L48 nativity: {native_repaired.sum():,}")


# ============================================================================
# 9. STANDARDIZE TRAIT TEXT FOR ECOLOGICAL PARSING
# ============================================================================

duration = (
    master["USDA_duration"]
    .fillna("")
    .str.lower()
)

growth = (
    master["USDA_growth_habit"]
    .fillna("")
    .str.lower()
)

native = (
    master["USDA_native_status_L48"]
    .fillna("")
    .str.lower()
)


# ============================================================================
# 10. DERIVE USDA ECOLOGICAL GROUP
#
# This is intentionally conservative.
#
# Priority:
#   tree
#   shrub/subshrub
#   graminoid -> annual/perennial
#   forb/herb -> exotic/native
#
# Ambiguous/missing combinations stay unresolved rather than being guessed.
# ============================================================================

derived = pd.Series(
    pd.NA,
    index=master.index,
    dtype="string"
)


# --------------------------------------------------------------------------
# Trees
# --------------------------------------------------------------------------

tree_mask = growth.str.contains(
    r"\btree\b",
    regex=True,
    na=False,
)

derived.loc[tree_mask] = "Tree"


# --------------------------------------------------------------------------
# Shrubs / subshrubs
# --------------------------------------------------------------------------

shrub_mask = (
    derived.isna()
    & growth.str.contains(
        r"shrub|subshrub",
        regex=True,
        na=False,
    )
)

derived.loc[shrub_mask] = "Shrub"


# --------------------------------------------------------------------------
# Graminoids
#
# USDA growth-habit strings commonly contain Graminoid or Grass.
# --------------------------------------------------------------------------

graminoid_mask = (
    derived.isna()
    & growth.str.contains(
        r"graminoid|grass|grass-like",
        regex=True,
        na=False,
    )
)

annual_mask = duration.str.contains(
    r"annual",
    regex=True,
    na=False,
)

biennial_mask = duration.str.contains(
    r"biennial",
    regex=True,
    na=False,
)

perennial_mask = duration.str.contains(
    r"perennial",
    regex=True,
    na=False,
)


derived.loc[
    graminoid_mask
    & annual_mask
    & ~perennial_mask
] = "AnnualGrass"

derived.loc[
    graminoid_mask
    & perennial_mask
] = "PerennialGrass"


# --------------------------------------------------------------------------
# Forbs / herbs
# --------------------------------------------------------------------------

forb_mask = (
    derived.isna()
    & growth.str.contains(
        r"forb|herb",
        regex=True,
        na=False,
    )
)


# Native-status interpretation:
#
# Treat explicit "introduced", "non-native", "not native", "exotic",
# or "alien" language as exotic.
#
# Treat explicit native presence as native unless an exotic marker is also
# present.
# --------------------------------------------------------------------------

exotic_mask = native.str.contains(
    r"introduced|non[- ]?native|not native|exotic|alien",
    regex=True,
    na=False,
)

native_mask = (
    native.str.contains(
        r"\bnative\b",
        regex=True,
        na=False,
    )
    & ~exotic_mask
)


derived.loc[
    forb_mask
    & exotic_mask
] = "ExoticForb"

derived.loc[
    forb_mask
    & native_mask
] = "NativeForb"


master["USDA_derived_group"] = derived


# ============================================================================
# 11. REBUILD USDA TRAIT STATUS
# ============================================================================

def trait_status_row(row):

    code_class = row.get("code_class")

    # pd.NA must be handled explicitly before comparison.
    if pd.isna(code_class):
        return pd.NA

    if str(code_class).strip() != "Plant":
        return pd.NA

    accepted_symbol = row.get(
        "USDA_accepted_symbol"
    )

    if pd.isna(accepted_symbol):
        return "No_USDA_taxon"

    missing = []

    duration = row.get(
        "USDA_duration"
    )

    growth_habit = row.get(
        "USDA_growth_habit"
    )

    nativity = row.get(
        "USDA_native_status_L48"
    )

    derived_group = row.get(
        "USDA_derived_group"
    )

    if pd.isna(duration):
        missing.append("duration")

    if pd.isna(growth_habit):
        missing.append("growth_habit")

    if pd.isna(nativity):
        missing.append("nativity")

    if missing:
        return (
            "Missing_"
            + "_".join(missing)
        )

    if pd.isna(derived_group):
        return "Traits_present_group_unresolved"

    return "Traits_complete"


master["USDA_trait_status"] = (
    master.apply(
        trait_status_row,
        axis=1,
    )
)

print(
    "USDA trait-status rebuild complete."
)

display(
    master["USDA_trait_status"]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "USDA_trait_status"
    )
    .reset_index(
        name="n_codes"
    )
)


# ============================================================================
# 12. UPDATE FUNCTIONAL GROUP ONLY WHERE USDA TRAITS WERE THE MISSING PIECE
#
# Preserve:
#   - manual ecological overrides
#   - protocol-derived classifications
#   - existing resolved MOSAIC_FG
#
# Replace only:
#   - TraitUnresolvedPlant
#   - blank MOSAIC_FG on Plant rows
#   - USDA_taxon_missing_traits provenance
# ============================================================================

current_fg = clean_string(master["MOSAIC_FG"])

repairable_fg = (
    master["code_class"].eq("Plant")
    & master["USDA_derived_group"].notna()
    & (
        current_fg.isna()
        | current_fg.eq("TraitUnresolvedPlant")
        | master["FG_resolution_source"].eq(
            "USDA_taxon_missing_traits"
        )
    )
)

master.loc[
    repairable_fg,
    "resolved_plant_group"
] = master.loc[
    repairable_fg,
    "USDA_derived_group"
]

master.loc[
    repairable_fg,
    "MOSAIC_FG"
] = master.loc[
    repairable_fg,
    "USDA_derived_group"
]

master.loc[
    repairable_fg,
    "FG_resolution_source"
] = "USDA_ecological_traits"

print(
    "\nFunctional-group rows repaired:",
    f"{repairable_fg.sum():,}"
)


# ============================================================================
# 13. RESTORE KNOWN MANUAL ECOLOGICAL OVERRIDES
#
# These explicitly outrank generic USDA-derived traits.
# ============================================================================

bassia_override = master["observed_code"].isin(
    ["BAPRV", "BAPRG"]
)

master.loc[
    bassia_override,
    "resolved_plant_group"
] = "Forb"

master.loc[
    bassia_override,
    "MOSAIC_FG"
] = "ExoticForb"

master.loc[
    bassia_override,
    "FG_resolution_source"
] = "manual_ecological_functional_override"


# ============================================================================
# 14. HARD QA — STRUCTURE
# ============================================================================

assert len(master) == 10960

assert master["observed_code"].nunique(dropna=False) == 10960

assert not master["observed_code"].isna().any()


# ============================================================================
# 15. HARD QA — SENTINELS / KNOWN REPAIRS
# ============================================================================

no_canopy = master.loc[
    master["observed_code"].eq("__NO_CANOPY__")
]

literal_none = master.loc[
    master["observed_code"].eq("NONE")
]

poar = master.loc[
    master["observed_code"].eq("POAR2R2")
]

assert len(no_canopy) == 1
assert len(literal_none) == 1
assert len(poar) == 1

assert (
    no_canopy.iloc[0]["MOSAIC_FG"]
    == "NO_CANOPY"
)

assert (
    poar.iloc[0]["USDA_accepted_symbol"]
    == "PONIN"
)


# ============================================================================
# 16. QA — COMMON TAXA SHOULD NOW HAVE TRAITS
# ============================================================================

common_codes = [
    "PSSP6",
    "POPR",
    "AGCR",
    "HECO26",
    "FEID",
    "ELEL5",
    "CHVI8",
    "ARTR2",
    "TACA8",
    "ERCI6",
]

common_qa = master.loc[
    master["observed_code"].isin(common_codes),
    [
        "observed_code",
        "USDA_accepted_symbol",
        "USDA_scientific_name",
        "USDA_duration",
        "USDA_growth_habit",
        "USDA_native_status_L48",
        "USDA_trait_status",
        "USDA_derived_group",
        "MOSAIC_FG",
        "FG_resolution_source",
        "n_records",
    ]
].sort_values(
    "observed_code"
)

print("\nCommon-taxon QA:")
display(common_qa)


# ============================================================================
# 17. COVERAGE QA
# ============================================================================

plant = master["code_class"].eq("Plant")

resolved_usda = (
    plant
    & master["USDA_accepted_symbol"].notna()
)

coverage = pd.DataFrame(
    {
        "metric": [
            "Plant codes",
            "USDA-resolved plant codes",
            "USDA-resolved with duration",
            "USDA-resolved with growth habit",
            "USDA-resolved with L48 nativity",
            "USDA-resolved with all 3 traits",
            "Plant codes still TraitUnresolvedPlant",
        ],
        "n": [
            int(plant.sum()),
            int(resolved_usda.sum()),
            int(
                (
                    resolved_usda
                    & master["USDA_duration"].notna()
                ).sum()
            ),
            int(
                (
                    resolved_usda
                    & master["USDA_growth_habit"].notna()
                ).sum()
            ),
            int(
                (
                    resolved_usda
                    & master["USDA_native_status_L48"].notna()
                ).sum()
            ),
            int(
                (
                    resolved_usda
                    & master["USDA_duration"].notna()
                    & master["USDA_growth_habit"].notna()
                    & master["USDA_native_status_L48"].notna()
                ).sum()
            ),
            int(
                (
                    plant
                    & master["MOSAIC_FG"].eq(
                        "TraitUnresolvedPlant"
                    )
                ).sum()
            ),
        ],
    }
)

print("\nTrait / FG coverage after repair:")
display(coverage)


# ============================================================================
# 18. REMOVE TEMPORARY JOIN COLUMNS
# ============================================================================

master = master.drop(
    columns=[
        "_USDA_JOIN_SYMBOL",
        "_repair_duration",
        "_repair_growth_habit",
        "_repair_native_L48",
    ],
    errors="ignore",
)


# ============================================================================
# 19. BACK UP CURRENT FILE, THEN OVERWRITE FINAL MASTER
#
# The requested product is overwritten, but we retain one timestamped backup
# of the pre-repair file for auditability.
# ============================================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_path = FINAL_MASTER_PATH.with_name(
    FINAL_MASTER_PATH.stem
    + f"_PRE_TRAIT_REPAIR_{timestamp}"
    + FINAL_MASTER_PATH.suffix
)

shutil.copy2(
    FINAL_MASTER_PATH,
    backup_path,
)

master.to_csv(
    FINAL_MASTER_PATH,
    index=False,
)


# ============================================================================
# 20. READ BACK FROM DISK AND VERIFY THE ACTUAL WRITTEN PRODUCT
# ============================================================================

check = pd.read_csv(
    FINAL_MASTER_PATH,
    dtype="string",
)

assert len(check) == 10960
assert check["observed_code"].nunique(dropna=False) == 10960

assert (
    check.loc[
        check["observed_code"].eq("POAR2R2"),
        "USDA_accepted_symbol",
    ].iloc[0]
    == "PONIN"
)

assert (
    check.loc[
        check["observed_code"].eq("__NO_CANOPY__"),
        "MOSAIC_FG",
    ].iloc[0]
    == "NO_CANOPY"
)


print("\n" + "=" * 80)
print("FINAL MASTER USDA TRAIT REPAIR COMPLETE")
print("=" * 80)

print("\nBackup of pre-repair master:")
print(backup_path)

print("\nOverwritten FINAL_MASTER:")
print(FINAL_MASTER_PATH)

print("\nRows:", f"{len(check):,}")
print(
    "Unique observed codes:",
    f"{check['observed_code'].nunique(dropna=False):,}"
)

print("\nDONE")

FINAL MASTER:
C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\species_dictionary_outputs\LDC_LPI_species_dictionary_FINAL_MASTER.csv

USDA trait cache:
C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\species_dictionary_outputs\USDA_PLANTS_ecological_attributes.csv

Loaded:
  final master: 10,960 rows
  USDA cache:   7,269 rows

Detected USDA-cache fields:
  symbol:       USDA_accepted_symbol
  duration:     USDA_duration_api
  growth habit: USDA_growth_habit_api
  L48 nativity: USDA_L48_native_status_api

Unique USDA symbols available for trait repair: 7,269

Trait values repaired:
  duration:     497
  growth habit: 497
  L48 nativity: 497
USDA trait-status rebuild complete.


,USDA_trait_status,n_codes
0,Traits_complete,6825
1,No_USDA_taxon,3144
2,Traits_present_group_unresolved,321
3,Missing_duration_nativity,231
4,NaN,211
5,Missing_nativity,153
6,Missing_duration_growth_habit_nativity,73
7,Missing_duration,2



Functional-group rows repaired: 575

Common-taxon QA:


,observed_code,USDA_accepted_symbol,USDA_scientific_name,USDA_duration,USDA_growth_habit,USDA_native_status_L48,USDA_trait_status,USDA_derived_group,MOSAIC_FG,FG_resolution_source,n_records
31,AGCR,AGCR,Agropyron cristatum (L.) Gaertn.,Perennial,Graminoid,I,Traits_complete,PerennialGrass,PerennialGrass,USDA_ecological_traits,86637
82,ARTR2,ARTR2,Artemisia tridentata Nutt.,Perennial,Shrub|Tree,N,Traits_complete,Tree,Tree,USDA_ecological_traits,27981
12,CHVI8,CHVI8,Chrysothamnus viscidiflorus (Hook.) Nutt.,Perennial,Shrub,N,Traits_complete,Shrub,Shrub,USDA_ecological_traits,52275
9,ELEL5,ELEL5,Elymus elymoides (Raf.) Swezey,Perennial,Graminoid,N,Traits_complete,PerennialGrass,PerennialGrass,USDA_ecological_traits,58454
73,ERCI6,ERCI6,Erodium cicutarium (L.) L'Hér. ex Aiton,Annual|Biennial,Forb/herb,I,Traits_present_group_unresolved,<NA>,TraitUnresolvedPlant,USDA_taxon_missing_traits,16863
34,FEID,FEID,Festuca idahoensis Elmer,Perennial,Graminoid,N,Traits_complete,PerennialGrass,PerennialGrass,USDA_ecological_traits,78604
19,HECO26,HECO26,Hesperostipa comata (Trin. & Rupr.) Barkworth,Perennial,Graminoid,N,Traits_complete,PerennialGrass,PerennialGrass,USDA_ecological_traits,80033
29,POPR,POPR,Poa pratensis L.,Perennial,Graminoid,I,Traits_complete,PerennialGrass,PerennialGrass,USDA_ecological_traits,115614
10,PSSP6,PSSP6,Pseudoroegneria spicata (Pursh) Á. Löve,Perennial,Graminoid,N,Traits_complete,PerennialGrass,PerennialGrass,USDA_ecological_traits,163393
115,TACA8,TACA8,Taeniatherum caput-medusae (L.) Nevski,Annual,Graminoid,I,Traits_complete,AnnualGrass,AnnualGrass,USDA_ecological_traits,26357



Trait / FG coverage after repair:


,metric,n
0,Plant codes,10749
1,USDA-resolved plant codes,7605
2,USDA-resolved with duration,7299
3,USDA-resolved with growth habit,7532
4,USDA-resolved with L48 nativity,7148
5,USDA-resolved with all 3 traits,7146
6,Plant codes still TraitUnresolvedPlant,604



FINAL MASTER USDA TRAIT REPAIR COMPLETE

Backup of pre-repair master:
C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\species_dictionary_outputs\LDC_LPI_species_dictionary_FINAL_MASTER_PRE_TRAIT_REPAIR_20260902_130541.csv

Overwritten FINAL_MASTER:
C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\species_dictionary_outputs\LDC_LPI_species_dictionary_FINAL_MASTER.csv

Rows: 10,960
Unique observed codes: 10,960

DONE


In [31]:
# ============================================================================
# RECORD-WEIGHTED REPRESENTATION OF UNRESOLVED PLANT CODES
# ============================================================================

# Ensure numeric record counts.
master["n_records_num"] = pd.to_numeric(
    master["n_records"],
    errors="coerce"
).fillna(0)

plant_mask = master["code_class"].eq("Plant")

unresolved_mask = (
    plant_mask
    & master["MOSAIC_FG"].eq("TraitUnresolvedPlant")
)

# --------------------------------------------------------------------------
# Overall weighted representation
# --------------------------------------------------------------------------

total_plant_codes = int(plant_mask.sum())

total_plant_records = master.loc[
    plant_mask,
    "n_records_num"
].sum()

unresolved_codes = int(unresolved_mask.sum())

unresolved_records = master.loc[
    unresolved_mask,
    "n_records_num"
].sum()

summary = pd.DataFrame({
    "metric": [
        "Total plant codes",
        "TraitUnresolvedPlant codes",
        "TraitUnresolvedPlant code share (%)",
        "Total plant records",
        "TraitUnresolvedPlant records",
        "TraitUnresolvedPlant record share (%)",
    ],
    "value": [
        total_plant_codes,
        unresolved_codes,
        100 * unresolved_codes / total_plant_codes,
        int(total_plant_records),
        int(unresolved_records),
        100 * unresolved_records / total_plant_records,
    ]
})

print("=" * 80)
print("WEIGHTED UNRESOLVED REPRESENTATION")
print("=" * 80)

display(summary)


# ============================================================================
# USDA-RESOLVED VS NON-USDA UNRESOLVED
# ============================================================================

unresolved_detail = master.loc[
    unresolved_mask
].copy()

unresolved_detail["resolution_bucket"] = np.where(
    unresolved_detail["USDA_accepted_symbol"].notna(),
    "USDA-resolved but FG unresolved",
    "No USDA taxon"
)

bucket_summary = (
    unresolved_detail
    .groupby(
        "resolution_bucket",
        dropna=False
    )
    .agg(
        n_codes=("observed_code", "nunique"),
        n_records=("n_records_num", "sum"),
    )
    .reset_index()
)

bucket_summary["pct_of_unresolved_records"] = (
    100
    * bucket_summary["n_records"]
    / unresolved_records
)

bucket_summary["pct_of_all_plant_records"] = (
    100
    * bucket_summary["n_records"]
    / total_plant_records
)

print("\nUnresolved records by origin:")
display(
    bucket_summary.sort_values(
        "n_records",
        ascending=False
    )
)


# ============================================================================
# TOP UNRESOLVED CODES BY RECORD ABUNDANCE
# ============================================================================

top_unresolved = (
    unresolved_detail[
        [
            "observed_code",
            "resolved_label",
            "USDA_accepted_symbol",
            "USDA_scientific_name",
            "USDA_duration",
            "USDA_growth_habit",
            "USDA_native_status_L48",
            "USDA_trait_status",
            "FG_resolution_source",
            "n_records_num",
            "n_plot_visits",
        ]
    ]
    .sort_values(
        "n_records_num",
        ascending=False
    )
    .reset_index(drop=True)
)

top_unresolved["pct_of_unresolved_records"] = (
    100
    * top_unresolved["n_records_num"]
    / unresolved_records
)

top_unresolved["pct_of_all_plant_records"] = (
    100
    * top_unresolved["n_records_num"]
    / total_plant_records
)

top_unresolved["cumulative_unresolved_pct"] = (
    100
    * top_unresolved["n_records_num"].cumsum()
    / unresolved_records
)

print("\nTop unresolved codes:")
display(
    top_unresolved.head(50)
)


# ============================================================================
# HOW CONCENTRATED IS THE UNRESOLVED TAIL?
# ============================================================================

concentration_rows = []

for n in [5, 10, 20, 50, 100]:

    n = min(
        n,
        len(top_unresolved)
    )

    records_n = (
        top_unresolved
        .head(n)["n_records_num"]
        .sum()
    )

    concentration_rows.append({
        "top_n_codes": n,
        "records": int(records_n),
        "pct_of_unresolved_records":
            100 * records_n / unresolved_records,
        "pct_of_all_plant_records":
            100 * records_n / total_plant_records,
    })

concentration = pd.DataFrame(
    concentration_rows
)

print("\nConcentration of unresolved records:")
display(concentration)


# ============================================================================
# RECORD-WEIGHTED USDA TRAIT STATUS
# ============================================================================

trait_status_weighted = (
    master.loc[plant_mask]
    .groupby(
        "USDA_trait_status",
        dropna=False
    )
    .agg(
        n_codes=("observed_code", "nunique"),
        n_records=("n_records_num", "sum"),
    )
    .reset_index()
)

trait_status_weighted["pct_of_plant_records"] = (
    100
    * trait_status_weighted["n_records"]
    / total_plant_records
)

trait_status_weighted = trait_status_weighted.sort_values(
    "n_records",
    ascending=False
)

print("\nRecord-weighted USDA trait status:")
display(trait_status_weighted)

WEIGHTED UNRESOLVED REPRESENTATION


,metric,value
0,Total plant codes,1.074900e+04
1,TraitUnresolvedPlant codes,6.040000e+02
2,TraitUnresolvedPlant code share (%),5.619127e+00
3,Total plant records,5.436341e+06
4,TraitUnresolvedPlant records,3.039460e+05
5,TraitUnresolvedPlant record share (%),5.591003e+00



Unresolved records by origin:


,resolution_bucket,n_codes,n_records,pct_of_unresolved_records,pct_of_all_plant_records
0,USDA-resolved but FG unresolved,604,303946,100.0,5.591003



Top unresolved codes:


,observed_code,resolved_label,USDA_accepted_symbol,USDA_scientific_name,USDA_duration,USDA_growth_habit,USDA_native_status_L48,USDA_trait_status,FG_resolution_source,n_records_num,n_plot_visits,pct_of_unresolved_records,pct_of_all_plant_records,cumulative_unresolved_pct
0,ALDE,Alyssum desertorum Stapf,ALDE,Alyssum desertorum Stapf,Annual,Forb/herb,I,Traits_present_group_unresolved,USDA_taxon_missing_traits,26258,3321,8.639035,0.483009,8.639035
1,PHHO,Phlox hoodii Richardson,PHHO,Phlox hoodii Richardson,Perennial,Forb/herb,N,Traits_present_group_unresolved,USDA_taxon_missing_traits,22412,5933,7.373678,0.412263,16.012713
2,SIAL2,Sisymbrium altissimum L.,SIAL2,Sisymbrium altissimum L.,Annual|Biennial,Forb/herb,I,Traits_present_group_unresolved,USDA_taxon_missing_traits,18009,2851,5.925066,0.331271,21.937778
3,ERCI6,Erodium cicutarium (L.) L'Hér. ex Aiton,ERCI6,Erodium cicutarium (L.) L'Hér. ex Aiton,Annual|Biennial,Forb/herb,I,Traits_present_group_unresolved,USDA_taxon_missing_traits,16863,1831,5.548025,0.31019,27.485803
4,ACMI2,Achillea millefolium L.,ACMI2,Achillea millefolium L.,Perennial,Forb/herb,I,Traits_present_group_unresolved,USDA_taxon_missing_traits,12985,3766,4.27214,0.238856,31.757944
5,COPA3,Collinsia parviflora Lindl.,COPA3,Collinsia parviflora Lindl.,Annual,Forb/herb,N,Traits_present_group_unresolved,USDA_taxon_missing_traits,9380,2213,3.086075,0.172543,34.844018
6,TAOF,Taraxacum officinale F.H. Wigg.,TAOF,Taraxacum officinale F.H. Wigg.,Perennial,Forb/herb,I,Traits_present_group_unresolved,USDA_taxon_missing_traits,9240,2211,3.040014,0.169967,37.884032
7,DEPI,Descurainia pinnata (Walter) Britton,DEPI,Descurainia pinnata (Walter) Britton,Annual|Biennial|Perennial,Forb/herb,N,Traits_present_group_unresolved,USDA_taxon_missing_traits,9183,2719,3.02126,0.168919,40.905292
8,LEPE2,Lepidium perfoliatum L.,LEPE2,Lepidium perfoliatum L.,Annual|Biennial,Forb/herb,I,Traits_present_group_unresolved,USDA_taxon_missing_traits,8619,1060,2.835701,0.158544,43.740993
9,HAGL,Halogeton glomeratus (M. Bieb.) C.A. Mey.,HAGL,Halogeton glomeratus (M. Bieb.) C.A. Mey.,Annual,Forb/herb,I,Traits_present_group_unresolved,USDA_taxon_missing_traits,8041,1044,2.645536,0.147912,46.386529



Concentration of unresolved records:


,top_n_codes,records,pct_of_unresolved_records,pct_of_all_plant_records
0,5,96527,31.757944,1.775588
1,10,140990,46.386529,2.593472
2,20,191848,63.119107,3.528991
3,50,246722,81.172972,4.538383
4,100,277297,91.232324,5.100802



Record-weighted USDA trait status:


,USDA_trait_status,n_codes,n_records,pct_of_plant_records
5,Traits_complete,6825,4899935,90.132959
6,Traits_present_group_unresolved,321,291900,5.36942
4,No_USDA_taxon,3144,219306,4.034074
2,Missing_duration_nativity,231,12894,0.237182
3,Missing_nativity,153,11516,0.211834
1,Missing_duration_growth_habit_nativity,73,737,0.013557
0,Missing_duration,2,53,0.000975


In [32]:
# ============================================================================
# DIAGNOSTIC — DECOMPOSE USDA TRAIT-FAILURE GROUPS
# ============================================================================

# Ensure record counts are numeric.
master["n_records_num"] = pd.to_numeric(
    master["n_records"],
    errors="coerce"
).fillna(0)

plant_mask = master["code_class"].eq("Plant")

# --------------------------------------------------------------------------
# Define the specific USDA-resolved failure groups
# --------------------------------------------------------------------------

status_groups = {
    "missing_all_three": "Missing_duration_growth_habit_nativity",
    "missing_nativity_only": "Missing_nativity",
    "missing_duration_and_nativity": "Missing_duration_nativity",
    "missing_duration_only": "Missing_duration",
    "traits_present_group_unresolved": "Traits_present_group_unresolved",
    "no_usda_taxon": "No_USDA_taxon",
}

group_tables = {}

for group_name, status_value in status_groups.items():

    df = master.loc[
        plant_mask
        & master["USDA_trait_status"].eq(status_value)
    ].copy()

    df = df.sort_values(
        "n_records_num",
        ascending=False
    ).reset_index(drop=True)

    group_tables[group_name] = df


# ============================================================================
# 1. HIGH-LEVEL SUMMARY
# ============================================================================

summary_rows = []

total_plant_records = master.loc[
    plant_mask,
    "n_records_num"
].sum()

for group_name, df in group_tables.items():

    n_records = df["n_records_num"].sum()

    summary_rows.append({
        "group": group_name,
        "n_codes": len(df),
        "n_records": int(n_records),
        "pct_of_all_plant_records":
            100 * n_records / total_plant_records
            if total_plant_records else np.nan,
    })

group_summary = (
    pd.DataFrame(summary_rows)
    .sort_values(
        "n_records",
        ascending=False
    )
)

print("=" * 90)
print("TRAIT-FAILURE GROUP SUMMARY")
print("=" * 90)

display(group_summary)


# ============================================================================
# 2. HELPER FOR DISTRIBUTION TABLES
# ============================================================================

def summarize_field(df, field, top_n=30):

    if field not in df.columns:
        return pd.DataFrame()

    x = df.copy()

    x[field] = (
        x[field]
        .astype("string")
        .fillna("<NA>")
        .str.strip()
        .replace("", "<BLANK>")
    )

    out = (
        x.groupby(
            field,
            dropna=False
        )
        .agg(
            n_codes=("observed_code", "nunique"),
            n_records=("n_records_num", "sum"),
        )
        .reset_index()
        .sort_values(
            "n_records",
            ascending=False
        )
        .head(top_n)
    )

    total = df["n_records_num"].sum()

    if total:
        out["pct_group_records"] = (
            100
            * out["n_records"]
            / total
        )

    return out


# ============================================================================
# 3. INSPECT EACH FAILURE GROUP
# ============================================================================

fields_to_summarize = [
    "USDA_growth_habit",
    "USDA_duration",
    "USDA_native_status_L48",
    "taxonomic_level",
    "USDA_rank",
    "resolved_plant_group",
    "MOSAIC_FG",
    "FG_resolution_source",
]

for group_name, df in group_tables.items():

    print("\n\n")
    print("=" * 90)
    print(group_name.upper())
    print("=" * 90)

    print(
        f"Codes: {len(df):,} | "
        f"Records: {df['n_records_num'].sum():,.0f}"
    )

    # ----------------------------------------------------------------------
    # Field distributions
    # ----------------------------------------------------------------------

    for field in fields_to_summarize:

        if field not in df.columns:
            continue

        print(f"\n--- {field} ---")

        display(
            summarize_field(
                df,
                field,
                top_n=25
            )
        )

    # ----------------------------------------------------------------------
    # Highest-weight individual codes
    # ----------------------------------------------------------------------

    show_cols = [
        "observed_code",
        "resolved_label",
        "USDA_accepted_symbol",
        "USDA_scientific_name",
        "USDA_duration",
        "USDA_growth_habit",
        "USDA_native_status_L48",
        "taxonomic_level",
        "USDA_rank",
        "USDA_trait_status",
        "resolved_plant_group",
        "MOSAIC_FG",
        "FG_resolution_source",
        "n_records_num",
        "n_plot_visits",
    ]

    show_cols = [
        c for c in show_cols
        if c in df.columns
    ]

    top = df[show_cols].head(50).copy()

    group_records = df["n_records_num"].sum()

    if group_records:

        top["pct_group_records"] = (
            100
            * top["n_records_num"]
            / group_records
        )

        top["cumulative_group_pct"] = (
            100
            * top["n_records_num"].cumsum()
            / group_records
        )

    print("\nTop taxa / codes:")
    display(top)


# ============================================================================
# 4. SPECIFIC FAILURE-REASON LOGIC
#
# This classifies WHY a row is still unresolved from the perspective of
# our functional-group rules.
# ============================================================================

def infer_failure_reason(row):

    status = row.get("USDA_trait_status")

    growth = row.get("USDA_growth_habit")
    duration = row.get("USDA_duration")
    native = row.get("USDA_native_status_L48")

    growth = (
        "" if pd.isna(growth)
        else str(growth).strip().lower()
    )

    duration = (
        "" if pd.isna(duration)
        else str(duration).strip().lower()
    )

    native = (
        "" if pd.isna(native)
        else str(native).strip().upper()
    )

    # --------------------------------------------------------
    # No USDA taxon
    # --------------------------------------------------------

    if status == "No_USDA_taxon":
        return "No_USDA_taxonomic_match"

    # --------------------------------------------------------
    # Completely missing traits
    # --------------------------------------------------------

    if status == "Missing_duration_growth_habit_nativity":
        return "USDA_profile_has_no_ecological_traits"

    # --------------------------------------------------------
    # Habit known, but nativity missing
    # --------------------------------------------------------

    if "forb/herb" in growth:

        if native == "":
            return "Forb_known_but_native_exotic_unknown"

        return "Forb_should_now_resolve"

    if "shrub" in growth:
        return "Shrub_should_resolve_without_duration_or_nativity"

    if "tree" in growth:
        return "Tree_should_resolve_without_duration_or_nativity"

    if (
        "graminoid" in growth
        or "grass" in growth
    ):

        if duration == "":
            return "Grass_known_but_annual_perennial_unknown"

        return "Grass_should_resolve_from_duration"

    if "nonvascular" in growth:
        return "Nonvascular_should_resolve"

    if growth:
        return "Other_growth_habit_requires_rule_review"

    return "Growth_habit_missing"


master["trait_failure_reason"] = master.apply(
    infer_failure_reason,
    axis=1
)


# ============================================================================
# 5. RECORD-WEIGHTED FAILURE REASONS
# ============================================================================

failure_reason_summary = (
    master.loc[
        plant_mask
        & master["USDA_trait_status"].isin(
            list(status_groups.values())
        )
    ]
    .groupby(
        "trait_failure_reason",
        dropna=False
    )
    .agg(
        n_codes=("observed_code", "nunique"),
        n_records=("n_records_num", "sum"),
    )
    .reset_index()
    .sort_values(
        "n_records",
        ascending=False
    )
)

failure_reason_summary[
    "pct_of_all_plant_records"
] = (
    100
    * failure_reason_summary["n_records"]
    / total_plant_records
)

print("\n")
print("=" * 90)
print("INFERRED FAILURE REASONS")
print("=" * 90)

display(failure_reason_summary)

TRAIT-FAILURE GROUP SUMMARY


,group,n_codes,n_records,pct_of_all_plant_records
4,traits_present_group_unresolved,321,291900,5.369420
5,no_usda_taxon,3144,219306,4.034074
2,missing_duration_and_nativity,231,12894,0.237182
1,missing_nativity_only,153,11516,0.211834
0,missing_all_three,73,737,0.013557
3,missing_duration_only,2,53,0.000975





MISSING_ALL_THREE
Codes: 73 | Records: 737

--- USDA_growth_habit ---


,USDA_growth_habit,n_codes,n_records,pct_group_records
0,<NA>,73,737,100.0



--- USDA_duration ---


...

,FG_resolution_source,n_codes,n_records,pct_group_records
0,protocol_or_project_group,3144,219306,100.0



Top taxa / codes:


,observed_code,resolved_label,USDA_accepted_symbol,USDA_scientific_name,USDA_duration,USDA_growth_habit,USDA_native_status_L48,taxonomic_level,USDA_trait_status,resolved_plant_group,MOSAIC_FG,FG_resolution_source,n_records_num,n_plot_visits,pct_group_records,cumulative_group_pct
0,Perennial grasses,Perennial grasses,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,No_USDA_taxon,PerennialGraminoid,PerennialGraminoid,protocol_or_project_group,61012,1814,27.820488,27.820488
1,Annual plants,Annual plants,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,No_USDA_taxon,AnnualPlant,AnnualPlant,protocol_or_project_group,35750,1507,16.301424,44.121912
2,Shrubs,Shrubs,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,No_USDA_taxon,Shrub,SHR,protocol_or_project_group,21122,1221,9.631291,53.753203
3,Sub-shrubs and perennial forbs,Sub-shrubs and perennial forbs,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,No_USDA_taxon,SubshrubOrPerennialForb,SubshrubOrPF,protocol_or_project_group,20140,1330,9.183515,62.936719
4,Plant base,Plant base,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,No_USDA_taxon,UnknownPlant,UnknownPlant,protocol_or_project_group,17980,1105,8.19859,71.135309
5,Trees,Trees,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,No_USDA_taxon,Tree,TREE,protocol_or_project_group,14631,841,6.6715,77.806809
6,SH00,Shrub,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,No_USDA_taxon,Shrub,SHR,protocol_or_project_group,4187,903,1.909204,79.716013
7,AF00,Annual forb,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,No_USDA_taxon,AnnualForb,AF,protocol_or_project_group,2058,661,0.938415,80.654428
8,PF00,Perennial forb,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,No_USDA_taxon,PerennialForb,PF,protocol_or_project_group,1268,454,0.578188,81.232616
9,TR00,Tree,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,No_USDA_taxon,Tree,TREE,protocol_or_project_group,972,94,0.443216,81.675832




INFERRED FAILURE REASONS


,trait_failure_reason,n_codes,n_records,pct_of_all_plant_records
1,Forb_should_now_resolve,280,290341,5.340743
4,No_USDA_taxonomic_match,3144,219306,4.034074
5,Nonvascular_should_resolve,137,8865,0.163069
7,Shrub_should_resolve_without_duration_or_nativity,40,5460,0.100435
3,Grass_should_resolve_from_duration,34,5245,0.09648
6,Other_growth_habit_requires_rule_review,113,4002,0.073616
2,Grass_known_but_annual_perennial_unknown,26,1646,0.030278
0,Forb_known_but_native_exotic_unknown,72,738,0.013575
9,USDA_profile_has_no_ecological_traits,73,737,0.013557
8,Tree_should_resolve_without_duration_or_nativity,5,66,0.001214


In [36]:
# ============================================================================
# CANONICAL TWO-LEVEL MOSAIC FUNCTIONAL-GROUP CLASSIFICATION — REVISED
#
# LEVEL 1: MOSAIC_FG_structural
#   Uses growth habit + duration.
#
# LEVEL 2: MOSAIC_FG
#   Adds nativity where available.
#   Falls back to structural FG where nativity is unavailable.
#
# IMPORTANT:
#   - USDA source values remain untouched, including piped values.
#   - Both N/I and Native/Introduced are recognized.
#   - Duration is structurally meaningful for BOTH graminoids and forbs.
#   - Mixed woody habits retain Woody.
#   - Manual/project/protocol overrides retain precedence.
# ============================================================================

import pandas as pd
import numpy as np


# ============================================================================
# 1. HELPERS
# ============================================================================

def pipe_tokens(value):
    """
    Split USDA pipe-delimited values into normalized lowercase tokens.
    Original USDA fields are NOT modified.
    """

    if pd.isna(value):
        return set()

    value = str(value).strip()

    if value == "":
        return set()

    return {
        x.strip().lower()
        for x in value.split("|")
        if x.strip()
    }


def normalize_nativity_tokens(value):
    """
    USDA cache contains both abbreviated and full-word forms:

        N
        I
        Native
        Introduced

    Normalize all of them to:
        native
        introduced
    """

    tokens = pipe_tokens(value)

    normalized = set()

    for x in tokens:

        if x in {
            "n",
            "native",
        }:
            normalized.add("native")

        elif x in {
            "i",
            "introduced",
            "non-native",
            "nonnative",
            "exotic",
        }:
            normalized.add("introduced")

        else:
            normalized.add(x)

    return normalized


# ============================================================================
# 2. PARSE USDA FIELDS WITHOUT ALTERING SOURCE VALUES
# ============================================================================

growth_tokens = (
    master["USDA_growth_habit"]
    .apply(pipe_tokens)
)

duration_tokens = (
    master["USDA_duration"]
    .apply(pipe_tokens)
)

nativity_tokens = (
    master["USDA_native_status_L48"]
    .apply(normalize_nativity_tokens)
)


# ============================================================================
# 3. STRUCTURAL CLASSIFICATION
#
# Uses HABIT + DURATION.
#
# Nativity is deliberately NOT used here.
# ============================================================================

def classify_structural(
    growth_set,
    duration_set,
):
    """
    Derive structural / life-history functional group.
    """

    if not growth_set:
        return pd.NA


    # ========================================================================
    # NONVASCULAR
    # ========================================================================

    if growth_set == {
        "nonvascular"
    }:
        return "Nonvascular"


    # ========================================================================
    # GRAMINOIDS
    # ========================================================================

    if growth_set == {
        "graminoid"
    }:

        annual = (
            "annual"
            in duration_set
        )

        perennial = (
            "perennial"
            in duration_set
        )

        # Annual or Annual|Biennial
        if (
            annual
            and not perennial
        ):
            return "AnnualGrass"

        # Perennial or Biennial|Perennial
        if (
            perennial
            and not annual
        ):
            return "PerennialGrass"

        # Annual|Perennial or missing duration
        return "Grass"


    # ========================================================================
    # FORBS / HERBS
    #
    # Forb/herb|Vine still behaves structurally as a forb.
    #
    # But duration IS retained.
    # ========================================================================

    if "forb/herb" in growth_set:

        remaining = (
            growth_set
            - {
                "forb/herb",
                "vine",
            }
        )

        # Pure herbaceous forb / forb-vine
        if len(remaining) == 0:

            annual = (
                "annual"
                in duration_set
            )

            perennial = (
                "perennial"
                in duration_set
            )

            # Annual or Annual|Biennial
            if (
                annual
                and not perennial
            ):
                return "AnnualForb"

            # Perennial or Biennial|Perennial
            if (
                perennial
                and not annual
            ):
                return "PerennialForb"

            # Annual|Perennial, missing duration,
            # or otherwise ambiguous duration.
            return "Forb"


    # ========================================================================
    # PURE WOODY HABITS
    # ========================================================================

    if growth_set == {
        "shrub"
    }:
        return "Shrub"

    if growth_set == {
        "subshrub"
    }:
        return "Subshrub"

    if growth_set == {
        "tree"
    }:
        return "Tree"


    # ========================================================================
    # SHRUB + SUBSHRUB
    #
    # Treat as Shrub at our desired resolution.
    # ========================================================================

    if (
        growth_set
        and growth_set.issubset(
            {
                "shrub",
                "subshrub",
            }
        )
    ):
        return "Shrub"


    # ========================================================================
    # MIXED WOODY HABITS
    #
    # Examples:
    #   Shrub|Tree
    #   Shrub|Subshrub|Tree
    #
    # Do NOT arbitrarily choose Tree or Shrub.
    # ========================================================================

    woody = {
        "shrub",
        "subshrub",
        "tree",
    }

    woody_intersection = (
        growth_set
        .intersection(
            woody
        )
    )

    if (
        len(woody_intersection) >= 2
        and growth_set.issubset(
            woody
        )
    ):
        return "Woody"


    # ========================================================================
    # FORB + WOODY HABIT
    #
    # USDA genuinely regards these as spanning growth forms.
    # ========================================================================

    if (
        "forb/herb"
        in growth_set
        and len(
            growth_set.intersection(
                woody
            )
        ) > 0
    ):
        return "Woody"


    # ========================================================================
    # VINE ONLY
    # ========================================================================

    if growth_set == {
        "vine"
    }:
        return "Vine"


    # ========================================================================
    # REMAINING COMPLEX USDA HABITS
    #
    # Example:
    #   Graminoid|Shrub|Vine
    #
    # Do not invent a dominant structural class.
    # ========================================================================

    return "MixedGrowthHabit"


master[
    "MOSAIC_FG_structural"
] = pd.Series(
    [
        classify_structural(
            g,
            d,
        )
        for g, d
        in zip(
            growth_tokens,
            duration_tokens,
        )
    ],
    index=master.index,
    dtype="string",
)


# ============================================================================
# 4. NATIVITY CLASS
# ============================================================================

def classify_nativity(
    token_set
):

    if not token_set:
        return pd.NA

    # Introduced wins if USDA somehow contains both.
    if (
        "introduced"
        in token_set
    ):
        return "Introduced"

    if (
        "native"
        in token_set
    ):
        return "Native"

    return pd.NA


master[
    "MOSAIC_nativity_class"
] = pd.Series(
    [
        classify_nativity(x)
        for x
        in nativity_tokens
    ],
    index=master.index,
    dtype="string",
)


# ============================================================================
# 5. BUILD NATIVITY-REFINED FG
# ============================================================================

structural = (
    master[
        "MOSAIC_FG_structural"
    ]
    .astype("string")
)

native_class = (
    master[
        "MOSAIC_nativity_class"
    ]
    .astype("string")
)

refined = (
    structural.copy()
)


# ============================================================================
# ANNUAL GRASS
# ============================================================================

mask = (
    structural.eq(
        "AnnualGrass"
    )
    & native_class.eq(
        "Introduced"
    )
)

refined.loc[
    mask
] = "EAG"


mask = (
    structural.eq(
        "AnnualGrass"
    )
    & native_class.eq(
        "Native"
    )
)

refined.loc[
    mask
] = "NAG"


# ============================================================================
# PERENNIAL GRASS
# ============================================================================

mask = (
    structural.eq(
        "PerennialGrass"
    )
    & native_class.eq(
        "Introduced"
    )
)

refined.loc[
    mask
] = "EPG"


mask = (
    structural.eq(
        "PerennialGrass"
    )
    & native_class.eq(
        "Native"
    )
)

refined.loc[
    mask
] = "NPG"


# ============================================================================
# ANNUAL FORBS
# ============================================================================

mask = (
    structural.eq(
        "AnnualForb"
    )
    & native_class.eq(
        "Introduced"
    )
)

refined.loc[
    mask
] = "ExoticAnnualForb"


mask = (
    structural.eq(
        "AnnualForb"
    )
    & native_class.eq(
        "Native"
    )
)

refined.loc[
    mask
] = "NativeAnnualForb"


# ============================================================================
# PERENNIAL FORBS
# ============================================================================

mask = (
    structural.eq(
        "PerennialForb"
    )
    & native_class.eq(
        "Introduced"
    )
)

refined.loc[
    mask
] = "ExoticPerennialForb"


mask = (
    structural.eq(
        "PerennialForb"
    )
    & native_class.eq(
        "Native"
    )
)

refined.loc[
    mask
] = "NativePerennialForb"


# ============================================================================
# DURATION-AMBIGUOUS / DURATION-MISSING FORBS
#
# We still know they are forbs.
# ============================================================================

mask = (
    structural.eq(
        "Forb"
    )
    & native_class.eq(
        "Introduced"
    )
)

refined.loc[
    mask
] = "ExoticForb"


mask = (
    structural.eq(
        "Forb"
    )
    & native_class.eq(
        "Native"
    )
)

refined.loc[
    mask
] = "NativeForb"


# ============================================================================
# 6. STORE USDA-DERIVED VALUES
# ============================================================================

master[
    "USDA_derived_group"
] = (
    master[
        "MOSAIC_FG_structural"
    ]
)

master[
    "USDA_MOSAIC_FG"
] = refined


# ============================================================================
# 7. PROTECT MANUAL / PROJECT / PROTOCOL CLASSIFICATIONS
# ============================================================================

fg_source = (
    master[
        "FG_resolution_source"
    ]
    .astype("string")
    .fillna("")
    .str.lower()
    .str.strip()
)

protected = pd.Series(
    False,
    index=master.index,
)

for pattern in [
    "manual",
    "protocol",
    "project",
    "local",
]:

    protected = (
        protected
        | fg_source.str.contains(
            pattern,
            regex=False,
            na=False,
        )
    )


# Explicit known Bassia overrides.
protected = (
    protected
    | master[
        "observed_code"
    ].isin(
        [
            "BAPRV",
            "BAPRG",
        ]
    )
)


# ============================================================================
# 8. APPLY USDA TWO-LEVEL CLASSIFICATION
# ============================================================================

eligible = (
    master[
        "code_class"
    ].eq(
        "Plant"
    )
    & refined.notna()
    & ~protected
)


master.loc[
    eligible,
    "resolved_plant_group"
] = master.loc[
    eligible,
    "MOSAIC_FG_structural"
]

master.loc[
    eligible,
    "MOSAIC_FG"
] = refined.loc[
    eligible
]

master.loc[
    eligible,
    "FG_resolution_source"
] = (
    "USDA_two_level_traits"
)


# ============================================================================
# 9. REASSERT MANUAL BASSIA OVERRIDE
#
# Preserve the ecological decision already made for these records.
# ============================================================================

bassia = master[
    "observed_code"
].isin(
    [
        "BAPRV",
        "BAPRG",
    ]
)

master.loc[
    bassia,
    "MOSAIC_FG_structural"
] = "Forb"

master.loc[
    bassia,
    "resolved_plant_group"
] = "Forb"

master.loc[
    bassia,
    "MOSAIC_FG"
] = "ExoticForb"

master.loc[
    bassia,
    "FG_resolution_source"
] = (
    "manual_ecological_functional_override"
)


# ============================================================================
# 10. RECORD COUNTS
# ============================================================================

master[
    "n_records_num"
] = pd.to_numeric(
    master["n_records"],
    errors="coerce",
).fillna(0)


# ============================================================================
# 11. QA — CRITICAL TAXA
# ============================================================================

qa_codes = [
    "BRTE",
    "POSE",
    "PSSP6",
    "AGCR",
    "HECO26",
    "FEID",
    "ELEL5",
    "TACA8",
    "ALDE",
    "PHHO",
    "SIAL2",
    "ERCI6",
    "BAPRV",
    "BAPRG",
]

qa_cols = [
    "observed_code",
    "USDA_accepted_symbol",
    "USDA_scientific_name",
    "USDA_duration",
    "USDA_growth_habit",
    "USDA_native_status_L48",
    "MOSAIC_nativity_class",
    "MOSAIC_FG_structural",
    "USDA_MOSAIC_FG",
    "MOSAIC_FG",
    "FG_resolution_source",
    "n_records_num",
]

print(
    "=" * 90
)

print(
    "CRITICAL TAXON QA"
)

print(
    "=" * 90
)

display(
    master.loc[
        master[
            "observed_code"
        ].isin(
            qa_codes
        ),
        qa_cols,
    ]
    .sort_values(
        "observed_code"
    )
)


# ============================================================================
# 12. HARD BRTE ASSERTION
# ============================================================================

brte = master.loc[
    master[
        "observed_code"
    ].eq(
        "BRTE"
    )
]

assert (
    len(brte)
    == 1
)

assert (
    brte.iloc[0][
        "MOSAIC_FG_structural"
    ]
    == "AnnualGrass"
), (
    "BRTE structural FG is not AnnualGrass."
)


brte_nat = brte.iloc[0][
    "MOSAIC_nativity_class"
]

assert not pd.isna(
    brte_nat
), (
    "BRTE nativity did not parse."
)

assert (
    brte_nat
    == "Introduced"
), (
    f"Expected BRTE Introduced; got {brte_nat!r}"
)

assert (
    brte.iloc[0][
        "MOSAIC_FG"
    ]
    == "EAG"
), (
    "BRTE did not resolve to EAG."
)


# ============================================================================
# 13. POSE ASSERTION
# ============================================================================

pose = master.loc[
    master[
        "observed_code"
    ].eq(
        "POSE"
    )
]

if len(pose) == 1:

    assert (
        pose.iloc[0][
            "MOSAIC_FG_structural"
        ]
        == "PerennialGrass"
    )

    pose_nat = pose.iloc[0][
        "MOSAIC_nativity_class"
    ]

    if not pd.isna(
        pose_nat
    ):

        assert (
            pose_nat
            == "Native"
        )

        assert (
            pose.iloc[0][
                "MOSAIC_FG"
            ]
            == "NPG"
        )


# ============================================================================
# 14. FORB ASSERTIONS
# ============================================================================

alde = master.loc[
    master[
        "observed_code"
    ].eq(
        "ALDE"
    )
]

if len(alde) == 1:

    assert (
        alde.iloc[0][
            "MOSAIC_FG_structural"
        ]
        == "AnnualForb"
    )

    assert (
        alde.iloc[0][
            "MOSAIC_FG"
        ]
        == "ExoticAnnualForb"
    )


phho = master.loc[
    master[
        "observed_code"
    ].eq(
        "PHHO"
    )
]

if len(phho) == 1:

    assert (
        phho.iloc[0][
            "MOSAIC_FG_structural"
        ]
        == "PerennialForb"
    )

    assert (
        phho.iloc[0][
            "MOSAIC_FG"
        ]
        == "NativePerennialForb"
    )


# ============================================================================
# 15. DISTRIBUTION OF BOTH LABEL LEVELS
# ============================================================================

plant = master[
    "code_class"
].eq(
    "Plant"
)

total_plant_records = (
    master.loc[
        plant,
        "n_records_num"
    ]
    .sum()
)


structural_summary = (
    master.loc[
        plant
    ]
    .groupby(
        "MOSAIC_FG_structural",
        dropna=False,
    )
    .agg(
        n_codes=(
            "observed_code",
            "nunique",
        ),
        n_records=(
            "n_records_num",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        "n_records",
        ascending=False,
    )
)

structural_summary[
    "pct_of_plant_records"
] = (
    100
    * structural_summary[
        "n_records"
    ]
    / total_plant_records
)


refined_summary = (
    master.loc[
        plant
    ]
    .groupby(
        "MOSAIC_FG",
        dropna=False,
    )
    .agg(
        n_codes=(
            "observed_code",
            "nunique",
        ),
        n_records=(
            "n_records_num",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        "n_records",
        ascending=False,
    )
)

refined_summary[
    "pct_of_plant_records"
] = (
    100
    * refined_summary[
        "n_records"
    ]
    / total_plant_records
)


print(
    "\n"
    + "=" * 90
)

print(
    "STRUCTURAL FUNCTIONAL GROUPS"
)

print(
    "=" * 90
)

display(
    structural_summary
)


print(
    "\n"
    + "=" * 90
)

print(
    "NATIVITY-REFINED FUNCTIONAL GROUPS"
)

print(
    "=" * 90
)

display(
    refined_summary
)


print(
    "\nTwo-level MOSAIC classification: PASS"
)

CRITICAL TAXON QA


,observed_code,USDA_accepted_symbol,USDA_scientific_name,USDA_duration,USDA_growth_habit,USDA_native_status_L48,MOSAIC_nativity_class,MOSAIC_FG_structural,USDA_MOSAIC_FG,MOSAIC_FG,FG_resolution_source,n_records_num
31,AGCR,AGCR,Agropyron cristatum (L.) Gaertn.,Perennial,Graminoid,I,Introduced,PerennialGrass,EPG,EPG,USDA_two_level_traits,86637
38,ALDE,ALDE,Alyssum desertorum Stapf,Annual,Forb/herb,I,Introduced,AnnualForb,ExoticAnnualForb,ExoticAnnualForb,USDA_two_level_traits,26258
5664,BAPRG,BAPR5,<NA>,Perennial,Subshrub,I,Introduced,Forb,Subshrub,ExoticForb,manual_ecological_functional_override,4
4193,BAPRV,BAPR5,Barleria prionitis L.,Perennial,Subshrub,I,Introduced,Forb,Subshrub,ExoticForb,manual_ecological_functional_override,14
5,BRTE,BRTE,Bromus tectorum L.,Annual,Graminoid,Introduced,Introduced,AnnualGrass,EAG,EAG,USDA_two_level_traits,429816
9,ELEL5,ELEL5,Elymus elymoides (Raf.) Swezey,Perennial,Graminoid,N,Native,PerennialGrass,NPG,NPG,USDA_two_level_traits,58454
73,ERCI6,ERCI6,Erodium cicutarium (L.) L'Hér. ex Aiton,Annual|Biennial,Forb/herb,I,Introduced,AnnualForb,ExoticAnnualForb,ExoticAnnualForb,USDA_two_level_traits,16863
34,FEID,FEID,Festuca idahoensis Elmer,Perennial,Graminoid,N,Native,PerennialGrass,NPG,NPG,USDA_two_level_traits,78604
19,HECO26,HECO26,Hesperostipa comata (Trin. & Rupr.) Barkworth,Perennial,Graminoid,N,Native,PerennialGrass,NPG,NPG,USDA_two_level_traits,80033
22,PHHO,PHHO,Phlox hoodii Richardson,Perennial,Forb/herb,N,Native,PerennialForb,NativePerennialForb,NativePerennialForb,USDA_two_level_traits,22412



STRUCTURAL FUNCTIONAL GROUPS


,MOSAIC_FG_structural,n_codes,n_records,pct_of_plant_records
7,PerennialGrass,896,2194527,40.367722
12,Woody,1374,769074,14.146905
1,AnnualGrass,163,684519,12.591539
8,Shrub,656,518383,9.535513
0,AnnualForb,1261,326670,6.009005
6,PerennialForb,2173,267135,4.913875
10,Tree,171,254418,4.679949
13,<NA>,3217,220043,4.047631
2,Forb,328,76229,1.402212
3,Grass,104,59209,1.089133



NATIVITY-REFINED FUNCTIONAL GROUPS


,MOSAIC_FG,n_codes,n_records,pct_of_plant_records
15,NPG,758,1718849,31.617755
34,Woody,1373,769071,14.14685
5,EAG,94,629830,11.58555
26,Shrub,656,518383,9.535513
6,EPG,104,470461,8.654001
31,Tree,171,254418,4.679949
18,NativePerennialForb,1924,204474,3.761243
7,ExoticAnnualForb,247,176308,3.243137
16,NativeAnnualForb,1009,150277,2.764304
22,PerennialGraminoid,599,72584,1.335163



Two-level MOSAIC classification: PASS


In [34]:
# ============================================================
# REPAIR MALFORMED USDA TRAIT CACHE ROWS
#
# Problem:
# Old recursive parser wrote rows such as:
#   USDA_plant_id = NA
#   USDA_rank = Species|Kingdom|Subkingdom|...
#   traits = NA
#
# Those symbols are "present" in the cache, so ordinary
# missing-symbol recovery will not query them.
#
# This cell:
#   1. backs up cache
#   2. identifies malformed rows
#   3. re-queries them with ROOT-LEVEL parser
#   4. replaces malformed rows
#   5. validates priority taxa and cache integrity
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import shutil
import time
import pandas as pd
import requests


# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

USDA_SERVICE_BASE = "https://plantsservices.sc.egov.usda.gov/api"

REQUEST_TIMEOUT = 30
REQUEST_DELAY_SECONDS = 0.25
MAX_RETRIES = 3
SAVE_EVERY = 20

OUTPUT_DIR = Path(OUTPUT_DIR)

REPAIR_PROGRESS_FILE = (
    OUTPUT_DIR /
    "USDA_PLANTS_ecological_attributes_malformed_repair_progress.csv"
)

REPAIR_FAILURE_FILE = (
    OUTPUT_DIR /
    "USDA_PLANTS_ecological_attributes_malformed_repair_failures.csv"
)


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def clean_text(x):
    if x is None:
        return None

    x = str(x).strip()

    if x == "" or x.lower() in {
        "nan",
        "none",
        "null",
        "<na>",
    }:
        return None

    return x


def unique_pipe(values):
    if values is None:
        return None

    if not isinstance(values, (list, tuple, set)):
        values = [values]

    out = []

    for value in values:
        value = clean_text(value)

        if value is not None and value not in out:
            out.append(value)

    return "|".join(out) if out else None


# ------------------------------------------------------------
# CORRECT ROOT-LEVEL USDA PARSER
# ------------------------------------------------------------

def parse_profile_payload(symbol, payload):

    if not isinstance(payload, dict):
        raise TypeError(
            f"{symbol}: expected dict, "
            f"got {type(payload).__name__}"
        )

    returned_symbol = clean_text(
        payload.get("Symbol")
    )

    if returned_symbol != symbol:
        raise ValueError(
            f"Requested {symbol}, "
            f"USDA returned {returned_symbol}"
        )

    duration = unique_pipe(
        payload.get("Durations")
    )

    growth_habit = unique_pipe(
        payload.get("GrowthHabits")
    )

    l48_status = None
    l48_type = None

    for item in (
        payload.get("NativeStatuses")
        or []
    ):

        if not isinstance(item, dict):
            continue

        if clean_text(
            item.get("Region")
        ) == "L48":

            l48_status = clean_text(
                item.get("Status")
            )

            l48_type = clean_text(
                item.get("Type")
            )

            break

    return {
        "USDA_accepted_symbol":
            symbol,

        "USDA_plant_id":
            payload.get("Id"),

        "USDA_rank":
            clean_text(
                payload.get("Rank")
            ),

        "USDA_duration_api":
            duration,

        "USDA_growth_habit_api":
            growth_habit,

        "USDA_L48_native_status_api":
            l48_status,

        "USDA_L48_native_type_api":
            l48_type,

        "retrieved_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }


# ------------------------------------------------------------
# FETCH ONE PROFILE
# ------------------------------------------------------------

def fetch_usda_traits(symbol):

    url = (
        f"{USDA_SERVICE_BASE}/PlantProfile"
    )

    last_error = None

    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):

        try:

            response = requests.get(
                url,
                params={
                    "symbol": symbol
                },
                timeout=REQUEST_TIMEOUT,
                headers={
                    "Accept":
                        "application/json",

                    "User-Agent":
                        "MOSAIC-USDA-cache-repair/1.0",
                },
            )

            response.raise_for_status()

            return parse_profile_payload(
                symbol,
                response.json(),
            )

        except Exception as exc:

            last_error = exc

            if attempt < MAX_RETRIES:

                time.sleep(
                    REQUEST_DELAY_SECONDS
                    * (2 ** (attempt - 1))
                )

    raise RuntimeError(
        f"{symbol}: {repr(last_error)}"
    )


# ------------------------------------------------------------
# LOAD CACHE
# ------------------------------------------------------------

cache = pd.read_csv(
    USDA_ATTRIBUTE_CACHE,
    dtype="string",
    low_memory=False,
).copy()


# Normalize possible historical names
column_aliases = {
    "accepted_symbol":
        "USDA_accepted_symbol",

    "USDA_duration":
        "USDA_duration_api",

    "USDA_growth_habit":
        "USDA_growth_habit_api",

    "USDA_native_status_L48":
        "USDA_L48_native_status_api",
}


for old, new in column_aliases.items():

    if (
        old in cache.columns
        and new not in cache.columns
    ):

        cache = cache.rename(
            columns={
                old: new
            }
        )


required = [
    "USDA_accepted_symbol",
    "USDA_plant_id",
    "USDA_rank",
    "USDA_duration_api",
    "USDA_growth_habit_api",
    "USDA_L48_native_status_api",
    "USDA_L48_native_type_api",
]


for col in required:

    if col not in cache.columns:
        cache[col] = pd.NA


cache[
    "USDA_accepted_symbol"
] = (
    cache[
        "USDA_accepted_symbol"
    ]
    .astype("string")
    .str.strip()
)


# ------------------------------------------------------------
# BACK UP CURRENT CACHE BEFORE MODIFYING
# ------------------------------------------------------------

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

backup_file = (
    OUTPUT_DIR /
    f"USDA_PLANTS_ecological_attributes_PRE_REPAIR_{timestamp}.csv"
)

cache.to_csv(
    backup_file,
    index=False,
)

print(
    "Backup written to:"
)

print(
    backup_file
)


# ------------------------------------------------------------
# IDENTIFY MALFORMED ROWS
# ------------------------------------------------------------
#
# Strong evidence of recursive-parser corruption:
#
# 1. USDA plant ID missing
#       Root PlantProfile should provide Id.
#
# 2. Rank contains "|"
#       Root rank is one value.
#       "Species|Kingdom|..." proves recursive aggregation.
#
# Either condition is enough to trigger repair.
# ------------------------------------------------------------

plant_id_missing = (
    cache[
        "USDA_plant_id"
    ]
    .isna()
    |
    cache[
        "USDA_plant_id"
    ]
    .astype("string")
    .str.strip()
    .isin(
        [
            "",
            "<NA>",
            "nan",
            "None",
        ]
    )
)


rank_has_pipe = (
    cache[
        "USDA_rank"
    ]
    .astype("string")
    .str.contains(
        r"\|",
        regex=True,
        na=False,
    )
)


malformed_mask = (
    plant_id_missing
    |
    rank_has_pipe
)


malformed = (
    cache.loc[
        malformed_mask
    ]
    .copy()
)


repair_symbols = (
    malformed[
        "USDA_accepted_symbol"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)


print()
print(
    f"Current cache taxa: "
    f"{cache['USDA_accepted_symbol'].nunique():,}"
)

print(
    f"Malformed cache rows detected: "
    f"{len(malformed):,}"
)

print(
    f"Unique taxa requiring repair: "
    f"{len(repair_symbols):,}"
)


# ------------------------------------------------------------
# SHOW WHY THEY WERE FLAGGED
# ------------------------------------------------------------

display(
    malformed[
        [
            "USDA_accepted_symbol",
            "USDA_plant_id",
            "USDA_rank",
            "USDA_duration_api",
            "USDA_growth_habit_api",
            "USDA_L48_native_status_api",
            "USDA_L48_native_type_api",
            "retrieved_utc",
        ]
    ]
    .head(30)
)


# ------------------------------------------------------------
# LOAD ANY VALID REPAIR PROGRESS
# ------------------------------------------------------------

if REPAIR_PROGRESS_FILE.exists():

    repaired = pd.read_csv(
        REPAIR_PROGRESS_FILE,
        dtype="string",
        low_memory=False,
    )

    repaired_symbols = set(
        repaired[
            "USDA_accepted_symbol"
        ]
        .dropna()
        .astype(str)
        .str.strip()
    )

else:

    repaired = pd.DataFrame()
    repaired_symbols = set()


remaining = [
    symbol
    for symbol in repair_symbols
    if symbol not in repaired_symbols
]


print()
print(
    f"Previously repaired: "
    f"{len(repaired_symbols):,}"
)

print(
    f"Remaining this run: "
    f"{len(remaining):,}"
)


# ------------------------------------------------------------
# REPAIR LOOP
# ------------------------------------------------------------

pending_rows = []
pending_failures = []


for i, symbol in enumerate(
    remaining,
    start=1,
):

    try:

        row = fetch_usda_traits(
            symbol
        )

        pending_rows.append(
            row
        )

        print(
            f"[{i:03d}/{len(remaining):03d}] "
            f"{symbol:<10} "
            f"ID={str(row['USDA_plant_id']):<8} "
            f"R={str(row['USDA_rank']):<12} "
            f"D={str(row['USDA_duration_api']):<22} "
            f"H={str(row['USDA_growth_habit_api']):<24} "
            f"L48={str(row['USDA_L48_native_type_api'])}"
        )

    except Exception as exc:

        pending_failures.append(
            {
                "USDA_accepted_symbol":
                    symbol,

                "error":
                    repr(exc),

                "failed_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
            }
        )

        print(
            f"[{i:03d}/{len(remaining):03d}] "
            f"{symbol:<10} FAILED | "
            f"{repr(exc)}"
        )


    # --------------------------------------------------------
    # CHECKPOINT
    # --------------------------------------------------------

    if (
        i % SAVE_EVERY == 0
        or i == len(remaining)
    ):

        if pending_rows:

            batch = pd.DataFrame(
                pending_rows
            )

            if repaired.empty:

                repaired = batch.copy()

            else:

                repaired = pd.concat(
                    [
                        repaired,
                        batch,
                    ],
                    ignore_index=True,
                    sort=False,
                )


            repaired = (
                repaired
                .drop_duplicates(
                    subset=[
                        "USDA_accepted_symbol"
                    ],
                    keep="last",
                )
                .sort_values(
                    "USDA_accepted_symbol"
                )
                .reset_index(
                    drop=True
                )
            )


            repaired.to_csv(
                REPAIR_PROGRESS_FILE,
                index=False,
            )

            pending_rows = []


        if pending_failures:

            failure_batch = pd.DataFrame(
                pending_failures
            )

            if REPAIR_FAILURE_FILE.exists():

                old_failures = pd.read_csv(
                    REPAIR_FAILURE_FILE,
                    dtype="string",
                    low_memory=False,
                )

                failure_batch = pd.concat(
                    [
                        old_failures,
                        failure_batch,
                    ],
                    ignore_index=True,
                    sort=False,
                )


            failure_batch.to_csv(
                REPAIR_FAILURE_FILE,
                index=False,
            )

            pending_failures = []


    time.sleep(
        REQUEST_DELAY_SECONDS
    )


# ------------------------------------------------------------
# RELOAD COMPLETE REPAIR TABLE
# ------------------------------------------------------------

if REPAIR_PROGRESS_FILE.exists():

    repaired = pd.read_csv(
        REPAIR_PROGRESS_FILE,
        dtype="string",
        low_memory=False,
    )

else:

    repaired = pd.DataFrame()


print()
print(
    f"Successfully repaired taxa: "
    f"{len(repaired):,}"
)


# ------------------------------------------------------------
# REPLACE MALFORMED CACHE ROWS
# ------------------------------------------------------------
#
# IMPORTANT:
# This is replacement, not append + keep-first.
# ------------------------------------------------------------

repaired_symbols = set(
    repaired[
        "USDA_accepted_symbol"
    ]
    .dropna()
    .astype(str)
    .str.strip()
)


# Remove ONLY rows successfully replaced.
# Failed queries remain in original cache for inspection.

cache_good = cache.loc[
    ~cache[
        "USDA_accepted_symbol"
    ].isin(
        repaired_symbols
    )
].copy()


updated_cache = pd.concat(
    [
        cache_good,
        repaired,
    ],
    ignore_index=True,
    sort=False,
)


updated_cache = (
    updated_cache
    .drop_duplicates(
        subset=[
            "USDA_accepted_symbol"
        ],
        keep="last",
    )
    .sort_values(
        "USDA_accepted_symbol"
    )
    .reset_index(
        drop=True
    )
)


updated_cache.to_csv(
    USDA_ATTRIBUTE_CACHE,
    index=False,
)


print()
print(
    f"Updated cache taxa: "
    f"{updated_cache['USDA_accepted_symbol'].nunique():,}"
)

print(
    "Repaired cache written to:"
)

print(
    USDA_ATTRIBUTE_CACHE
)


# ------------------------------------------------------------
# FINAL MALFORMED-ROW QA
# ------------------------------------------------------------

id_still_missing = (
    updated_cache[
        "USDA_plant_id"
    ]
    .isna()
)

rank_still_bad = (
    updated_cache[
        "USDA_rank"
    ]
    .astype("string")
    .str.contains(
        r"\|",
        regex=True,
        na=False,
    )
)


still_bad = updated_cache.loc[
    id_still_missing
    |
    rank_still_bad
].copy()


print()
print(
    f"Rows still structurally malformed: "
    f"{len(still_bad):,}"
)


if len(still_bad):

    display(
        still_bad[
            [
                "USDA_accepted_symbol",
                "USDA_plant_id",
                "USDA_rank",
                "USDA_duration_api",
                "USDA_growth_habit_api",
                "USDA_L48_native_status_api",
                "USDA_L48_native_type_api",
            ]
        ]
        .head(50)
    )


# ------------------------------------------------------------
# PRIORITY TAXON QA
# ------------------------------------------------------------

priority_symbols = [
    "PSSP6",
    "POPR",
    "AGCR",
    "HECO26",
    "FEID",
    "ELEL5",
    "BRIN2",
    "CHVI8",
    "SCAR7",
    "ARTR2",
    "TACA8",
    "ALDE",
    "SIAL2",
    "ERCI6",
    "TRDU",
]


priority_qa = (
    updated_cache.loc[
        updated_cache[
            "USDA_accepted_symbol"
        ].isin(
            priority_symbols
        ),
        [
            "USDA_accepted_symbol",
            "USDA_plant_id",
            "USDA_rank",
            "USDA_duration_api",
            "USDA_growth_habit_api",
            "USDA_L48_native_status_api",
            "USDA_L48_native_type_api",
            "retrieved_utc",
        ],
    ]
    .sort_values(
        "USDA_accepted_symbol"
    )
)


print(
    "\nPriority taxa after repair:"
)

display(
    priority_qa
)


print(
    "\nCACHE REPAIR COMPLETE"
)

Backup written to:
C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\species_dictionary_outputs\USDA_PLANTS_ecological_attributes_PRE_REPAIR_20260903_094210.csv

Current cache taxa: 7,269
Malformed cache rows detected: 1
Unique taxa requiring repair: 1


,USDA_accepted_symbol,USDA_plant_id,USDA_rank,USDA_duration_api,USDA_growth_habit_api,USDA_L48_native_status_api,USDA_L48_native_type_api,retrieved_utc
5458,POAR22,<NA>,Variety|Kingdom|Subkingdom|Superdivision|Divis...,<NA>,<NA>,<NA>,<NA>,2026-09-01T20:09:22.201544+00:00



Previously repaired: 476
Remaining this run: 1
[001/001] POAR22     FAILED | RuntimeError("POAR22: ValueError('Requested POAR22, USDA returned PONIN')")

Successfully repaired taxa: 476

Updated cache taxa: 7,269
Repaired cache written to:
C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\species_dictionary_outputs\USDA_PLANTS_ecological_attributes.csv

Rows still structurally malformed: 1


,USDA_accepted_symbol,USDA_plant_id,USDA_rank,USDA_duration_api,USDA_growth_habit_api,USDA_L48_native_status_api,USDA_L48_native_type_api
5458,POAR22,<NA>,Variety|Kingdom|Subkingdom|Superdivision|Divis...,<NA>,<NA>,<NA>,<NA>



Priority taxa after repair:


,USDA_accepted_symbol,USDA_plant_id,USDA_rank,USDA_duration_api,USDA_growth_habit_api,USDA_L48_native_status_api,USDA_L48_native_type_api,retrieved_utc
131,AGCR,20068,Species,Perennial,Graminoid,I,Introduced,2026-09-01T20:19:14.115727+00:00
207,ALDE,61805,Species,Annual,Forb/herb,I,Introduced,2026-09-01T20:19:23.118227+00:00
642,ARTR2,32390,Species,Perennial,Shrub|Tree,N,Native,2026-09-01T20:19:59.612017+00:00
1085,BRIN2,20954,Species,Perennial,Graminoid,I,Introduced,2026-09-01T20:20:20.233090+00:00
1755,CHVI8,33492,Species,Perennial,Shrub,N,Native,2026-09-01T20:20:52.319662+00:00
2543,ELEL5,22420,Species,Perennial,Graminoid,N,Native,2026-09-01T20:21:38.980060+00:00
2711,ERCI6,83665,Species,Annual|Biennial,Forb/herb,I,Introduced,2026-09-01T20:21:59.522310+00:00
3093,FEID,23094,Species,Perennial,Graminoid,N,Native,2026-09-01T20:22:28.108292+00:00
3411,HECO26,23323,Species,Perennial,Graminoid,N,Native,2026-09-01T20:22:52.765229+00:00
5592,POPR,25028,Species,Perennial,Graminoid,I,Introduced,2026-09-01T20:25:09.294033+00:00



CACHE REPAIR COMPLETE


In [ ]:
# ============================================================
# PRE-QA HOTFIX / NORMALIZATION CELL
#
# Purpose:
#   Apply known final corrections directly to dictionary_fg
#   before running the hard QA/QC gate.
#
# This does NOT overwrite observed_code provenance.
# ============================================================

import pandas as pd

final = dictionary_fg.copy()


# ============================================================
# 1. POAR2R2 -> accepted USDA taxon PONIN
# ============================================================

mask = final["observed_code"].eq("POAR2R2")

assert mask.sum() == 1, (
    f"Expected exactly one POAR2R2 row, found {mask.sum()}."
)

idx = final.index[mask][0]

final.loc[idx, "USDA_symbol"] = "POAR2R2"
final.loc[idx, "USDA_accepted_symbol"] = "PONIN"

if "resolved_label" in final.columns:
    final.loc[idx, "resolved_label"] = "Potentilla nivea var. nivea"

if "code_class" in final.columns:
    final.loc[idx, "code_class"] = "Plant"

if "resolution_source" in final.columns:
    final.loc[idx, "resolution_source"] = "manual_USDA_synonym_resolution"

if "resolved" in final.columns:
    final.loc[idx, "resolved"] = True

if "needs_review" in final.columns:
    final.loc[idx, "needs_review"] = False

if "manual_note" in final.columns:
    final.loc[idx, "manual_note"] = (
        "POAR2R2 manually resolved through USDA synonym POAR22 "
        "to accepted symbol PONIN, Potentilla nivea var. nivea."
    )


# ============================================================
# 2. Pull PONIN taxonomy/traits from an existing PONIN row
#    if available in the current dictionary
# ============================================================

ponin = final.loc[
    final["USDA_accepted_symbol"].eq("PONIN")
    & ~final["observed_code"].eq("POAR2R2")
].copy()

if len(ponin):

    src = ponin.iloc[0]

    copy_cols = [
        "USDA_scientific_name",
        "USDA_common_name",
        "USDA_family",
        "USDA_genus",
        "USDA_rank",
        "USDA_duration",
        "USDA_growth_habit",
        "USDA_native_status",
        "USDA_native_status_L48",
        "USDA_wetland_status",
        "scientific_name",
        "common_name",
        "family",
        "genus",
        "taxonomic_rank",
        "native_status",
        "duration",
        "growth_habit",
        "wetland_status",
        "accepted_name_USDA",
        "SG_Group_resolved",
        "MOSAIC_FG",
        "FG_resolution_source",
    ]

    for col in copy_cols:
        if col in final.columns and col in src.index:
            final.loc[idx, col] = src[col]


# ============================================================
# 3. Restore __NO_CANOPY__ / literal NONE invariant
#    if stale notebook state reintroduced the collision
# ============================================================

# Identify by known record signatures, not row index.
no_canopy_mask = (
    final["n_records"].eq(442612)
    & final["n_top"].eq(442612)
)

literal_none_mask = (
    final["n_records"].eq(398)
    & final["n_top"].eq(394)
)

assert no_canopy_mask.sum() == 1
assert literal_none_mask.sum() == 1

no_canopy_idx = final.index[no_canopy_mask][0]
literal_none_idx = final.index[literal_none_mask][0]


# True blank sentinel
final.loc[no_canopy_idx, "observed_code"] = "__NO_CANOPY__"

if "code_upper" in final.columns:
    final.loc[no_canopy_idx, "code_upper"] = "__NO_CANOPY__"

if "resolved_label" in final.columns:
    final.loc[no_canopy_idx, "resolved_label"] = "No canopy contact"

if "code_class" in final.columns:
    final.loc[no_canopy_idx, "code_class"] = "NoCanopy"

if "MOSAIC_FG" in final.columns:
    final.loc[no_canopy_idx, "MOSAIC_FG"] = "NO_CANOPY"

if "resolution_source" in final.columns:
    final.loc[no_canopy_idx, "resolution_source"] = "blank_source_sentinel"


taxonomy_cols = [
    "scientific_name",
    "common_name",
    "family",
    "genus",
    "taxonomic_rank",
    "native_status",
    "duration",
    "growth_habit",
    "wetland_status",
    "synonym",
    "accepted_name_USDA",
    "USDA_symbol",
    "USDA_accepted_symbol",
    "USDA_match_type",
    "USDA_scientific_name",
    "USDA_common_name",
    "USDA_family",
    "USDA_genus",
    "USDA_rank",
    "USDA_duration",
    "USDA_growth_habit",
    "USDA_native_status",
    "USDA_native_status_L48",
    "USDA_wetland_status",
]

for col in taxonomy_cols:
    if col in final.columns:
        final.loc[no_canopy_idx, col] = pd.NA


# Genuine USDA plant symbol NONE
final.loc[literal_none_idx, "observed_code"] = "NONE"

if "code_upper" in final.columns:
    final.loc[literal_none_idx, "code_upper"] = "NONE"

if "code_class" in final.columns:
    final.loc[literal_none_idx, "code_class"] = "Plant"

if "USDA_symbol" in final.columns:
    final.loc[literal_none_idx, "USDA_symbol"] = "NONE"

if "USDA_accepted_symbol" in final.columns:
    final.loc[literal_none_idx, "USDA_accepted_symbol"] = "NONE"


# ============================================================
# 4. BASSIA final ecological override
# ============================================================

bassia_mask = final["observed_code"].isin(["BAPRV", "BAPRG"])

if bassia_mask.any():

    final.loc[
        bassia_mask,
        "USDA_accepted_symbol"
    ] = "BAPR5"

    if "MOSAIC_FG" in final.columns:
        final.loc[
            bassia_mask,
            "MOSAIC_FG"
        ] = "ExoticForb"

    if "FG_resolution_source" in final.columns:
        final.loc[
            bassia_mask,
            "FG_resolution_source"
        ] = "manual_ecological_functional_override"


# ============================================================
# 5. REMOVE POAR22 AS AN ACCEPTED SYMBOL
# ============================================================

# POAR22 is a synonym/legacy symbol, not the accepted taxon.
assert not final[
    "USDA_accepted_symbol"
].eq("POAR22").any(), (
    "POAR22 still survives as an accepted USDA symbol."
)


# ============================================================
# 6. CORE STRUCTURAL CHECKS
# ============================================================

assert len(final) == 10960

assert (
    final["observed_code"]
    .nunique(dropna=False)
    == 10960
)

assert (
    final["observed_code"]
    .duplicated(keep=False)
    .sum()
    == 0
)

assert (
    final["observed_code"]
    .eq("__NO_CANOPY__")
    .sum()
    == 1
)

assert (
    final["observed_code"]
    .eq("NONE")
    .sum()
    == 1
)

poar_check = final.loc[
    final["observed_code"].eq("POAR2R2"),
    [
        c for c in [
            "observed_code",
            "resolved_label",
            "USDA_symbol",
            "USDA_accepted_symbol",
            "USDA_scientific_name",
            "USDA_rank",
            "MOSAIC_FG",
            "resolution_source",
        ]
        if c in final.columns
    ]
]

print("POAR2R2 after hotfix:")
display(poar_check)

print("\nSentinel check:")
display(
    final.loc[
        final["observed_code"].isin(
            ["__NO_CANOPY__", "NONE"]
        ),
        [
            c for c in [
                "observed_code",
                "n_records",
                "n_top",
                "code_class",
                "USDA_accepted_symbol",
                "USDA_scientific_name",
                "MOSAIC_FG",
            ]
            if c in final.columns
        ]
    ]
)

print("\nPre-QA hotfix: PASS")


# ============================================================
# 7. Make QA use this corrected object
# ============================================================

dictionary_fg = final.copy()

POAR2R2 after hotfix:


,observed_code,resolved_label,USDA_symbol,USDA_accepted_symbol,USDA_scientific_name,MOSAIC_FG,resolution_source
10143,POAR2R2,Potentilla nivea var. nivea,POAR2R2,PONIN,<NA>,TraitUnresolvedPlant,manual_USDA_synonym_resolution



Sentinel check:


,observed_code,n_records,n_top,code_class,USDA_accepted_symbol,USDA_scientific_name,MOSAIC_FG
15,__NO_CANOPY__,442612,442612,NoCanopy,<NA>,<NA>,NO_CANOPY
2527,NONE,398,394,Plant,NONE,Notholaena neglecta Maxon,TraitUnresolvedPlant



Pre-QA hotfix: PASS


In [35]:
# ============================================================
# TRUE FINAL QA/QC OF CURRENT IN-MEMORY DICTIONARY
#
# Run AFTER:
#   - manual alias corrections
#   - dictionary construction
#   - USDA taxonomy resolution
#   - USDA trait merge
#   - MOSAIC FG derivation
#
# Authoritative object: dictionary_fg
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present"
)

OUTPUT_DIR = BASE_DIR / "species_dictionary_outputs"

RAW_CODES_FILE = (
    BASE_DIR /
    "LDC_LPI_observed_code_unique.csv"
)

FINAL_MASTER_FILE = (
    OUTPUT_DIR /
    "LDC_LPI_species_dictionary_FINAL_MASTER.csv"
)

raw = pd.read_csv(
    RAW_CODES_FILE,
    dtype={"code": "string"},
    low_memory=False
)

final = dictionary_fg.copy()


# ============================================================
# 1. RECONSTRUCT AUTHORITATIVE RAW CANONICAL CODE
# ============================================================

raw["observed_code_raw"] = raw["code"]

raw["source_code_was_blank"] = (
    raw["observed_code_raw"].isna()
    |
    raw["observed_code_raw"].str.strip().eq("")
)

raw["observed_code"] = (
    raw["observed_code_raw"]
    .astype("string")
    .str.strip()
)

raw.loc[
    raw["source_code_was_blank"],
    "observed_code"
] = "__NO_CANOPY__"


# ============================================================
# 2. BASIC STRUCTURE
# ============================================================

print("=" * 78)
print("FINAL DICTIONARY STRUCTURE")
print("=" * 78)

print(f"Raw code rows:             {len(raw):,}")
print(f"Final dictionary rows:     {len(final):,}")

print(
    "Raw canonical codes:      "
    f"{raw['observed_code'].nunique(dropna=False):,}"
)

print(
    "Final unique codes:       "
    f"{final['observed_code'].nunique(dropna=False):,}"
)


duplicate_codes = final.loc[
    final["observed_code"].duplicated(keep=False)
].copy()

print(
    f"Duplicate final codes:     {len(duplicate_codes):,}"
)


# ============================================================
# 3. EXACT CODE-UNIVERSE ACCOUNTING
# ============================================================

raw_set = set(
    raw["observed_code"]
    .dropna()
    .astype(str)
)

final_set = set(
    final["observed_code"]
    .dropna()
    .astype(str)
)

missing_from_final = sorted(
    raw_set - final_set
)

extra_in_final = sorted(
    final_set - raw_set
)

print()
print("=" * 78)
print("CODE-UNIVERSE ACCOUNTING")
print("=" * 78)

print(
    f"Raw codes absent from final: {len(missing_from_final):,}"
)

print(
    f"Unexpected final codes:      {len(extra_in_final):,}"
)

if missing_from_final:
    print("\nMissing:")
    print(missing_from_final)

if extra_in_final:
    print("\nExtra:")
    print(extra_in_final)


# ============================================================
# 4. RECORD ACCOUNTING
# ============================================================

raw_records = (
    pd.to_numeric(
        raw["n_records"],
        errors="coerce"
    )
    .fillna(0)
    .sum()
)

final_records = (
    pd.to_numeric(
        final["n_records"],
        errors="coerce"
    )
    .fillna(0)
    .sum()
)

print()
print("=" * 78)
print("RECORD ACCOUNTING")
print("=" * 78)

print(f"Raw records:       {raw_records:,.0f}")
print(f"Final records:     {final_records:,.0f}")
print(f"Difference:        {final_records - raw_records:,.0f}")


# ============================================================
# 5. CRITICAL NONE / NO-CANOPY INVARIANT
# ============================================================

sentinel = final.loc[
    final["observed_code"].isin(
        ["__NO_CANOPY__", "NONE"]
    )
].copy()

print()
print("=" * 78)
print("NONE / __NO_CANOPY__")
print("=" * 78)

display(
    sentinel[
        [
            c for c in [
                "observed_code",
                "source_code_was_blank",
                "n_records",
                "n_top",
                "n_lower",
                "resolved_label",
                "code_class",
                "USDA_accepted_symbol",
                "USDA_scientific_name",
                "MOSAIC_FG",
                "resolution_source",
                "FG_resolution_source",
            ]
            if c in sentinel.columns
        ]
    ]
)

assert (
    final["observed_code"]
    .eq("__NO_CANOPY__")
    .sum()
    == 1
)

assert (
    final["observed_code"]
    .eq("NONE")
    .sum()
    == 1
)

no_canopy = final.loc[
    final["observed_code"].eq("__NO_CANOPY__")
].iloc[0]

literal_none = final.loc[
    final["observed_code"].eq("NONE")
].iloc[0]

assert no_canopy["n_records"] == 442612
assert literal_none["n_records"] == 398

assert no_canopy["code_class"] == "NoCanopy"
assert literal_none["code_class"] == "Plant"

assert pd.isna(
    no_canopy["USDA_accepted_symbol"]
)

assert (
    literal_none["USDA_accepted_symbol"]
    == "NONE"
)


# ============================================================
# 6. POAR2R2 CORRECTION
# ============================================================

poar = final.loc[
    final["observed_code"].eq("POAR2R2")
].copy()

print()
print("=" * 78)
print("POAR2R2")
print("=" * 78)

print("Rows:", len(poar))

display(
    poar[
        [
            c for c in [
                "observed_code",
                "n_records",
                "resolved_label",
                "code_class",
                "USDA_symbol",
                "USDA_accepted_symbol",
                "USDA_match_method",
                "USDA_scientific_name",
                "USDA_common_name",
                "USDA_family",
                "USDA_rank",
                "USDA_duration",
                "USDA_growth_habit",
                "USDA_native_status_L48",
                "MOSAIC_FG",
                "resolution_source",
                "FG_resolution_source",
            ]
            if c in poar.columns
        ]
    ]
)

assert len(poar) == 1

assert (
    poar.iloc[0]["USDA_accepted_symbol"]
    == "PONIN"
), (
    "POAR2R2 has not been corrected to accepted "
    "USDA symbol PONIN."
)


# ============================================================
# 7. BASSIA
# ============================================================

bassia = final.loc[
    final["observed_code"].isin(
        ["BAPRV", "BAPRG"]
    )
].copy()

print()
print("=" * 78)
print("BASSIA")
print("=" * 78)

display(
    bassia[
        [
            c for c in [
                "observed_code",
                "USDA_accepted_symbol",
                "USDA_scientific_name",
                "MOSAIC_FG",
                "resolution_source",
                "FG_resolution_source",
                "n_records",
            ]
            if c in bassia.columns
        ]
    ]
)

if len(bassia):

    assert (
        bassia["USDA_accepted_symbol"]
        .eq("BAPR5")
        .all()
    )

    assert (
        bassia["MOSAIC_FG"]
        .eq("ExoticForb")
        .all()
    )


# ============================================================
# 8. USDA ACCEPTED SYMBOLS THAT SHOULD NEVER SURVIVE
# ============================================================

bad_accepted_symbols = {
    "POAR22",
}

bad_taxa = final.loc[
    final["USDA_accepted_symbol"].isin(
        bad_accepted_symbols
    )
].copy()

print()
print("=" * 78)
print("OBSOLETE / INVALID ACCEPTED SYMBOLS")
print("=" * 78)

print(
    f"Rows carrying obsolete accepted symbols: "
    f"{len(bad_taxa):,}"
)

if len(bad_taxa):
    display(bad_taxa)

assert len(bad_taxa) == 0


# ============================================================
# 9. PLANT FG COMPLETENESS
# ============================================================

plants = final.loc[
    final["code_class"].eq("Plant")
].copy()

missing_fg = plants.loc[
    plants["MOSAIC_FG"].isna()
].copy()

print()
print("=" * 78)
print("PLANT FUNCTIONAL-GROUP COVERAGE")
print("=" * 78)

print(f"Plant codes:              {len(plants):,}")
print(
    f"Plant records:            "
    f"{plants['n_records'].sum():,}"
)

print(
    f"Plant codes missing FG:   "
    f"{len(missing_fg):,}"
)

print(
    f"Plant records missing FG: "
    f"{missing_fg['n_records'].sum():,}"
)


# ============================================================
# 10. ROW-LEVEL CONTEXT FLAGS
# ============================================================

if "requires_row_level_resolution" in final.columns:

    context_rows = final.loc[
        final[
            "requires_row_level_resolution"
        ].fillna(False)
    ].copy()

    print()
    print("=" * 78)
    print("ROW-LEVEL CONTEXT-DEPENDENT CODES")
    print("=" * 78)

    print(
        f"Context-dependent codes:   "
        f"{len(context_rows):,}"
    )

    print(
        f"Represented records:        "
        f"{context_rows['n_records'].sum():,}"
    )

    display(
        context_rows[
            [
                c for c in [
                    "observed_code",
                    "n_records",
                    "resolved_label",
                    "code_class",
                    "protocol_source",
                    "context_protocol_label",
                    "context_protocol_class",
                    "context_protocol_rule",
                    "MOSAIC_FG",
                ]
                if c in context_rows.columns
            ]
        ]
        .sort_values(
            "n_records",
            ascending=False
        )
    )


# ============================================================
# 11. HARD FINAL GATE
# ============================================================

checks = {
    "10,960 dictionary rows":
        len(final) == 10960,

    "10,960 unique observed codes":
        final["observed_code"].nunique(
            dropna=False
        ) == 10960,

    "No duplicate observed codes":
        len(duplicate_codes) == 0,

    "All raw codes represented":
        len(missing_from_final) == 0,

    "No unexpected codes":
        len(extra_in_final) == 0,

    "Exact record accounting":
        np.isclose(
            raw_records,
            final_records
        ),

    "One __NO_CANOPY__":
        final["observed_code"]
        .eq("__NO_CANOPY__")
        .sum() == 1,

    "One literal NONE":
        final["observed_code"]
        .eq("NONE")
        .sum() == 1,

    "POAR2R2 resolves to PONIN":
        (
            len(poar) == 1
            and
            poar.iloc[0][
                "USDA_accepted_symbol"
            ] == "PONIN"
        ),

    "No POAR22 accepted symbol remains":
        len(bad_taxa) == 0,

    "All plant codes have MOSAIC_FG":
        len(missing_fg) == 0,
}


print()
print("=" * 78)
print("FINAL GATE")
print("=" * 78)

all_pass = True

for name, passed in checks.items():

    status = (
        "PASS"
        if passed
        else "FAIL"
    )

    print(
        f"{status:<5}  {name}"
    )

    if not passed:
        all_pass = False


# ============================================================
# 12. WRITE ONLY IF EVERYTHING PASSES
# ============================================================

if not all_pass:

    raise RuntimeError(
        "Final dictionary failed one or more hard QA gates. "
        "Nothing was written."
    )


final.to_csv(
    FINAL_MASTER_FILE,
    index=False
)

print()
print("=" * 78)
print("FINAL MASTER WRITTEN")
print("=" * 78)

print(FINAL_MASTER_FILE)

FINAL DICTIONARY STRUCTURE
Raw code rows:             10,960
Final dictionary rows:     10,960
Raw canonical codes:      10,960
Final unique codes:       10,960
Duplicate final codes:     0

CODE-UNIVERSE ACCOUNTING
Raw codes absent from final: 0
Unexpected final codes:      0

RECORD ACCOUNTING
Raw records:       16,150,101
Final records:     16,150,101
Difference:        0

NONE / __NO_CANOPY__


,observed_code,source_code_was_blank,n_records,n_top,n_lower,resolved_label,code_class,USDA_accepted_symbol,USDA_scientific_name,MOSAIC_FG,resolution_source,FG_resolution_source
15,__NO_CANOPY__,True,442612,442612,0,No top-canopy contact,NoCanopy,<NA>,<NA>,NaN,source_blank_sentinel,<NA>
2527,NONE,False,398,394,4,Notholaena neglecta Maxon,Plant,NONE,Notholaena neglecta Maxon,TraitUnresolvedPlant,USDA_accepted_symbol,USDA_taxon_missing_traits



POAR2R2
Rows: 1


,observed_code,n_records,resolved_label,code_class,USDA_accepted_symbol,USDA_match_method,USDA_scientific_name,USDA_common_name,USDA_family,USDA_duration,USDA_growth_habit,USDA_native_status_L48,MOSAIC_FG,resolution_source,FG_resolution_source
10143,POAR2R2,10,Potentilla nivea var. nivea,Plant,PONIN,manual_alias,Potentilla nivea L. var. nivea,snow cinquefoil,Rosaceae,<NA>,<NA>,<NA>,TraitUnresolvedPlant,manual_USDA_synonym_resolution,USDA_taxon_missing_traits



BASSIA


,observed_code,USDA_accepted_symbol,USDA_scientific_name,MOSAIC_FG,resolution_source,FG_resolution_source,n_records
4193,BAPRV,BAPR,Barleria prionitis L.,Forb,manual_taxonomic_alias,protocol_or_project_group,14
5664,BAPRG,<NA>,<NA>,NaN,<NA>,protocol_or_project_group,4


AssertionError: 

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present"
)

OUTPUT_DIR = BASE_DIR / "species_dictionary_outputs"

files_to_check = [
    "USDA_PLANTS_ecological_attributes.csv",
    "USDA_PLANTS_ecological_attributes_final.csv",
    "LDC_LPI_species_dictionary_MASTER.csv",
    "LDC_LPI_species_dictionary_pre_functional_group.csv",
    "LDC_LPI_species_dictionary_QA.csv",
]

for fname in files_to_check:
    path = OUTPUT_DIR / fname

    print("\n" + "=" * 80)
    print(fname)
    print("=" * 80)

    df = pd.read_csv(path, low_memory=False)

    print("rows:", len(df))
    print("columns:")
    for c in df.columns:
        print("  ", repr(c))


USDA_PLANTS_ecological_attributes.csv
rows: 7268
columns:
   'USDA_accepted_symbol'
   'USDA_plant_id'
   'USDA_rank'
   'USDA_duration_api'
   'USDA_growth_habit_api'
   'USDA_L48_native_status_api'
   'USDA_L48_native_type_api'
   'retrieved_utc'

USDA_PLANTS_ecological_attributes_final.csv
rows: 6790
columns:
   'accepted_symbol'
   'USDA_plant_id'
   'USDA_rank'
   'USDA_duration_api'
   'USDA_growth_habit_api'
   'USDA_L48_native_status_api'
   'USDA_L48_native_type_api'
   'retrieved_utc'
   'duration_set'
   'growth_habit_set'
   'n_duration_values'
   'n_growth_habit_values'
   'duration_ambiguous'
   'growth_habit_ambiguous'

LDC_LPI_species_dictionary_MASTER.csv
rows: 10960
columns:
   'observed_code'
   'code_upper'
   'n_records'
   'n_plot_visits'
   'n_top'
   'n_lower'
   'n_soil_surface'
   'n_top_plots'
   'n_multilayer_plots'
   'n_scope_top'
   'n_scope_multilayer'
   'n_scope_raw'
   'first_year'
   'last_year'
   'resolved_label'
   'code_class'
   'resolution_sou

In [ ]:
from pathlib import Path

BASE_DIR = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present"
)

OUTPUT_DIR = BASE_DIR / "species_dictionary_outputs"

for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print(p.name)

LDC_LPI_canopymultilayer_dictionary.csv
LDC_LPI_canopymultilayer_review_codes.csv
LDC_LPI_canopymultilayer_species_list.csv
LDC_LPI_multilayer_dictionary.csv
LDC_LPI_multilayer_species_list.csv
LDC_LPI_multilayer_unresolved_codes.csv
LDC_LPI_protocol_code_rules.csv
LDC_LPI_rawallcontacts_dictionary.csv
LDC_LPI_rawallcontacts_review_codes.csv
LDC_LPI_rawallcontacts_species_list.csv
LDC_LPI_rawallcontacts_unresolved_codes.csv
LDC_LPI_species_dictionary_MASTER.csv
LDC_LPI_species_dictionary_pre_functional_group.csv
LDC_LPI_species_dictionary_QA.csv
LDC_LPI_topcanopy_dictionary.csv
LDC_LPI_topcanopy_review_codes.csv
LDC_LPI_topcanopy_species_list.csv
LDC_LPI_topcanopy_unresolved_codes.csv
USDA_PLANTS_ecological_attributes.csv
USDA_PLANTS_ecological_attributes_final.csv
USDA_PLANTS_ecological_attributes_malformed_repair_failures.csv
USDA_PLANTS_ecological_attributes_malformed_repair_progress.csv
USDA_PLANTS_ecological_attributes_missing_taxa_progress_BAD_PARSER_BACKUP.csv
USDA_PLANTS_ecolog

In [ ]:
# ============================================================
# TRUE FINAL QA/QC OF CURRENT IN-MEMORY DICTIONARY
#
# Run AFTER:
#   - manual alias corrections
#   - dictionary construction
#   - USDA taxonomy resolution
#   - USDA trait merge
#   - MOSAIC FG derivation
#
# Authoritative object: dictionary_fg
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present"
)

OUTPUT_DIR = BASE_DIR / "species_dictionary_outputs"

RAW_CODES_FILE = (
    BASE_DIR /
    "LDC_LPI_observed_code_unique.csv"
)

FINAL_MASTER_FILE = (
    OUTPUT_DIR /
    "LDC_LPI_species_dictionary_FINAL_MASTER.csv"
)

raw = pd.read_csv(
    RAW_CODES_FILE,
    dtype={"code": "string"},
    low_memory=False
)

final = dictionary_fg.copy()


# ============================================================
# 1. RECONSTRUCT AUTHORITATIVE RAW CANONICAL CODE
# ============================================================

raw["observed_code_raw"] = raw["code"]

raw["source_code_was_blank"] = (
    raw["observed_code_raw"].isna()
    |
    raw["observed_code_raw"].str.strip().eq("")
)

raw["observed_code"] = (
    raw["observed_code_raw"]
    .astype("string")
    .str.strip()
)

raw.loc[
    raw["source_code_was_blank"],
    "observed_code"
] = "__NO_CANOPY__"


# ============================================================
# 2. BASIC STRUCTURE
# ============================================================

print("=" * 78)
print("FINAL DICTIONARY STRUCTURE")
print("=" * 78)

print(f"Raw code rows:             {len(raw):,}")
print(f"Final dictionary rows:     {len(final):,}")

print(
    "Raw canonical codes:      "
    f"{raw['observed_code'].nunique(dropna=False):,}"
)

print(
    "Final unique codes:       "
    f"{final['observed_code'].nunique(dropna=False):,}"
)


duplicate_codes = final.loc[
    final["observed_code"].duplicated(keep=False)
].copy()

print(
    f"Duplicate final codes:     {len(duplicate_codes):,}"
)


# ============================================================
# 3. EXACT CODE-UNIVERSE ACCOUNTING
# ============================================================

raw_set = set(
    raw["observed_code"]
    .dropna()
    .astype(str)
)

final_set = set(
    final["observed_code"]
    .dropna()
    .astype(str)
)

missing_from_final = sorted(
    raw_set - final_set
)

extra_in_final = sorted(
    final_set - raw_set
)

print()
print("=" * 78)
print("CODE-UNIVERSE ACCOUNTING")
print("=" * 78)

print(
    f"Raw codes absent from final: {len(missing_from_final):,}"
)

print(
    f"Unexpected final codes:      {len(extra_in_final):,}"
)

if missing_from_final:
    print("\nMissing:")
    print(missing_from_final)

if extra_in_final:
    print("\nExtra:")
    print(extra_in_final)


# ============================================================
# 4. RECORD ACCOUNTING
# ============================================================

raw_records = (
    pd.to_numeric(
        raw["n_records"],
        errors="coerce"
    )
    .fillna(0)
    .sum()
)

final_records = (
    pd.to_numeric(
        final["n_records"],
        errors="coerce"
    )
    .fillna(0)
    .sum()
)

print()
print("=" * 78)
print("RECORD ACCOUNTING")
print("=" * 78)

print(f"Raw records:       {raw_records:,.0f}")
print(f"Final records:     {final_records:,.0f}")
print(f"Difference:        {final_records - raw_records:,.0f}")


# ============================================================
# 5. CRITICAL NONE / NO-CANOPY INVARIANT
# ============================================================

sentinel = final.loc[
    final["observed_code"].isin(
        ["__NO_CANOPY__", "NONE"]
    )
].copy()

print()
print("=" * 78)
print("NONE / __NO_CANOPY__")
print("=" * 78)

display(
    sentinel[
        [
            c for c in [
                "observed_code",
                "source_code_was_blank",
                "n_records",
                "n_top",
                "n_lower",
                "resolved_label",
                "code_class",
                "USDA_accepted_symbol",
                "USDA_scientific_name",
                "MOSAIC_FG",
                "resolution_source",
                "FG_resolution_source",
            ]
            if c in sentinel.columns
        ]
    ]
)

assert (
    final["observed_code"]
    .eq("__NO_CANOPY__")
    .sum()
    == 1
)

assert (
    final["observed_code"]
    .eq("NONE")
    .sum()
    == 1
)

no_canopy = final.loc[
    final["observed_code"].eq("__NO_CANOPY__")
].iloc[0]

literal_none = final.loc[
    final["observed_code"].eq("NONE")
].iloc[0]

assert no_canopy["n_records"] == 442612
assert literal_none["n_records"] == 398

assert no_canopy["code_class"] == "NoCanopy"
assert literal_none["code_class"] == "Plant"

assert pd.isna(
    no_canopy["USDA_accepted_symbol"]
)

assert (
    literal_none["USDA_accepted_symbol"]
    == "NONE"
)


# ============================================================
# 6. POAR2R2 CORRECTION
# ============================================================

poar = final.loc[
    final["observed_code"].eq("POAR2R2")
].copy()

print()
print("=" * 78)
print("POAR2R2")
print("=" * 78)

print("Rows:", len(poar))

display(
    poar[
        [
            c for c in [
                "observed_code",
                "n_records",
                "resolved_label",
                "code_class",
                "USDA_symbol",
                "USDA_accepted_symbol",
                "USDA_match_method",
                "USDA_scientific_name",
                "USDA_common_name",
                "USDA_family",
                "USDA_rank",
                "USDA_duration",
                "USDA_growth_habit",
                "USDA_native_status_L48",
                "MOSAIC_FG",
                "resolution_source",
                "FG_resolution_source",
            ]
            if c in poar.columns
        ]
    ]
)

assert len(poar) == 1

assert (
    poar.iloc[0]["USDA_accepted_symbol"]
    == "PONIN"
), (
    "POAR2R2 has not been corrected to accepted "
    "USDA symbol PONIN."
)


# ============================================================
# 7. BASSIA
# ============================================================

bassia = final.loc[
    final["observed_code"].isin(
        ["BAPRV", "BAPRG"]
    )
].copy()

print()
print("=" * 78)
print("BASSIA")
print("=" * 78)

display(
    bassia[
        [
            c for c in [
                "observed_code",
                "USDA_accepted_symbol",
                "USDA_scientific_name",
                "MOSAIC_FG",
                "resolution_source",
                "FG_resolution_source",
                "n_records",
            ]
            if c in bassia.columns
        ]
    ]
)

if len(bassia):

    assert (
        bassia["USDA_accepted_symbol"]
        .eq("BAPR5")
        .all()
    )

    assert (
        bassia["MOSAIC_FG"]
        .eq("ExoticForb")
        .all()
    )


# ============================================================
# 8. USDA ACCEPTED SYMBOLS THAT SHOULD NEVER SURVIVE
# ============================================================

bad_accepted_symbols = {
    "POAR22",
}

bad_taxa = final.loc[
    final["USDA_accepted_symbol"].isin(
        bad_accepted_symbols
    )
].copy()

print()
print("=" * 78)
print("OBSOLETE / INVALID ACCEPTED SYMBOLS")
print("=" * 78)

print(
    f"Rows carrying obsolete accepted symbols: "
    f"{len(bad_taxa):,}"
)

if len(bad_taxa):
    display(bad_taxa)

assert len(bad_taxa) == 0


# ============================================================
# 9. PLANT FG COMPLETENESS
# ============================================================

plants = final.loc[
    final["code_class"].eq("Plant")
].copy()

missing_fg = plants.loc[
    plants["MOSAIC_FG"].isna()
].copy()

print()
print("=" * 78)
print("PLANT FUNCTIONAL-GROUP COVERAGE")
print("=" * 78)

print(f"Plant codes:              {len(plants):,}")
print(
    f"Plant records:            "
    f"{plants['n_records'].sum():,}"
)

print(
    f"Plant codes missing FG:   "
    f"{len(missing_fg):,}"
)

print(
    f"Plant records missing FG: "
    f"{missing_fg['n_records'].sum():,}"
)


# ============================================================
# 10. ROW-LEVEL CONTEXT FLAGS
# ============================================================

if "requires_row_level_resolution" in final.columns:

    context_rows = final.loc[
        final[
            "requires_row_level_resolution"
        ].fillna(False)
    ].copy()

    print()
    print("=" * 78)
    print("ROW-LEVEL CONTEXT-DEPENDENT CODES")
    print("=" * 78)

    print(
        f"Context-dependent codes:   "
        f"{len(context_rows):,}"
    )

    print(
        f"Represented records:        "
        f"{context_rows['n_records'].sum():,}"
    )

    display(
        context_rows[
            [
                c for c in [
                    "observed_code",
                    "n_records",
                    "resolved_label",
                    "code_class",
                    "protocol_source",
                    "context_protocol_label",
                    "context_protocol_class",
                    "context_protocol_rule",
                    "MOSAIC_FG",
                ]
                if c in context_rows.columns
            ]
        ]
        .sort_values(
            "n_records",
            ascending=False
        )
    )


# ============================================================
# 11. HARD FINAL GATE
# ============================================================

checks = {
    "10,960 dictionary rows":
        len(final) == 10960,

    "10,960 unique observed codes":
        final["observed_code"].nunique(
            dropna=False
        ) == 10960,

    "No duplicate observed codes":
        len(duplicate_codes) == 0,

    "All raw codes represented":
        len(missing_from_final) == 0,

    "No unexpected codes":
        len(extra_in_final) == 0,

    "Exact record accounting":
        np.isclose(
            raw_records,
            final_records
        ),

    "One __NO_CANOPY__":
        final["observed_code"]
        .eq("__NO_CANOPY__")
        .sum() == 1,

    "One literal NONE":
        final["observed_code"]
        .eq("NONE")
        .sum() == 1,

    "POAR2R2 resolves to PONIN":
        (
            len(poar) == 1
            and
            poar.iloc[0][
                "USDA_accepted_symbol"
            ] == "PONIN"
        ),

    "No POAR22 accepted symbol remains":
        len(bad_taxa) == 0,

    "All plant codes have MOSAIC_FG":
        len(missing_fg) == 0,
}


print()
print("=" * 78)
print("FINAL GATE")
print("=" * 78)

all_pass = True

for name, passed in checks.items():

    status = (
        "PASS"
        if passed
        else "FAIL"
    )

    print(
        f"{status:<5}  {name}"
    )

    if not passed:
        all_pass = False


# ============================================================
# 12. WRITE ONLY IF EVERYTHING PASSES
# ============================================================

if not all_pass:

    raise RuntimeError(
        "Final dictionary failed one or more hard QA gates. "
        "Nothing was written."
    )


final.to_csv(
    FINAL_MASTER_FILE,
    index=False
)

print()
print("=" * 78)
print("FINAL MASTER WRITTEN")
print("=" * 78)

print(FINAL_MASTER_FILE)

FINAL DICTIONARY STRUCTURE
Raw code rows:             10,960
Final dictionary rows:     10,960
Raw canonical codes:      10,960
Final unique codes:       10,960
Duplicate final codes:     0

CODE-UNIVERSE ACCOUNTING
Raw codes absent from final: 0
Unexpected final codes:      0

RECORD ACCOUNTING
Raw records:       16,150,101
Final records:     16,150,101
Difference:        0

NONE / __NO_CANOPY__


,observed_code,source_code_was_blank,n_records,n_top,n_lower,resolved_label,code_class,USDA_accepted_symbol,USDA_scientific_name,MOSAIC_FG,resolution_source,FG_resolution_source
15,__NO_CANOPY__,True,442612,442612,0,No top-canopy contact,NoCanopy,<NA>,<NA>,NaN,source_blank_sentinel,<NA>
2527,NONE,False,398,394,4,Notholaena neglecta Maxon,Plant,NONE,Notholaena neglecta Maxon,TraitUnresolvedPlant,USDA_accepted_symbol,USDA_taxon_missing_traits



POAR2R2
Rows: 1


,observed_code,n_records,resolved_label,code_class,USDA_accepted_symbol,USDA_match_method,USDA_scientific_name,USDA_common_name,USDA_family,USDA_duration,USDA_growth_habit,USDA_native_status_L48,MOSAIC_FG,resolution_source,FG_resolution_source
10143,POAR2R2,10,Potentilla gracilis,Plant,POAR22,manual_alias,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,TraitUnresolvedPlant,manual_typo_correction,USDA_taxon_missing_traits


AssertionError: POAR2R2 has not been corrected to accepted USDA symbol PONIN.

In [ ]:
from pathlib import Path
import pandas as pd

LPI_GEOREF_CSV = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\LDC_LPI_georeferenced_2018_present.csv"
)

print("Loading georeferenced LPI table...")
lpi_georeferenced = pd.read_csv(
    LPI_GEOREF_CSV,
    low_memory=False
)

print(f"Rows: {len(lpi_georeferenced):,}")
print(f"Columns: {len(lpi_georeferenced.columns):,}")
print(f"Plot visits: {lpi_georeferenced['PrimaryKey'].nunique():,}")

Loading georeferenced LPI table...
Rows: 16,150,114
Columns: 33
Plot visits: 51,812


In [ ]:
from pathlib import Path
import pandas as pd

LPI_GEOREF_CSV = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\LDC_LPI_georeferenced_2018_present.csv"
)

OUT_CSV = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\LDC_LPI_plot_visits_2018_present.csv"
)

print("Loading georeferenced LPI table...")

lpi = pd.read_csv(
    LPI_GEOREF_CSV,
    low_memory=False
)

# ------------------------------------------------------------
# Keep one row per PrimaryKey
#
# PrimaryKey = plot visit / sampling event
# Temporal revisits at the same physical location remain distinct
# because they have distinct PrimaryKeys.
# ------------------------------------------------------------

keep_cols = [
    "PrimaryKey",
    "DBKey",
    "ProjectKey",
    "DateVisited",
    "Latitude_NAD83",
    "Longitude_NAD83",
]

keep_cols = [
    c for c in keep_cols
    if c in lpi.columns
]

plot_visits = (
    lpi[keep_cols]
    .dropna(
        subset=[
            "PrimaryKey",
            "Latitude_NAD83",
            "Longitude_NAD83",
        ]
    )
    .copy()
)

# ------------------------------------------------------------
# Derive visit year
# ------------------------------------------------------------

if "DateVisited" in plot_visits.columns:

    plot_visits["DateVisited"] = pd.to_datetime(
        plot_visits["DateVisited"],
        errors="coerce"
    )

    plot_visits["Year"] = (
        plot_visits["DateVisited"]
        .dt.year
        .astype("Int64")
    )

# ------------------------------------------------------------
# Collapse repeated tall LPI rows ONLY within the same PrimaryKey
#
# The tall LPI table contains many contact records per visit.
# We are reducing those repeated rows to one visit-level center.
# We are NOT collapsing across different PrimaryKeys.
# ------------------------------------------------------------

visit_coord_counts = (
    plot_visits
    .groupby("PrimaryKey")[
        ["Latitude_NAD83", "Longitude_NAD83"]
    ]
    .nunique(dropna=False)
)

ambiguous = visit_coord_counts.loc[
    (visit_coord_counts["Latitude_NAD83"] > 1)
    |
    (visit_coord_counts["Longitude_NAD83"] > 1)
]

print(
    f"PrimaryKeys with multiple coordinate values: "
    f"{len(ambiguous):,}"
)

if len(ambiguous):
    display(
        plot_visits.loc[
            plot_visits["PrimaryKey"].isin(
                ambiguous.index
            )
        ]
        .sort_values(
            ["PrimaryKey", "DateVisited"]
        )
        .head(100)
    )

# ------------------------------------------------------------
# One row per visit
#
# Safe because we collapse ONLY repeated rows carrying the same
# PrimaryKey from the tall LPI table.
# ------------------------------------------------------------

if len(ambiguous) == 0:

    plot_visits = (
        plot_visits
        .drop_duplicates(
            subset=["PrimaryKey"],
            keep="first"
        )
        .reset_index(drop=True)
    )

else:

    raise ValueError(
        "Some PrimaryKeys contain multiple coordinate values. "
        "Resolve those before creating the GEE upload table."
    )

# ------------------------------------------------------------
# QA
# ------------------------------------------------------------

print()
print(
    f"Visit-level rows: "
    f"{len(plot_visits):,}"
)

print(
    f"Unique PrimaryKeys: "
    f"{plot_visits['PrimaryKey'].nunique():,}"
)

if "Year" in plot_visits.columns:

    print("\nVisits by year:")

    print(
        plot_visits["Year"]
        .value_counts()
        .sort_index()
    )

# ------------------------------------------------------------
# Save for GEE upload
# ------------------------------------------------------------

plot_visits.to_csv(
    OUT_CSV,
    index=False
)

print()
print(
    f"Written:\n{OUT_CSV}"
)

display(
    plot_visits.head()
)

Loading georeferenced LPI table...
PrimaryKeys with multiple coordinate values: 0

Visit-level rows: 51,812
Unique PrimaryKeys: 51,812

Visits by year:
Year
2018    7313
2019    9039
2020    7345
2021    8597
2022    9331
2023    8490
2024    1689
2025       8
Name: count, dtype: Int64

Written:
C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\LDC_LPI_plot_visits_2018_present.csv


,PrimaryKey,DateVisited,Latitude_NAD83,Longitude_NAD83,Year
0,2105280935464382021-05-28,2021-05-28 00:00:00+00:00,41.5294,-116.978,2021
1,2105290848197392021-05-29,2021-05-29 00:00:00+00:00,41.5369,-116.975,2021
2,21052913471776492021-05-29,2021-05-29 00:00:00+00:00,41.5416,-116.966,2021
3,21053011203293392021-05-30,2021-05-30 00:00:00+00:00,41.5931,-117.371,2021
4,21053109290646562021-05-31,2021-05-31 00:00:00+00:00,41.5772,-117.382,2021
